<a href="https://colab.research.google.com/github/tommbk50-hub/Cheminformatics/blob/main/pyOpenMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenMS, py_hplc and Gemini AI Agents: Automated HPLC-GCMS Chromatrogram Analysis of Complex Metabolite Mixtures.

OpenMS is an open-source, C++ framework hosted on Github <https://github.com/OpenMS/OpenMS> for analyzing large volumes of mass spectrometry data. It has been specially designed for analyzing high performance LC-MS data but over recent times, has been extended to analyze data generated by other techniques.

https://pyopenms.readthedocs.io/en/latest/user_guide/background.html


**Install dependencies**

- Install Colab scientific, LCMS, chemistry and plotting packages using uv/pip.
- Load the HPLC peak-deconvolution library and a shared Plotly color helper.


In [ ]:
#@title 1. Install required Python libraries with pinned versions
!uv pip install --system -q \
    "ipyketcher" \
    "rdkit>=2023.9.1" \
    "mols2grid==2.0.0" \
    "pyteomics==4.6.3" \
    "pyopenms>=3.1.0" \
    "psims==1.3.1" \
    "lxml>=5.1.0" \
    "hdf5plugin==4.3.0" \
    "pandas>=2.0.0,<3.0.0" \
    "pubchempy==1.0.4" \
    "mendeleev" \
    "marimo==0.2.5" \
    "wigglystuff==0.1.3" \
    "matchms" \
    "onnxruntime>=1.17.1" \
    "bottleneck==1.3.8" \
    "plotly>=5.18.0"

!pip -q install kaleido==0.1.0

import os
import glob
import pandas as pd
import matplotlib.colors as mcolors

# Helper to convert matplotlib colors to Plotly-compatible RGB strings
def to_plotly_color(color_val):
    """Converts any matplotlib-compatible color (including RGB tuples) to an rgb(r,g,b) string for Plotly."""
    rgba = mcolors.to_rgba(color_val)
    return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"

# 1. Clone hplc-py for chromatographic deconvolution
if not os.path.exists('hplc-py'):
    !git clone -q https://github.com/cremerlab/hplc-py.git

**Load LCMS data**

- Download and parse the example mzML file into an MS1 TIC DataFrame (retention times in seconds).
- Show the raw chromatogram and retain a plot for later visual quality checks.


In [ ]:
#@title 2. Plot HPLC Chromatogram, Convert the HPLC Trace into a Pandas dataframe
import urllib.request
import os
import matplotlib.pyplot as plt
from pyteomics import mzml
import pandas as pd

# 1. Download the new dataset
url_new = "https://raw.githubusercontent.com/OpenMS/OpenMS/develop/src/tests/topp/FeatureFinderMetaboIdent_1_input.mzML"
filename_new = "FeatureFinderMetaboIdent_1_input.mzML"

if not os.path.exists(filename_new):
    print(f"Downloading {filename_new}...")
    urllib.request.urlretrieve(url_new, filename_new)

# 2. Parse the new dataset
print(f"Parsing {filename_new}...")
rt_new = []
tic_new = []

try:
    with mzml.read(filename_new) as reader:
        for spectrum in reader:
            if spectrum['ms level'] == 1:
                scan = spectrum.get('scanList', {}).get('scan', [{}])[0]
                rt = scan.get('scan start time', 0)
                tic = spectrum.get('total ion current', 0)
                rt_new.append(rt)
                tic_new.append(tic)

    # Create a pandas DataFrame from the parsed data
    hplc_df = pd.DataFrame({
        'Signal': tic_new,
        'Time': rt_new
    })

    print("HPLC Trace DataFrame successfully created:")
    print(hplc_df.head())

    # 3. Plot the new HPLC trace using the DataFrame
    plt.figure(figsize=(10, 6))
    plt.plot(hplc_df['Time'], hplc_df['Signal'], color='green', linewidth=1.5)
    plt.title('HPLC Trace: FeatureFinderMetaboIdent_1_input.mzML')
    plt.xlabel('Retention Time (seconds)')
    plt.ylabel('Total Ion Current (Signal)')
    plt.grid(True)
    plt.tight_layout()
    # Cache this Figure (before matplotlib's inline backend auto-closes it after
    # this cell finishes) so later cells can feed it into Gemini's vision context.
    globals().setdefault('_notebook_figure_registry', {})['raw_hplc_chromatogram'] = plt.gcf()

    plt.show()

except Exception as e:
    print(f"An error occurred: {e}")

**Enter candidate structures**

- Sketch molecules or edit the Starting Material / Reagent / Product SMILES table.
- Calculate neutral exact masses locally with RDKit for downstream peak matching.


In [ ]:
#@title 3. List SMILES to find
import pandas as pd
import os
import sys
import traitlets
import subprocess
import ipywidgets as widgets
from rdkit import Chem
from rdkit.Chem import Descriptors

# Dynamic patch to solve 'TypeError: type "List" is not subscriptable' on Python 3.13 inside ipyketcher[cite: 1]
if not hasattr(traitlets.List, '__class_getitem__'):
    traitlets.List.__class_getitem__ = classmethod(lambda cls, item: cls)

# --- PATCH IPYKETCHER BUG ---[cite: 1]
ipk_init = f"/usr/local/lib/python{sys.version_info.major}.{sys.version_info.minor}/dist-packages/ipyketcher/__init__.py"
if os.path.exists(ipk_init):
    with open(ipk_init, 'r') as f:
        content = f.read()
    if "traitlets.List[str]()" in content:
        with open(ipk_init, 'w') as f:
            f.write(content.replace("traitlets.List[str]()", "traitlets.List()"))
# ----------------------------

# Force reload in case the broken version was already cached in memory[cite: 1]
if 'ipyketcher' in sys.modules:
    del sys.modules['ipyketcher']
import ipyketcher
from IPython.display import display, HTML

# 1. Initialize and display the molecular sketching canvas[cite: 1]
editor = ipyketcher.KetcherEditor()
print("🎨 Canvas initialized. (Draw molecules to add them to your interactive table below)")
display(editor)

# Helper function to calculate exact mass via RDKit
def calculate_exact_mass(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol:
        return f"{Descriptors.ExactMolWt(mol):.4f}"
    return "Invalid"

# Define reaction scheme list dynamically with the requested molecules[cite: 1]
reaction_smiles = [
    ("adenosine", "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
    ("n-deoxyadenosine", "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)C1"),
    ("n-inosine", "Product", "O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
    ("n2-o-methylcytidine", "Product", "CO[C@@H]1[C@H](O[C@H](CO)[C@@H]1O)n2ccc(N)nc2=O"),
    ("n5-methylcytidine", "Product", "Cc1cn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)c(=O)nc1N"),
    ("n5-formylcytidine", "Product", "Nc1nc(=O)n([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)cc1C=O")
]

# Create a UI Row for the interactive table
def create_table_row(comp_no, name, classification, smiles):
    w_no = widgets.HTML(value=f"<b>{comp_no}</b>", layout=widgets.Layout(width='40px', padding='0 5px'))
    w_name = widgets.Text(value=name, placeholder='Molecule Name', layout=widgets.Layout(width='160px', padding='0 5px'))
    w_class = widgets.Dropdown(
        options=['Reactant', 'Reagent', 'Product'],
        value=classification,
        layout=widgets.Layout(width='100px', padding='0 5px')
    )
    w_smiles = widgets.Text(value=smiles, layout=widgets.Layout(width='320px', padding='0 5px'))
    w_mass = widgets.HTML(value=calculate_exact_mass(smiles), layout=widgets.Layout(width='80px', padding='0 5px'))

    return widgets.HBox([w_no, w_name, w_class, w_smiles, w_mass])

# Build Table Header
header_row = widgets.HBox([
    widgets.HTML(value="<b>No.</b>", layout=widgets.Layout(width='40px', padding='0 5px')),
    widgets.HTML(value="<b>Name</b>", layout=widgets.Layout(width='160px', padding='0 5px')),
    widgets.HTML(value="<b>Classification</b>", layout=widgets.Layout(width='100px', padding='0 5px')),
    widgets.HTML(value="<b>SMILES String</b>", layout=widgets.Layout(width='320px', padding='0 5px')),
    widgets.HTML(value="<b>Exact Mass</b>", layout=widgets.Layout(width='80px', padding='0 5px'))
])

# Populate initial table rows
table_rows = [create_table_row(i + 1, name, cls, smi) for i, (name, cls, smi) in enumerate(reaction_smiles)]
table_container = widgets.VBox([header_row] + table_rows)

btn_update = widgets.Button(description='Update List', button_style='success')
out = widgets.Output()

def on_update_clicked(b):
    with out:
        out.clear_output()
        # ipyketcher stores current smiles in editor.smiles[cite: 1]
        drawn_smi = getattr(editor, 'smiles', None)
        if drawn_smi:
            # Check for duplicates based on existing SMILES in the table
            existing_smiles = [row.children[3].value for row in table_container.children[1:]]
            if drawn_smi not in existing_smiles:
                new_index = len(table_container.children) # Header counts as 1, so len equals the next compound number
                new_row = create_table_row(new_index, "Drawn Molecule", "Reactant", drawn_smi)

                # Append the new row to the table container
                table_container.children = list(table_container.children) + [new_row]
                print(f"Added to table: {drawn_smi}")
            else:
                print(f"Molecule already exists in the table: {drawn_smi}")
        else:
            print("No SMILES found in editor. Please draw a molecule first.")[cite: 1]

btn_update.on_click(on_update_clicked)

display(table_container)
display(widgets.HBox([btn_update]))
display(out)


# --- AUTO-FUNCTION CALLING PATCH FOR STEP 3 CHAT UI ---
if 'client' in globals():
    _orig_chats_create = client.chats.create

    def _patched_chats_create(*args, **kwargs):
        config = kwargs.get('config')
        if isinstance(config, types.GenerateContentConfig):
            config.automatic_function_calling = types.AutomaticFunctionCallingConfig(
                disable=False,
                maximum_remote_calls=15
            )
        elif isinstance(config, dict):
            config['automatic_function_calling'] = {'disable': False, 'maximum_remote_calls': 15}
        return _orig_chats_create(*args, **kwargs)

    client.chats.create = _patched_chats_create
    if 'init_chat_session' in globals():
        init_chat_session()
    print("✅ Patched Gemini client: Automatic Function Calling is now active!")


* Here we use an **LC-MS1 alignment/extraction pipeline** from the OpenMS Python wrapper, **pyOpenMS**, directly inside this colab notebook to perform a precise, high-resolution Extracted Ion Chromatogram (EIC) extraction and retention time alignment for your target molecular masses.

**Detect and assign peaks**

- Load mzML spectra with pyOpenMS and deconvolute the chromatogram.
- Score dominant MS1 ions against candidate adducts and save detected peaks for later tools.


In [ ]:
#@title 4. Unified Peak Detection & Dominant-m/z Confidence Scoring (dataframe = detected_df)
import subprocess
import sys
import os
import re
import warnings

try:
    import scipy.signal
except ImportError:
    print("🔧 Installing missing dependencies (scipy, numpy) via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "numpy", "scipy"], capture_output=True)
    import scipy.signal
    print("✅ Repair complete.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display, clear_output, Markdown
import ipywidgets as widgets
from pyteomics import mzml
import pyopenms as po
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

warnings.filterwarnings('ignore')

# 1. Ensure hplc-py and pyOpenMS experiment are loaded
if 'hplc-py' not in sys.path:
    sys.path.append(os.path.abspath('hplc-py'))
from hplc.quant import Chromatogram

mzml_filename = "FeatureFinderMetaboIdent_1_input.mzML"
if not os.path.exists(mzml_filename):
    mzml_filename = "/content/FeatureFinderMetaboIdent_1_input.mzML"

if 'od' not in globals() and os.path.exists(mzml_filename):
    od = po.MSExperiment()
    po.FileHandler().loadExperiment(mzml_filename, od)

signal_raw = hplc_df['Signal'].values
signal_norm = (signal_raw - signal_raw.min()) / (signal_raw.max() - signal_raw.min())

# --- 2. Dynamic Hyperparameter Optimization (Elbow Method) ---
print("Running Dynamic Hyperparameter Optimization...")
prominence_range = np.logspace(-2, -0.5, 8)
buffer_val = 0
results = []

chrom_base = Chromatogram(hplc_df, cols={'time': 'Time', 'signal': 'Signal'}, time_window=[200, 300])

for p in prominence_range:
    try:
        peaks = chrom_base.fit_peaks(prominence=p, buffer=buffer_val)
        results.append((p, len(peaks)))
    except Exception:
        pass

if not results:
    default_prominence = 0.05
    print("Optimization failed. Using default prominence: 0.05")
else:
    proms_all = np.array([x[0] for x in results])
    counts_all = np.array([x[1] for x in results])
    max_idx = np.argmax(counts_all)
    valid_proms = proms_all[max_idx:]
    valid_counts = counts_all[max_idx:]

    if len(valid_proms) > 1:
        p1 = np.array([np.log10(valid_proms[0]), valid_counts[0]])
        p2 = np.array([np.log10(valid_proms[-1]), valid_counts[-1]])
        distances = []
        for i in range(len(valid_proms)):
            p3 = np.array([np.log10(valid_proms[i]), valid_counts[i]])
            if np.linalg.norm(p2 - p1) == 0:
                dist = 0
            else:
                cross_prod_2d = (p2[0] - p1[0]) * (p1[1] - p3[1]) - (p2[1] - p1[1]) * (p1[0] - p3[0])
                dist = np.abs(cross_prod_2d) / np.linalg.norm(p2 - p1)
            distances.append(dist)
        optimal_idx_relative = np.argmax(distances)
        optimal_prominence = valid_proms[optimal_idx_relative]
        optimal_count = valid_counts[optimal_idx_relative]
    else:
        optimal_prominence = valid_proms[0]
        optimal_count = valid_counts[0]

    default_prominence = float(optimal_prominence)
    prominence_val = default_prominence

    display(Markdown("---"))
    display(Markdown("### ✅ Optimization Complete"))
    display(Markdown(f"**Optimal Prominence Threshold:** `{optimal_prominence:.4f}` | **Peaks Resolved:** `{optimal_count}`"))

    fig_opt, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [2.5, 1]})
    ax1.plot(proms_all, counts_all, marker='o', linestyle='-', color='#2c3e50', linewidth=2, label='Fitted Peaks')
    ax1.axvline(optimal_prominence, color='#e74c3c', linestyle='--', linewidth=2, label=f'Optimal Elbow ({optimal_prominence:.4f})')
    ax1.scatter([optimal_prominence], [optimal_count], color='#e74c3c', s=150, zorder=5)
    ax1.set_xscale('log')
    ax1.set_xticks(prominence_range)
    ax1.set_xticklabels([f"{x:.4f}" for x in prominence_range], rotation=45)
    ax1.set_title('Hyperparameter Optimization: Signal-to-Noise Ratio (L-Curve)', fontsize=13, pad=12)
    ax1.set_xlabel('Prominence Threshold (Log Scale)', fontsize=11)
    ax1.set_ylabel('Number of Deconvoluted Peaks', fontsize=11)
    ax1.legend(fontsize=10)
    ax1.grid(True, which="both", ls="--", alpha=0.4)

    ax2.axis('tight')
    ax2.axis('off')
    table_data = [[f"{p:.4f}", int(c)] for p, c in zip(proms_all, counts_all)]
    col_labels = ['Prominence Threshold', 'Number of\nDeconvoluted Peaks']
    table = ax2.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.1, 1.8)
    optimal_row_idx = list(proms_all).index(optimal_prominence) + 1
    for (row, col), cell in table.get_celld().items():
        if row == optimal_row_idx:
            cell.set_edgecolor('red')
            cell.set_linewidth(2)
    ax2.set_title('Tabular Data', fontsize=12, pad=12)
    plt.tight_layout()
    plt.show()

# --- 3. Candidate Fetching & Theoretical Ion / Isotope Generators ---
def get_candidates_from_table():
    if 'table_container' in globals():
        candidates = []
        for row in table_container.children[1:]:
            name = row.children[1].value.strip()
            classification = row.children[2].value
            smi = row.children[3].value.strip()
            if smi:
                candidates.append((name, classification, smi))
        if candidates:
            return candidates
    return [
        ("adenosine",           "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
        ("n-deoxyadenosine",    "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)C1"),
        ("n-inosine",           "Product", "O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
        ("n2-o-methylcytidine", "Product", "CO[C@@H]1[C@H](O[C@H](CO)[C@@H]1O)n2ccc(N)nc2=O"),
        ("n5-methylcytidine",   "Product", "Cc1cn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)c(=O)nc1N"),
        ("n5-formylcytidine",   "Product", "Nc1nc(=O)n([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)cc1C=O")
    ]

esi_pos_adducts = [
    ("[M+H]+",       1, 1.007276,  1),
    ("[M+NH4]+",     1, 18.033823, 1),
    ("[M+Na]+",      1, 22.989218, 1),
    ("[M+CH3OH+H]+", 1, 33.033489, 1),
    ("[M+K]+",       1, 38.963158, 1),
    ("[M+ACN+H]+",   1, 42.033823, 1),
    ("[2M+H]+",      2, 1.007276,  1),
    ("[2M+Na]+",     2, 22.989218, 1),
]

insource_neutral_losses = [
    ("[M+H-Ribose]+",      -132.042259),
    ("[M+H-Deoxyribose]+", -116.047344),
    ("[M+H-MeRibose]+",    -146.057909),
    ("[M+H-H2O]+",         -18.010565),
]

def parse_dominant_mz_string(dom_str):
    if not dom_str or dom_str == "None":
        return []
    parsed = []
    for item in str(dom_str).split(","):
        m = re.search(r"([0-9]+\.[0-9]+)\s*\(([0-9.]+)%\)", item.strip())
        if m:
            parsed.append((float(m.group(1)), float(m.group(2))))
    return parsed

def compute_confidence_from_dominant_mz(best_group, dom_mz_list, full_window_spectrum):
    adducts = best_group["adducts"]
    primary = adducts[0]
    mol = best_group["mols"][0]
    exact_mass = best_group["exact_mass"]

    # Mass Accuracy Score
    mean_abs_ppm = np.mean([abs(a["ppm"]) for a in adducts])
    ppm_score = max(0.0, 35.0 * (1.0 - (mean_abs_ppm / 15.0)))

    # Dominant m/z Rank
    matched_dom_rel_int = 0.0
    for a in adducts:
        for dom_mz, dom_rel in dom_mz_list:
            if abs(dom_mz - a["calc_mz"]) <= 0.015:
                matched_dom_rel_int = max(matched_dom_rel_int, dom_rel)
    if matched_dom_rel_int >= 95.0:
        dom_rank_score = 35.0
    elif matched_dom_rel_int > 0:
        dom_rank_score = 12.0 + 23.0 * (matched_dom_rel_int / 100.0)
    else:
        dom_rank_score = 5.0

    # Isotope Envelope Verification
    num_C = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6)
    num_N = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 7)
    num_S = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 16)
    theo_m1_rel = (num_C * 1.07) + (num_N * 0.37) + (num_S * 0.76)
    if theo_m1_rel == 0:
        theo_m1_rel = 12.0

    m0_mz = primary["calc_mz"]
    m1_target = m0_mz + 1.003355
    m2_target = m0_mz + 2.006710

    iso_score = 0.0
    iso_evidence = []
    m1_obs_rel = None
    for dom_mz, dom_rel in dom_mz_list:
        if abs(dom_mz - m1_target) <= 0.015 and matched_dom_rel_int > 0:
            m1_obs_rel = (dom_rel / matched_dom_rel_int) * 100.0
            break
    if m1_obs_rel is None and full_window_spectrum and matched_dom_rel_int > 0:
        m0_int = max([inten for mz_k, inten in full_window_spectrum.items() if abs(mz_k - m0_mz) <= 0.015], default=0)
        m1_int = max([inten for mz_k, inten in full_window_spectrum.items() if abs(mz_k - m1_target) <= 0.015], default=0)
        if m0_int > 0 and m1_int > 0:
            m1_obs_rel = (m1_int / m0_int) * 100.0

    if m1_obs_rel is not None:
        rel_diff = abs(m1_obs_rel - theo_m1_rel)
        iso_score += max(6.0, 16.0 * (1.0 - min(rel_diff / 15.0, 1.0)))
        iso_evidence.append(f"[M+H+1]+ iso ({m1_obs_rel:.0f}% vs {theo_m1_rel:.0f}% theo)")

    for dom_mz, dom_rel in dom_mz_list:
        if abs(dom_mz - m2_target) <= 0.015:
            iso_score += 4.0
            iso_evidence.append("[M+H+2]+ iso")
            break

    # Multi-Adduct & In-Source Fragments
    corrob_score = 0.0
    if len(adducts) > 1:
        corrob_score += min(7.0, (len(adducts) - 1) * 4.5)

    mh_mz = exact_mass + 1.007276
    for loss_name, shift in insource_neutral_losses:
        frag_mz = mh_mz + shift
        if frag_mz > 80.0:
            for dom_mz, dom_rel in dom_mz_list:
                if abs(dom_mz - frag_mz) <= 0.015:
                    corrob_score += 5.0
                    iso_evidence.append(f"{loss_name} ({dom_mz:.3f})")
                    break

    corrob_score = min(10.0, corrob_score)
    total_conf = round(min(99.9, ppm_score + dom_rank_score + iso_score + corrob_score), 1)

    evidence_summary = f"Dom.Rank: {matched_dom_rel_int:.0f}%"
    if iso_evidence:
        evidence_summary += " | " + ", ".join(iso_evidence[:2])
    return total_conf, evidence_summary

# --- 4. Interactive UI & Unified Execution Pipeline ---
prominence_input = widgets.FloatText(
    value=default_prominence,
    description='Prominence:',
    step=0.001,
    style={'description_width': 'initial'}
)
rerun_button = widgets.Button(
    description='Rerun Peak Fitting & Assignment',
    button_style='primary',
    icon='refresh',
    layout=widgets.Layout(width='240px')
)
output_area = widgets.Output()

# Global stores shared with Cell 5 and Step 3 Gemini Tools
peak_ms1_spectra = {}
peak_assignments = {}
chrom = None
current_prominence_val = default_prominence

def run_unified_pipeline(prom_val):
    global detected_df, peak_assignments, peak_ms1_spectra, chrom, current_prominence_val
    current_prominence_val = prom_val
    with output_area:
        clear_output(wait=True)
        print(f"Using prominence threshold: {prom_val:.4f}")

        # Topographical peak detection
        peak_indices, _ = scipy.signal.find_peaks(signal_norm, prominence=prom_val)
        _, left_bases, right_bases = scipy.signal.peak_prominences(signal_norm, peak_indices)
        start_times = hplc_df['Time'].iloc[left_bases].values
        end_times = hplc_df['Time'].iloc[right_bases].values

        print(f"Detected {len(peak_indices)} raw topographical anomalies. Deconvoluting true peaks...")

        chrom = Chromatogram(hplc_df, cols={'time': 'Time', 'signal': 'Signal'}, time_window=[200, 300])
        peaks = chrom.fit_peaks(prominence=prom_val, buffer=0).reset_index(drop=True)

        mapped_starts, mapped_ends, mapped_topos = [], [], []
        peak_rts = hplc_df['Time'].values[peak_indices]
        for rt in peaks['retention_time']:
            if len(peak_rts) > 0:
                closest_idx = np.argmin(np.abs(peak_rts - rt))
                mapped_starts.append(start_times[closest_idx])
                mapped_ends.append(end_times[closest_idx])
                mapped_topos.append(peak_rts[closest_idx])
            else:
                mapped_starts.append(np.nan)
                mapped_ends.append(np.nan)
                mapped_topos.append(np.nan)

        peaks['peak start time'] = mapped_starts
        peaks['peak end time'] = mapped_ends
        peaks['topographical_apex'] = mapped_topos

        if peaks['area'].sum() > 0:
            peaks['normalised area'] = (peaks['area'] / peaks['area'].sum() * 100).map('{:.2f}%'.format)
        else:
            peaks['normalised area'] = '0.00%'

        # Extract top 10 dominant m/z signals
        peak_spectra = {i: {} for i in range(len(peaks))}
        if os.path.exists(mzml_filename):
            with mzml.read(mzml_filename) as reader:
                for spectrum in reader:
                    if spectrum['ms level'] == 1:
                        scan_data = spectrum.get('scanList', {}).get('scan', [{}])
                        rt = scan_data[0].get('scan start time', -1) if isinstance(scan_data, list) and scan_data else (scan_data.get('scan start time', -1) if isinstance(scan_data, dict) else -1)
                        mz_array = spectrum.get('m/z array', [])
                        int_array = spectrum.get('intensity array', [])
                        if len(mz_array) > 0:
                            mzs_rounded = np.round(mz_array, 3)
                            for i, row in peaks.iterrows():
                                if not pd.isna(row['peak start time']) and row['peak start time'] <= rt <= row['peak end time']:
                                    for m, intensity in zip(mzs_rounded, int_array):
                                        peak_spectra[i][m] = peak_spectra[i].get(m, 0) + intensity

        TOP_N_PEAKS = 10
        MIN_REL_INTENSITY = 2.0
        dominant_mz_col = []
        for i in range(len(peaks)):
            spectra = peak_spectra[i]
            if not spectra:
                dominant_mz_col.append("None")
                continue
            sorted_mzs = sorted(spectra.items(), key=lambda x: x[1], reverse=True)
            max_int = sorted_mzs[0][1]
            top_strs = []
            for m, inten in sorted_mzs:
                rel_int = (inten / max_int) * 100
                if rel_int >= MIN_REL_INTENSITY:
                    top_strs.append(f"{m:.3f} ({rel_int:.0f}%)")
                if len(top_strs) >= TOP_N_PEAKS:
                    break
            dominant_mz_col.append(", ".join(top_strs) if top_strs else "None")

        peaks['dominant m/z'] = dominant_mz_col

        # Screen ESI+ adducts & score
        candidates = get_candidates_from_table()
        theoretical_ions = []
        for name, classification, smi in candidates:
            mol = Chem.MolFromSmiles(smi)
            if mol:
                exact_mass = Descriptors.ExactMolWt(mol)
                formula = rdMolDescriptors.CalcMolFormula(mol)
                for adduct_name, mult, shift, z in esi_pos_adducts:
                    calc_mz = (exact_mass * mult + shift) / z
                    theoretical_ions.append({
                        "name": f"{name} ({classification})",
                        "raw_name": name,
                        "classification": classification,
                        "smiles": smi,
                        "mol": mol,
                        "formula": formula,
                        "exact_mass": round(exact_mass, 4),
                        "adduct": adduct_name,
                        "calc_mz": calc_mz
                    })

        PPM_TOLERANCE = 15.0
        MIN_PEAK_INTENSITY = 5e5
        peak_assignments = {}

        if 'od' in globals():
            for idx_row, peak_row in peaks.iterrows():
                p_id = int(peak_row['peak_id'])
                rt_start = peak_row['peak start time']
                rt_end = peak_row['peak end time']
                rt_apex = peak_row['retention_time']
                dom_mz_parsed = parse_dominant_mz_string(peak_row['dominant m/z'])
                if pd.isna(rt_start) or pd.isna(rt_end):
                    continue
                window_scans = [
                    spec for spec in od
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end
                ]
                if not window_scans:
                    continue

                mass_group_hits = {}
                for ion in theoretical_ions:
                    target_mz = ion["calc_mz"]
                    tol_da = target_mz * (PPM_TOLERANCE * 1e-6)
                    max_int = 0.0
                    obs_mz_at_max = target_mz
                    rt_at_max = rt_apex
                    for spec in window_scans:
                        mzs, ints = spec.get_peaks()
                        mask = (mzs >= target_mz - tol_da) & (mzs <= target_mz + tol_da)
                        if np.any(mask):
                            best_idx = np.argmax(ints[mask])
                            peak_int = float(ints[mask][best_idx])
                            if peak_int > max_int:
                                max_int = peak_int
                                obs_mz_at_max = float(mzs[mask][best_idx])
                                rt_at_max = float(spec.getRT())

                    if max_int >= MIN_PEAK_INTENSITY and abs(rt_at_max - rt_apex) <= 6.0:
                        em = ion["exact_mass"]
                        if em not in mass_group_hits:
                            mass_group_hits[em] = {
                                "exact_mass": em,
                                "formula": ion["formula"],
                                "names": [],
                                "raw_names": [],
                                "mols": [],
                                "total_intensity": 0.0,
                                "adducts": {}
                            }
                        if ion["name"] not in mass_group_hits[em]["names"]:
                            mass_group_hits[em]["names"].append(ion["name"])
                            mass_group_hits[em]["raw_names"].append(ion["raw_name"])
                            mass_group_hits[em]["mols"].append(ion["mol"])
                        if ion["adduct"] not in mass_group_hits[em]["adducts"]:
                            ppm_err = ((obs_mz_at_max - target_mz) / target_mz) * 1e6
                            mass_group_hits[em]["adducts"].setdefault(ion["adduct"], {
                                "adduct": ion["adduct"],
                                "obs_mz": obs_mz_at_max,
                                "calc_mz": target_mz,
                                "ppm": ppm_err,
                                "intensity": max_int
                            })
                            mass_group_hits[em]["total_intensity"] += max_int

                if mass_group_hits:
                    best_group = max(mass_group_hits.values(), key=lambda x: x["total_intensity"])
                    sorted_adducts = sorted(best_group["adducts"].values(), key=lambda x: x["intensity"], reverse=True)
                    best_group["adducts"] = sorted_adducts
                    is_isobaric = len(best_group["raw_names"]) > 1
                    iso_tag = f" [Isobaric Isomers, {best_group['formula']}]" if is_isobaric else ""
                    conf_score, evidence_str = compute_confidence_from_dominant_mz(
                        best_group, dom_mz_parsed, peak_spectra.get(idx_row, {})
                    )
                    peak_assignments[p_id] = {
                        "display_name": " / ".join(best_group["raw_names"]),
                        "full_assignment": " / ".join(best_group["names"]) + iso_tag,
                        "is_isobaric": is_isobaric,
                        "formula": best_group["formula"],
                        "mols": best_group["mols"],
                        "adducts": sorted_adducts,
                        "confidence": conf_score,
                        "evidence": evidence_str
                    }

        assignment_col = []
        confidence_col = []
        for idx_row, row in peaks.iterrows():
            p_id = int(row['peak_id'])
            if p_id in peak_assignments:
                assign_info = peak_assignments[p_id]
                top_adduct = assign_info["adducts"][0]
                assignment_col.append(
                    f"{assign_info['full_assignment']} [{top_adduct['adduct']}: {top_adduct['obs_mz']:.4f} m/z ({top_adduct['ppm']:+.1f} ppm)]"
                )
                confidence_col.append(f"{assign_info['confidence']:.1f}% ({assign_info['evidence']})")
            else:
                dom_list = parse_dominant_mz_string(row['dominant m/z'])
                base_str = f"Base m/z: {dom_list[0][0]:.3f}" if dom_list else "No MS1 signal"
                assignment_col.append(f"Unassigned ({base_str})")
                confidence_col.append("N/A")

        peaks['Assignment'] = assignment_col
        peaks['Confidence (%)'] = confidence_col
        detected_df = peaks.copy()

        # Populate the per-peak MS1 spectra store for Cell 5 & Step 3
        peak_ms1_spectra = {int(row['peak_id']): peak_spectra.get(idx_row, {}) for idx_row, row in peaks.iterrows()}

        def highlight_peaks(row):
            colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
            c = mcolors.to_rgba(colors[(int(row['peak_id']) - 1) % len(colors)], alpha=0.4)
            bg_color = f'rgba({int(c[0]*255)}, {int(c[1]*255)}, {int(c[2]*255)}, {c[3]})'
            return [f'background-color: {bg_color}' for _ in row]

        display(detected_df.style.apply(highlight_peaks, axis=1))
        print(f"\nFinal deconvoluted peak count: {len(detected_df)} | Assigned peaks: {len(peak_assignments)}")
        print("✅ Data ready! Run Cell 5 below to render the interactive chromatogram & MS1 comparison widgets.")

def on_rerun_clicked(b):
    run_unified_pipeline(prominence_input.value)

rerun_button.on_click(on_rerun_clicked)
display(widgets.HBox([prominence_input, rerun_button]))
display(output_area)
run_unified_pipeline(default_prominence)

**Review chromatogram and spectra**

- Plot deconvoluted peaks and candidate structure hover cards.
- Compare selected MS1 peaks interactively without passing full spectra to a model.


In [ ]:
#@title 5. Plot Deconvoluted HPLC Chromatogram with Hover Structure Cards & Multi-Peak MS1 Comparison

import subprocess
import sys
import io
import json
import base64
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from rdkit.Chem import Draw

try:
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots
except ImportError:
    print("🔧 Installing missing dependencies (plotly, kaleido) via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "plotly", "kaleido"], capture_output=True)
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots
    print("✅ Repair complete.")

try:
    import kaleido
except ImportError:
    print("🔧 Installing/reloading missing kaleido via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "kaleido"], capture_output=True)
    print("✅ Kaleido installation complete.")

# Ensure Plotly renders natively in Colab
if 'google.colab' in sys.modules:
    pio.renderers.default = 'colab'

if 'to_plotly_color' not in globals():
    def to_plotly_color(color_val):
        rgba = mcolors.to_rgba(color_val)
        return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"

if 'detected_df' not in globals() or 'chrom' not in globals() or chrom is None:
    print("⚠️ Required data (`detected_df`, `chrom`) not found. Please run Cell 4 first.")
else:
    def _peak_comparison_label(p_id):
        """Human-readable label for a peak_id, reused by the chromatogram legend and
        the MS1 spectra comparison selector/subplot titles."""
        if 'peak_assignments' in globals() and p_id in peak_assignments:
            a_info = peak_assignments[p_id]
            iso_short = " [Isobaric Isomers]" if a_info["is_isobaric"] else ""
            adduct_summary = ", ".join([f"{x['adduct']} ({x['obs_mz']:.2f})" for x in a_info['adducts']])
            return f"Peak {p_id}: {a_info['display_name']}{iso_short} [{adduct_summary}] (Conf: {a_info['confidence']:.0f}%)"
        if 'detected_df' in globals():
            dom_rows = detected_df.loc[detected_df['peak_id'] == p_id, 'dominant m/z']
            if len(dom_rows):
                dom_str = dom_rows.values[0]
                first_ion = str(dom_str).split(',')[0] if dom_str != "None" else "N/A"
                return f"Peak {p_id}: Unassigned (Base m/z: {first_ion})"
        return f"Peak {p_id}: Unassigned"

    # --- 1. Build Clean Interactive Plotly Chromatogram with Floating Hover Structure Cards ---
    FIG_WIDTH_PX, FIG_HEIGHT_PX = 1300, 800
    MARGIN_L, MARGIN_R, MARGIN_T, MARGIN_B = 80, 40, 65, 210
    X_RANGE = [190, 310]

    time_arr = chrom.df[chrom.time_col].values
    raw_signal = chrom.df[chrom.int_col].values
    convolved = np.sum(chrom.unmixed_chromatograms, axis=1) if chrom.peaks is not None else None
    y_max_plot = float(np.max(raw_signal))
    prom_val = globals().get('current_prominence_val', 0.05)

    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    # Pre-build rich HTML hover cards (including Base64-encoded RDKit 2D structure PNGs) per peak
    hover_cards_by_peak = {}
    qc_struct_images = {}

    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        norm_area = peak_row.get('normalised area', 'N/A')
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            assign = peak_assignments[p_id]
            mol_images = [Draw.MolToImage(m, size=(175, 145)) for m in assign["mols"]]
            if len(mol_images) > 1:
                total_w = sum(im.width for im in mol_images)
                struct_img = Image.new('RGBA', (total_w, mol_images[0].height), (255, 255, 255, 255))
                x_off = 0
                for im in mol_images:
                    struct_img.paste(im, (x_off, 0))
                    x_off += im.width
            else:
                struct_img = mol_images[0]

            qc_struct_images[p_id] = struct_img

            buf = io.BytesIO()
            struct_img.save(buf, format='PNG')
            b64_png = base64.b64encode(buf.getvalue()).decode('utf-8')

            adduct_lines = "<br>".join([
                f"<b>{a['adduct']}</b>: {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ])
            iso_badge = (
                f"<div style='color:#b45309; font-weight:600; font-size:11px; margin-top:2px;'>"
                f"⚠️ Isobaric Isomers ({assign['formula']})</div>"
            ) if assign["is_isobaric"] else ""

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:10px; background:#ffffff; "
                f"padding:10px 12px; box-shadow:0 8px 20px rgba(0,0,0,0.18); font-family:sans-serif; "
                f"color:#111827; min-width:220px; max-width:390px; text-align:center;'>"
                f"<div style='font-weight:700; font-size:13px; border-bottom:1px solid #e5e7eb; "
                f"padding-bottom:4px; margin-bottom:4px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s): {assign['display_name']}</div>"
                f"{iso_badge}"
                f"<div style='font-size:11.5px; color:#374151; line-height:1.45; margin-bottom:6px;'>"
                f"{adduct_lines}<br>"
                f"<b>Area:</b> {norm_area} &nbsp;|&nbsp; "
                f"<b>Confidence:</b> <span style='color:#047857; font-weight:700;'>{assign['confidence']:.1f}%</span>"
                f"</div>"
                f"<img src='data:image/png;base64,{b64_png}' "
                f"style='max-width:100%; height:auto; display:block; margin:0 auto; border-radius:4px;'/>"
                f"</div>"
            )
        else:
            dom_str = str(peak_row.get('dominant m/z', 'None'))
            dom_tokens = [x.strip() for x in dom_str.split(',') if x.strip() and x.strip() != 'None']
            top_ions_str = "<br>".join(dom_tokens[:4]) if dom_tokens else "N/A"

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:8px; background:#f9fafb; "
                f"padding:8px 12px; box-shadow:0 6px 16px rgba(0,0,0,0.15); font-family:sans-serif; "
                f"color:#1f2937; min-width:190px; text-align:left;'>"
                f"<div style='font-weight:700; font-size:12.5px; border-bottom:1px solid #d1d5db; "
                f"padding-bottom:3px; margin-bottom:5px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s) — Unassigned</div>"
                f"<div style='font-size:11px; color:#4b5563; line-height:1.4;'>"
                f"<b>Norm. Area:</b> {norm_area}<br>"
                f"<b>Dominant m/z:</b><br>{top_ions_str}</div>"
                f"</div>"
            )

    chrom_fig = go.Figure()

    # Raw & convolved traces
    chrom_fig.add_trace(go.Scatter(
        x=time_arr, y=raw_signal, mode='lines', name='raw chromatogram',
        line=dict(color='black', width=1.6),
        hovertemplate='<b>Raw Chromatogram</b><br>RT=%{x:.2f}s<br>Signal=%{y:.3e}<extra></extra>'
    ))
    if convolved is not None:
        chrom_fig.add_trace(go.Scatter(
            x=time_arr, y=convolved, mode='lines', name='inferred mixture',
            line=dict(color='red', width=1.4, dash='dash'),
            hovertemplate='<b>Inferred Mixture</b><br>RT=%{x:.2f}s<br>Mixture=%{y:.3e}<extra></extra>'
        ))

    # Deconvoluted peak traces — tagged with peak_id in `meta` so the floating hover card triggers anywhere on the peak
    if chrom.peaks is not None:
        for g, _ in chrom.peaks.groupby('peak_id'):
            p_id = int(g)
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            component = chrom.unmixed_chromatograms[:, p_id - 1]
            legend_label = _peak_comparison_label(p_id)
            chrom_fig.add_trace(go.Scatter(
                x=time_arr, y=component, mode='lines', fill='tozeroy',
                line=dict(color=peak_color, width=1.0), opacity=0.55,
                name=legend_label,
                meta=p_id,
                hoverinfo='none'
            ))

    shapes, compact_annotations = [], []

    # Add subtle drop-lines, compact "P1, P2..." badges, and interactive apex hover markers
    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        c_idx = int(np.argmin(np.abs(time_arr - rt_apex)))
        apex_y = float(raw_signal[c_idx])
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        shapes.append(dict(
            type='line', x0=rt_apex, x1=rt_apex, y0=0, y1=apex_y,
            line=dict(color='dodgerblue', width=1.1, dash='dot'), opacity=0.55
        ))

        # Small, non-intrusive badge right above the peak apex
        compact_annotations.append(dict(
            x=rt_apex, y=apex_y + y_max_plot * 0.035, xref='x', yref='y',
            text=f"<b>P{p_id}</b>",
            showarrow=False,
            font=dict(size=10, color='#111827'),
            bgcolor='rgba(255,255,255,0.88)',
            bordercolor=peak_color,
            borderwidth=1.5,
            borderpad=2
        ))

        # Apex marker so hovering directly on the peak top always triggers that peak's structure card
        chrom_fig.add_trace(go.Scatter(
            x=[rt_apex], y=[apex_y],
            mode='markers',
            marker=dict(size=9, color=peak_color, line=dict(width=1.5, color='white')),
            showlegend=False,
            meta=p_id,
            hoverinfo='none'
        ))

    chrom_fig.update_layout(
        shapes=shapes,
        annotations=compact_annotations,
        title=dict(
            text=(
                f"HPLC Chromatogram Deconvolution & Isobaric Screening (Prominence: {prom_val:.4f}) "
                f"— Hover over any peak to view 2D structure & MS1 details"
            ),
            font=dict(size=15, color='#111827')
        ),
        xaxis=dict(
            title='Retention Time (seconds)',
            range=X_RANGE,
            rangeslider=dict(visible=True, thickness=0.10)
        ),
        yaxis=dict(
            title='Total Ion Count (baseline corrected)',
            range=[0, y_max_plot * 1.18]
        ),
        legend=dict(
            orientation='h',
            x=0.5, y=-0.26,
            xanchor='center', yanchor='top',
            font=dict(size=10),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#d1d5db',
            borderwidth=1
        ),
        margin=dict(l=MARGIN_L, r=MARGIN_R, t=MARGIN_T, b=MARGIN_B),
        template='plotly_white',
        width=FIG_WIDTH_PX,
        height=FIG_HEIGHT_PX,
        hovermode='closest'
    )

    # Render interactive HTML with custom floating hover card listener
    plot_div_id = "hplc_interactive_chrom_div"
    fig_html = chrom_fig.to_html(include_plotlyjs='cdn', full_html=False, div_id=plot_div_id)
    cards_json = json.dumps({str(k): v for k, v in hover_cards_by_peak.items()})

    custom_hover_html = f"""
    <div style="position:relative; width:{FIG_WIDTH_PX}px;">
        {fig_html}
        <div id="{plot_div_id}_tooltip" style="display:none; position:fixed; pointer-events:none; z-index:99999; transition: opacity 0.08s ease-in-out;"></div>
    </div>
    <script>
    (function() {{
        var cards = {cards_json};
        var gd = document.getElementById('{plot_div_id}');
        var tooltip = document.getElementById('{plot_div_id}_tooltip');

        function attachHoverEvents() {{
            if (!gd || typeof gd.on !== 'function') {{
                setTimeout(attachHoverEvents, 150);
                return;
            }}
            gd.on('plotly_hover', function(data) {{
                if (!data || !data.points || data.points.length === 0) return;
                var pt = data.points[0];
                var p_id = pt.data.meta;
                if (p_id && cards[String(p_id)]) {{
                    tooltip.innerHTML = cards[String(p_id)];
                    tooltip.style.display = 'block';
                    if (data.event) {{
                        var x = data.event.clientX + 18;
                        var y = data.event.clientY - 30;
                        var rect = tooltip.getBoundingClientRect();
                        if (x + rect.width > window.innerWidth - 15) {{
                            x = data.event.clientX - rect.width - 18;
                        }}
                        if (y + rect.height > window.innerHeight - 15) {{
                            y = window.innerHeight - rect.height - 15;
                        }}
                        if (y < 10) y = 10;
                        tooltip.style.left = x + 'px';
                        tooltip.style.top = y + 'px';
                    }}
                }}
            }});
            gd.on('plotly_unhover', function() {{
                tooltip.style.display = 'none';
            }});
            gd.addEventListener('mousemove', function(e) {{
                if (tooltip.style.display === 'block') {{
                    var x = e.clientX + 18;
                    var y = e.clientY - 30;
                    var rect = tooltip.getBoundingClientRect();
                    if (x + rect.width > window.innerWidth - 15) {{
                        x = e.clientX - rect.width - 18;
                    }}
                    if (y + rect.height > window.innerHeight - 15) {{
                        y = window.innerHeight - rect.height - 15;
                    }}
                    if (y < 10) y = 10;
                    tooltip.style.left = x + 'px';
                    tooltip.style.top = y + 'px';
                }}
            }});
        }}
        attachHoverEvents();
    }})();
    </script>
    """
    display(HTML(custom_hover_html))

    # Cache a static PNG snapshot via kaleido for the Multimodal Vision QC cell (Cell 9).
    # We place the structure cards in a clean top-row banner (above y_max_plot) solely for the
    # static QC snapshot so Gemini's vision model can still inspect the 2D structures without
    # obscuring the peaks!
    try:
        qc_fig = go.Figure(chrom_fig)
        qc_images = []
        qc_annotations = list(compact_annotations)
        banner_xs = [202.0, 226.0, 250.0, 274.0, 298.0]
        for idx_p, (p_id, struct_img) in enumerate(sorted(qc_struct_images.items())):
            bx = banner_xs[idx_p % len(banner_xs)]
            by = y_max_plot * 1.48
            assign = peak_assignments[p_id]
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            qc_annotations.append(dict(
                x=bx, y=by + y_max_plot * 0.04, xref='x', yref='y',
                text=f"<b>Peak {p_id}: {assign['display_name']}</b> ({assign['confidence']:.0f}%)",
                showarrow=False, font=dict(size=9.5, color='#111827'),
                bgcolor='white', bordercolor=peak_color, borderwidth=1.5
            ))
            qc_images.append(dict(
                source=struct_img, xref='x', yref='y',
                x=bx, y=by,
                sizex=18.0, sizey=y_max_plot * 0.32,
                xanchor='center', yanchor='top', layer='above'
            ))
        qc_fig.update_layout(
            yaxis=dict(title='Total Ion Count (baseline corrected)', range=[0, y_max_plot * 1.62]),
            annotations=qc_annotations,
            images=qc_images
        )
        chromatogram_png_bytes = qc_fig.to_image(format='png', scale=2)
        globals().setdefault('_notebook_figure_registry', {})['deconvoluted_chromatogram_with_structures'] = chromatogram_png_bytes
    except Exception as e:
        print(f"⚠️ Could not rasterize chromatogram for vision QC (kaleido missing?): {e}")

    globals()['_last_chromatogram_figure'] = chrom_fig

    # --- 2. Interactive Multi-Peak MS1 Stick-Spectrum Comparison Widget ---
    ms_peak_selector = widgets.SelectMultiple(
        options=[(_peak_comparison_label(p_id), p_id) for p_id in sorted(peak_ms1_spectra.keys())],
        description='Compare Peaks:',
        rows=6,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='600px')
    )

    ms_output_area = widgets.Output()

    def build_ms_comparison_figure(selected_peak_ids):
        """Builds a stacked, zoom-linked Plotly figure overlaying the MS1 stick spectrum of
        every selected peak so isotope envelopes, co-eluting isomers and fine (sub-0.01 Da)
        mass differences can be inspected side-by-side."""
        mpl_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
        n = max(len(selected_peak_ids), 1)

        fig_ms = make_subplots(
            rows=n, cols=1, shared_xaxes=True,
            vertical_spacing=min(0.12, 0.6 / n) if n > 1 else 0.0,
            subplot_titles=[_peak_comparison_label(p) for p in selected_peak_ids]
        )

        for row_i, p_id in enumerate(selected_peak_ids, start=1):
            plotly_color_val = to_plotly_color(mpl_colors[(p_id - 1) % len(mpl_colors)])
            spectrum = peak_ms1_spectra.get(p_id, {})

            if spectrum:
                mzs = np.array(sorted(spectrum.keys()))
                intens = np.array([spectrum[m] for m in mzs])
                max_int = intens.max()
                rel = (intens / max_int * 100.0) if max_int > 0 else intens

                stick_x = np.repeat(mzs, 3)
                stick_y = np.zeros(stick_x.shape, dtype=float)
                stick_y[1::3] = rel
                stick_y[2::3] = np.nan

                fig_ms.add_trace(
                    go.Scatter(
                        x=stick_x, y=stick_y, mode='lines',
                        line=dict(color=plotly_color_val, width=1.3),
                        hoverinfo='skip', showlegend=False
                    ),
                    row=row_i, col=1
                )
                fig_ms.add_trace(
                    go.Scatter(
                        x=mzs, y=rel, mode='markers',
                        marker=dict(color=plotly_color_val, size=5),
                        customdata=intens, showlegend=False,
                        hovertemplate=(
                            f"Peak {p_id}<br>m/z=%{{x:.4f}}<br>rel. int.=%{{y:.1f}}%"
                            "<br>abs. int.=%{customdata:.3e}<extra></extra>"
                        )
                    ),
                    row=row_i, col=1
                )

                if p_id in peak_assignments:
                    for adduct in peak_assignments[p_id]['adducts'][:3]:
                        fig_ms.add_vline(
                            x=adduct['calc_mz'], line=dict(color=plotly_color_val, width=1.0, dash='dot'),
                            annotation_text=f"{adduct['adduct']} calc.", annotation_font_size=8,
                            annotation_position="bottom", row=row_i, col=1
                        )
            else:
                fig_ms.add_annotation(
                    text="No MS1 signal captured in this peak's elution window",
                    x=0.5, y=0.5, xref=f'x{row_i} domain', yref=f'y{row_i} domain',
                    showarrow=False, font=dict(color='#9ca3af', size=11)
                )

            fig_ms.update_yaxes(title_text="Rel. Int. (%)", range=[0, 120], row=row_i, col=1)

        fig_ms.update_xaxes(matches='x', title_text="m/z", row=n, col=1)
        fig_ms.update_layout(
            width=FIG_WIDTH_PX,
            height=max(300, 260 * n),
            template='plotly_white',
            title_text="Multi-Peak MS1 Spectra Comparison — Isotope Envelopes, Co-eluting Isomers & Fine Mass Differences",
            margin=dict(t=90, b=50),
            hovermode='closest'
        )
        return fig_ms

    def render_ms_comparison(change=None):
        with ms_output_area:
            clear_output(wait=True)
            selected = list(ms_peak_selector.value)
            if not selected:
                print("Select one or more peaks above to overlay/stack their MS1 spectra for comparison.")
                return
            build_ms_comparison_figure(selected).show()

    ms_peak_selector.observe(render_ms_comparison, names='value')

    display(widgets.HTML(
        "<b>🔬 Multi-Peak MS1 Spectra Comparison</b> — select 2+ peaks below to overlay/stack their "
        "MS1 stick spectra on a shared, zoomable m/z axis and inspect isotope envelopes, co-eluting "
        "isomers, and fine mass differences side-by-side."
    ))
    display(ms_peak_selector)
    display(ms_output_area)

    # Pre-select the first 3 peaks and trigger initial render
    preselected = tuple(p_id for _, p_id in ms_peak_selector.options[:3])
    ms_peak_selector.value = preselected
    if preselected:
        render_ms_comparison()



First, add a new code cell to install the official Google GenAI SDK, the MCP client, and the RDKit MCP server package:

**Install Gemini SDK**

- Install the official Google GenAI SDK and RDKit/MCP dependencies in Colab.
- Enable the notebook tool-calling cells below.


In [ ]:
#@title 6. Install the RDKit MCP Bridge and Server

!uv pip install --system -q google-genai mcp rdkit

Since spinning up a separate JSON-RPC subprocess server inside a transient Colab cell can sometimes fight with Colab's I/O structure, the most robust way to give Gemini "MCP-like" tool-use capabilities in a notebook is to define the tools explicitly using Gemini's native function-calling API (google-genai SDK):

**RDKit tools**

- Validate SMILES, calculate masses and generate structure/fragment images.
- Expose deterministic peak and adduct checks before using Gemini.


In [ ]:
#@title 7. Self-Healing RDKit Tools, Automated Fragmenter & 2D SVG Generator
import re
import base64
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from rdkit.Chem.Draw import rdMolDraw2D

# 1. Silence noisy C++ stderr logs from RDKit so Colab output stays clean
RDLogger.DisableLog('rdApp.*')

_rendered_svg_registry = {}

def safe_mol_from_smiles(smiles_or_str: str):
    """
    Robust SMILES parser that automatically repairs common LLM kekulization mistakes
    (e.g., missing [nH] in cleaved purines/indoles/imidazoles like 'Nc1ncnc2c1ncn2').
    Returns (mol, canonical_smiles, error_msg).
    """
    s = smiles_or_str.strip()
    if not s:
        return None, None, "Empty input string."

    # Check if the model accidentally passed a Molecular Formula (e.g., 'C10H13N5O4')
    if re.fullmatch(r'([A-Z][a-z]?\d*)+([+-]\d*)?', s) and not any(c in s for c in ['c', 'n', 'o', 's', '=', '#', '(', ')', '[']):
        if any(char.isdigit() for char in s) and not re.search(r'[1-9].*[1-9]', s[:3]):
            return None, None, (
                f"'{s}' is a molecular formula, not a SMILES string. "
                "Use `calculate_formula_mass` for formulas, or pass a valid SMILES (e.g. 'Nc1ncnc2c1nc[nH]2' for adenine)."
            )

    # Attempt 1: Standard RDKit parse
    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        return mol, Chem.CanonSmiles(s), None

    # Attempt 2: Auto-repair unkekulized pyrrole/purine/imidazole nitrogens (missing [nH])
    mol_no_san = Chem.MolFromSmiles(s, sanitize=False)
    if mol_no_san is not None:
        try:
            mol_no_san.UpdatePropertyCache(strict=False)
            Chem.GetSymmSSSR(mol_no_san)
            for atom in mol_no_san.GetAtoms():
                # Aromatic N with degree 2, 0 formal charge, and 0 explicit H often needs 1 H in 5-membered rings
                if atom.GetAtomicNum() == 7 and atom.GetIsAromatic() and atom.GetDegree() == 2 and atom.GetFormalCharge() == 0 and atom.GetNumExplicitHs() == 0:
                    test_mol = Chem.Mol(mol_no_san)
                    test_mol.GetAtomWithIdx(atom.GetIdx()).SetNumExplicitHs(1)
                    try:
                        Chem.SanitizeMol(test_mol)
                        fixed_smi = Chem.MolToSmiles(test_mol)
                        return test_mol, fixed_smi, None
                    except Exception:
                        continue
        except Exception:
            pass

    return None, None, (
        f"RDKit could not parse or kekulize '{s}'. "
        "Tip: In aromatic 5-membered rings (purines, adenine, histidine, indoles), "
        "explicitly write '[nH]' for the protonated ring nitrogen (e.g., adenine is 'Nc1ncnc2c1nc[nH]2')."
    )


def get_user_candidate_molecules() -> str:
    """Returns the user's input table of candidate molecules, classifications, SMILES, formulas, and exact masses."""
    candidates = get_candidates_from_table() if 'get_candidates_from_table' in globals() else []
    if not candidates:
        return "No candidate molecules found."

    rows = []
    for name, classification, smi in candidates:
        mol, clean_smi, _ = safe_mol_from_smiles(smi)
        if mol:
            em = Descriptors.ExactMolWt(mol)
            formula = rdMolDescriptors.CalcMolFormula(mol)
            rows.append({
                "Name": name,
                "Classification": classification,
                "SMILES": clean_smi,
                "Formula": formula,
                "Exact_Mass": round(em, 4),
                "[M+H]+_mz": round(em + 1.007276, 4)
            })
    return pd.DataFrame(rows).to_markdown(index=False)


def get_detected_peaks_summary() -> str:
    """Returns a full tabular summary of all deconvoluted HPLC-MS peaks, retention times, areas, dominant m/z ions, and assignments."""
    if 'detected_df' not in globals():
        return "Error: detected_df not found in memory. Run the peak detection cell first."

    cols = [c for c in ['peak_id', 'retention_time', 'peak start time', 'peak end time', 'normalised area', 'dominant m/z', 'Assignment', 'Confidence (%)'] if c in detected_df.columns]
    return detected_df[cols].to_markdown(index=False)


def get_peak_details(peak_id: int) -> dict:
    """Retrieves detailed analytical evidence, observed adducts, and raw MS1 ions for a specific peak ID."""
    result = {}
    if 'detected_df' in globals():
        row = detected_df[detected_df['peak_id'] == peak_id]
        if not row.empty:
            r_dict = row.iloc[0].to_dict()
            result["chromatogram_row"] = {k: str(v) for k, v in r_dict.items()}
            # Also query raw spectrum across this peak's start and end window if 'od' is loaded
            rt_start = r_dict.get('peak start time', np.nan)
            rt_end = r_dict.get('peak end time', np.nan)
            if 'od' in globals() and not pd.isna(rt_start) and not pd.isna(rt_end):
                agg = {}
                for spec in od:
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
                        mzs, ints = spec.get_peaks()
                        for m, inten in zip(np.round(mzs, 4), ints):
                            if inten > 1e3:
                                agg[m] = agg.get(m, 0.0) + float(inten)
                if agg:
                    top_ions = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:25]
                    max_i = top_ions[0][1]
                    result["raw_ms1_top_25_ions_in_peak_window"] = [
                        {"mz": float(m), "intensity": round(i, 1), "rel_pct": round((i / max_i) * 100, 2)}
                        for m, i in top_ions
                    ]

    if 'peak_assignments' in globals() and peak_id in peak_assignments:
        assign = peak_assignments[peak_id]
        result["assigned_compound"] = {
            "display_name": assign["display_name"],
            "formula": assign["formula"],
            "confidence": assign["confidence"],
            "evidence": assign["evidence"],
            "adducts_observed": [
                f"{a['adduct']} at {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm, intensity: {a['intensity']:.1e})"
                for a in assign["adducts"]
            ]
        }
    return result if result else {"error": f"Peak {peak_id} not found."}


def check_mz_in_peak(peak_id: int, target_mz: float, tolerance_da: float = 0.05) -> dict:
    """
    Directly searches all raw MS1 scans across a specific peak's retention time window
    to verify whether a specific m/z (e.g. a fragment ion) is present in that peak.
    """
    if 'detected_df' not in globals() or 'od' not in globals():
        return {"error": "detected_df or pyOpenMS 'od' experiment not found in memory."}

    row = detected_df[detected_df['peak_id'] == peak_id]
    if row.empty:
        return {"error": f"Peak {peak_id} not found in detected_df."}

    rt_start = float(row.iloc[0]['peak start time'])
    rt_end = float(row.iloc[0]['peak end time'])
    dom_str = str(row.iloc[0]['dominant m/z'])

    matches = []
    max_scan_base_peak = 1.0
    for spec in od:
        if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
            mzs, ints = spec.get_peaks()
            if len(ints) > 0:
                max_scan_base_peak = max(max_scan_base_peak, float(np.max(ints)))
            mask = (mzs >= target_mz - tolerance_da) & (mzs <= target_mz + tolerance_da)
            if np.any(mask):
                for m, inten in zip(mzs[mask], ints[mask]):
                    matches.append({
                        "rt": round(float(spec.getRT()), 2),
                        "observed_mz": round(float(m), 4),
                        "error_da": round(float(m - target_mz), 4),
                        "ppm_error": round(float((m - target_mz) / target_mz * 1e6), 1),
                        "intensity": round(float(inten), 1)
                    })

    matches = sorted(matches, key=lambda x: x["intensity"], reverse=True)
    return {
        "peak_id": peak_id,
        "rt_window": f"{rt_start:.1f}s - {rt_end:.1f}s",
        "dominant_mz_summary": dom_str,
        "target_mz_searched": target_mz,
        "tolerance_da": tolerance_da,
        "found_in_raw_scans": len(matches) > 0,
        "max_observed_hit": matches[0] if matches else None,
        "top_5_scan_hits": matches[:5]
    }


def generate_in_silico_fragments(parent_smiles: str, check_peak_id: int = 0) -> dict:
    """
    Automatically generates chemically valid, RDKit-sanitized ESI+ fragments for a parent molecule
    (including glycosidic cleavage, neutral losses of H2O/NH3/CO, and single acyclic bond cleavages).
    Optionally checks if each fragment m/z is observed in `check_peak_id`.
    """
    mol, clean_parent, err = safe_mol_from_smiles(parent_smiles)
    if not mol:
        return {"error": err}

    parent_em = Descriptors.ExactMolWt(mol)
    parent_formula = rdMolDescriptors.CalcMolFormula(mol)
    parent_mh = round(parent_em + 1.007276, 4)

    fragments_dict = {}

    def _add_frag(frag_smi, pathway_label, is_intrinsic_cation=False):
        f_mol, f_clean, _ = safe_mol_from_smiles(frag_smi)
        if not f_mol:
            return
        em = Descriptors.ExactMolWt(f_mol)
        if em < 40.0 or em >= parent_em - 1.0:
            return
        form = rdMolDescriptors.CalcMolFormula(f_mol)
        f_charge = Chem.GetFormalCharge(f_mol)
        if is_intrinsic_cation or f_charge > 0:
            mz_val = round(em - 0.00054858, 4)
            ion_type = "[F]+ (oxocarbenium / intrinsic cation)"
        else:
            mz_val = round(em + 1.007276, 4)
            ion_type = "[F+H]+"

        key = (form, round(mz_val, 3))
        if key not in fragments_dict:
            entry = {
                "fragment_name": pathway_label,
                "smiles": f_clean,
                "formula": form,
                "neutral_exact_mass": round(em, 4),
                "ion_type": ion_type,
                "expected_esi_pos_mz": mz_val,
                "neutral_loss_from_parent_Da": round(parent_mh - mz_val, 4)
            }
            if check_peak_id > 0:
                hit_info = check_mz_in_peak(check_peak_id, mz_val, tolerance_da=0.05)
                entry["present_in_peak"] = hit_info.get("found_in_raw_scans", False)
                entry["peak_observation"] = hit_info.get("max_observed_hit", None)
            fragments_dict[key] = entry

    # 1. Check canonical nucleoside / riboside pathways if ribose/deoxyribose is present
    ribose_smarts = Chem.MolFromSmarts("[C@@H,C@H,CH]1O[C@H,C@@H,CH](CO)[C@@H,C@H,CH](O)[C@H,C@@H,CH]1")
    if mol.HasSubstructMatch(ribose_smarts):
        # Classic ribose oxocarbenium ion [C5H7O3]+ and dehydrated derivatives
        _add_frag("OC[C@H]1O[C+]=[C@H][C@@H]1O", "Ribose oxocarbenium fragment [C5H7O3]+", is_intrinsic_cation=True)
        _add_frag("O=CC1=CC(=O)CC1", "Dehydrated ribose fragment (C5H5O2+)", is_intrinsic_cation=False)

    # 2. Cleave all single non-ring bonds using RDKit FragmentOnBonds
    cleavable_bonds = []
    for bond in mol.GetBonds():
        if not bond.IsInRing() and bond.GetBondType() == Chem.BondType.SINGLE:
            cleavable_bonds.append(bond.GetIdx())

    for b_idx in cleavable_bonds:
        try:
            frag_mol = Chem.FragmentOnBonds(mol, [b_idx], addDummies=False)
            frags = Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False)
            for f in frags:
                raw_smi = Chem.MolToSmiles(f)
                _add_frag(raw_smi, "Single-bond cleavage fragment ([F+H]+)")
        except Exception:
            pass

    # 3. Common nucleoside cross-ring and base-loss pathways
    if "ncnc2c1ncn2" in clean_parent or "ncnc2c1nc[nH]2" in clean_parent:
        _add_frag("Nc1ncnc2c1nc[nH]2", "Protonated Adenine [B+H]+ (glycosidic cleavage, -Ribose -132.04 Da)")
        _add_frag("c1ncnc2c1nc[nH]2", "Purine core after NH3 loss from Adenine ([B+H-NH3]+, m/z 120.04)")
        _add_frag("Nc1ncnc2c1ncn2C=C", "0,2X Cross-ring cleavage (Adenine + C2H2O/vinyl remnant)")
        _add_frag("Nc1ncnc2c1ncn2CC=O", "0,2X Cross-ring ribose cleavage (Adenine + C2H3O, [M+H-C3H6O3]+)")
        _add_frag("Nc1ncnc2c1ncn2[C@@H](O)C=O", "0,3X Cross-ring ribose cleavage (Adenine + C2H3O2, [M+H-C3H6O2]+)")

    sorted_frags = sorted(fragments_dict.values(), key=lambda x: x["expected_esi_pos_mz"], reverse=True)
    return {
        "parent_smiles": clean_parent,
        "parent_formula": parent_formula,
        "parent_exact_mass": round(parent_em, 4),
        "parent_[M+H]+_mz": parent_mh,
        "generated_fragments": sorted_frags
    }


def analyze_structure_and_mass(smiles_or_formula: str, target_mz: float = 0.0) -> dict:
    """
    Calculates exact monoisotopic mass, molecular formula, and ESI m/z values ([M]+, [M+H]+, [M+Na]+)
    from EITHER a valid SMILES string OR a molecular formula (e.g., 'C5H5N5').
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles_or_formula)
    if mol is None:
        # Try evaluating as a chemical formula directly so it never crashes
        try:
            em = rdMolDescriptors.CalcExactMolWt(smiles_or_formula.strip())
            return {
                "input_interpreted_as": "Molecular Formula (no 2D topology)",
                "formula": smiles_or_formula.strip(),
                "exact_monoisotopic_mass": round(em, 5),
                "calculated_mz_species": {
                    "intrinsic_or_radical_[M]+": round(em - 0.00055, 4),
                    "protonated_[M+H]+": round(em + 1.007276, 4),
                    "sodiated_[M+Na]+": round(em + 22.989218, 4)
                },
                "note": err
            }
        except Exception:
            return {"error": err}

    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    formal_charge = Chem.GetFormalCharge(mol)

    mz_intrinsic = exact_mass - (formal_charge * 0.00054858) if formal_charge != 0 else exact_mass
    mz_mh_plus = exact_mass + 1.007276 if formal_charge == 0 else mz_intrinsic

    res = {
        "canonical_smiles": clean_smi,
        "formula": formula,
        "formal_charge": formal_charge,
        "exact_monoisotopic_mass": round(exact_mass, 5),
        "calculated_mz_species": {
            "intrinsic_or_radical_[M]+": round(exact_mass - 0.00055, 4),
            "protonated_[M+H]+": round(mz_mh_plus, 4),
            "sodiated_[M+Na]+": round(exact_mass + 22.989218, 4) if formal_charge == 0 else "N/A"
        }
    }
    if target_mz > 0:
        diff_da = mz_mh_plus - target_mz
        res["target_mz_comparison"] = {
            "target_mz": target_mz,
            "error_Da": round(diff_da, 4),
            "ppm_error": round((diff_da / target_mz) * 1e6, 1)
        }
    return res


def generate_2d_structure_svg(smiles: str, label: str = "", width: int = 280, height: int = 210) -> dict:
    """
    Uses RDKit to generate a 2D chemical structure SVG from a SMILES string.
    Stores the SVG in memory and returns a short placeholder token (e.g. '{{SVG_1}}')
    that you place in your response where the image should appear.
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles)
    if not mol:
        return {"error": err}

    AllChem.Compute2DCoords(mol)
    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addStereoAnnotation = True
    opts.padding = 0.12

    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    mh_mz = exact_mass + 1.007276 if Chem.GetFormalCharge(mol) == 0 else exact_mass
    legend_text = f"{label} | {formula} ([M+H]+: {mh_mz:.4f})" if label else f"{formula} (Exact: {exact_mass:.4f})"

    drawer.DrawMolecule(mol, legend=legend_text)
    drawer.FinishDrawing()
    svg_str = drawer.GetDrawingText()

    # Strip XML header so it embeds cleanly inline in HTML
    svg_clean = re.sub(r"<\?xml.*?\?>", "", svg_str, flags=re.DOTALL).strip()

    token_id = f"{{{{SVG_{len(_rendered_svg_registry) + 1}}}}}"
    _rendered_svg_registry[token_id] = (
        f"<div style='display:inline-block; margin:8px; padding:8px; border:1px solid #d1d5db; "
        f"border-radius:8px; background:white; text-align:center; box-shadow: 0 1px 3px rgba(0,0,0,0.08);'>"
        f"{svg_clean}</div>"
    )

    return {
        "status": "success",
        "canonical_smiles_drawn": clean_smi,
        "formula": formula,
        "exact_mass": round(exact_mass, 4),
        "expected_mh_mz": round(mh_mz, 4),
        "svg_placeholder_token": token_id,
        "instruction": f"Insert the exact token {token_id} on its own line in your response where you want this 2D structure displayed. Do NOT write raw SVG or base64."
    }

By providing these specific functions, analytical chemists can ask Gemini complex, multi-step questions directly in the chat interface, such as:

* *"What is the confidence score for the peak containing a pyrimidine ring?" (Gemini will call find_peaks_by_substructure, then get_peak_details).* *italicised text*

*  *"If I spiked this sample with Caffeine (CN1C=NC2=C1C(=O)N(C(=O)N2C)C), what [M+Na]+ m/z would I look for?" (Gemini will call calculate_expected_adduct_mz).*

* *Identify the unassigned peaks using PubChem and ChEMBL, resolve any isobaric isomers, and replot the interactive HPLC chromatogram with the new peaks assigned.*


You can pass these RDKit functions directly to the Gemini API so the model can autonomously invoke them when answering your natural language cheminformatics questions:

**Interactive synthetic LCMS workflow**

- Compute adducts, reaction mass-shift hypotheses, isotope envelopes, contaminants and neutral losses locally; retain raw MS1 scans only in Python.
- Detect and background-subtract the six strongest MS1 peaks; send only a compact summary to Gemini Flash (Pro is optional for difficult elucidation).
- Upload an optional UV/DAD CSV with a Time column in minutes and wavelength columns; display UV area percentages above toggleable TIC/BPC, with a configurable UV-to-MS delay.
- Use the local dashboard for XIC/scan/isotope overlays; cache and parallelize trimmed PubChem, ChEMBL and MassBank searches.


In [ ]:
#@title 8A. Imports, Global Registries & RDKit SVG Helpers

import re
import io
import json
import uuid
import base64
import html
import math
import sqlite3
import urllib.request
import urllib.parse
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import pubchempy as pcp
from scipy.integrate import trapezoid
from scipy.signal import find_peaks, peak_widths
from google.colab import userdata
from google import genai
from google.genai import types
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, Crippen, Draw
from rdkit.Chem.Draw import rdMolDraw2D

# 1. Silence RDKit C++ warnings
RDLogger.DisableLog('rdApp.*')

# Global dictionary mapping placeholder tokens ('[[MOL_IMG_X]]', '{{SVG_X}}', '[[PLOTLY_CHROM_X]]') to HTML blocks

# Preserve existing registries across cell re-runs so you don't lose SVGs/state
_rendered_svg_registry = globals().setdefault('_rendered_svg_registry', {})
SPECTRAL_STATE = globals().setdefault('SPECTRAL_STATE', {})
CHEM_CACHE_PATH = "/tmp/lcms_chem_cache.sqlite"
BENCH_CONTAMINANTS = {
    "TPPO": 279.093, "DIPEA": 130.159, "Et3N": 102.128,
    "DMAP": 123.092, "DCU": 225.196, "DIU": 145.134,
    "dibutyl phthalate": 279.159,
}
NEUTRAL_LOSSES = {
    "H2O": 18.0106, "NH3": 17.0265, "CO/N2": 28.0061,
    "HCl": 35.9767, "HCl (average)": 36.46, "CO2": 43.9898, "isobutene": 56.0626,
    "Boc": 100.0524,
}

if 'to_plotly_color' not in globals():
    def to_plotly_color(color_val):
        rgba = mcolors.to_rgba(color_val)
        return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"


def _register_mol_svg(mol, label: str = "", width: int = 300, height: int = 220) -> str:
    """
    Renders an RDKit Mol object to a Base64-encoded SVG <img> HTML card,
    stores it in `_rendered_svg_registry` under both `[[MOL_IMG_N]]` and `{{SVG_N}}`,
    and returns the token `[[MOL_IMG_N]]`.
    """
    mol_copy = Chem.Mol(mol)
    AllChem.Compute2DCoords(mol_copy)

    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addStereoAnnotation = True
    opts.padding = 0.12

    exact_mass = Descriptors.ExactMolWt(mol_copy)
    formula = rdMolDescriptors.CalcMolFormula(mol_copy)
    f_charge = Chem.GetFormalCharge(mol_copy)
    mh_mz = exact_mass + 1.007276 if f_charge == 0 else exact_mass - (f_charge * 0.00054858)

    drawer.DrawMolecule(mol_copy)
    drawer.FinishDrawing()
    svg_str = drawer.GetDrawingText()

    b64_svg = base64.b64encode(svg_str.encode('utf-8')).decode('utf-8')
    mol_count = sum(1 for k in _rendered_svg_registry if k.startswith("[[MOL_IMG_")) + 1

    caption_title = f"<div style='font-weight:700; font-size:12px; color:#111827; margin-bottom:4px;'>{label}</div>" if label else ""
    caption_sub = f"<div style='font-size:11px; color:#374151; margin-top:4px;'><b>{formula}</b> | Exact: {exact_mass:.4f} | m/z: {mh_mz:.4f}</div>"

    html_card = (
        f"<div style='display:inline-block; margin:10px 6px; padding:10px 14px; border:2px solid #cbd5e1; "
        f"border-radius:10px; background:#ffffff; text-align:center; box-shadow:0 2px 6px rgba(0,0,0,0.15);'>"
        f"{caption_title}"
        f"<img src='data:image/svg+xml;base64,{b64_svg}' width='{width}' height='{height}' style='display:block; margin:0 auto; background:#ffffff;'/>"
        f"{caption_sub}"
        f"</div>"
    )

    token_primary = f"[[MOL_IMG_{mol_count}]]"
    token_alt = f"{{{{SVG_{mol_count}}}}}"
    _rendered_svg_registry[token_primary] = html_card
    _rendered_svg_registry[token_alt] = html_card
    return token_primary


def safe_mol_from_smiles(smiles_or_str: str):
    """Auto-repairs unkekulized pyrrole/purine/imidazole nitrogens (missing [nH])."""
    s = smiles_or_str.strip()
    if not s:
        return None, None, "Empty input string."

    if re.fullmatch(r'([A-Z][a-z]?\d*)+([+-]\d*)?', s) and not any(c in s for c in ['c', 'n', 'o', 's', '=', '#', '(', ')', '[']):
        if any(char.isdigit() for char in s) and not re.search(r'[1-9].*[1-9]', s[:3]):
            return None, None, f"'{s}' is a molecular formula, not a SMILES string."

    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        return mol, Chem.CanonSmiles(s), None

    mol_no_san = Chem.MolFromSmiles(s, sanitize=False)
    if mol_no_san is not None:
        try:
            mol_no_san.UpdatePropertyCache(strict=False)
            Chem.GetSymmSSSR(mol_no_san)
            for atom in mol_no_san.GetAtoms():
                if atom.GetAtomicNum() == 7 and atom.GetIsAromatic() and atom.GetDegree() == 2 and atom.GetFormalCharge() == 0 and atom.GetNumExplicitHs() == 0:
                    test_mol = Chem.Mol(mol_no_san)
                    test_mol.GetAtomWithIdx(atom.GetIdx()).SetNumExplicitHs(1)
                    try:
                        Chem.SanitizeMol(test_mol)
                        return test_mol, Chem.MolToSmiles(test_mol), None
                    except Exception:
                        continue
        except Exception:
            pass

    return None, None, f"RDKit could not parse '{s}'. Ensure aromatic 5-membered ring NH groups are written as '[nH]'."


In [ ]:
#@title 8B. Local Spectral State, Isotope Math & Multi-Panel LCMS Dashboard
# --- LOCAL SPECTRAL STATE & UV/DAD DASHBOARD HELPERS ---

def calculate_expected_ions(smiles: str) -> dict:
    """Monoisotopic ESI ions (singly charged) from a neutral, valid SMILES."""
    mol, clean, error = safe_mol_from_smiles(smiles)
    if mol is None:
        return {"error": error}
    mass = Descriptors.ExactMolWt(mol)
    positive = {
        "[M+H]+": mass + 1.0073, "[M+NH4]+": mass + 18.0338,
        "[M+Na]+": mass + 22.9892, "[M+K]+": mass + 38.9632,
        "[M+CH3CN+H]+": mass + 42.0338,
        "[2M+H]+": 2 * mass + 1.0073, "[2M+Na]+": 2 * mass + 22.9892,
    }
    negative = {
        "[M-H]-": mass - 1.0073, "[M+Cl]-": mass + 34.9694,
        "[M+HCOO]-": mass + 44.9982, "[M+CH3COO]-": mass + 59.0139,
    }
    return {
        "smiles": clean, "formula": rdMolDescriptors.CalcMolFormula(mol),
        "exact_mass": round(mass, 5),
        "cLogP": round(Crippen.MolLogP(mol), 2),
        "TPSA": round(rdMolDescriptors.CalcTPSA(mol), 1),
        "positive": {k: round(v, 4) for k, v in positive.items()},
        "negative": {k: round(v, 4) for k, v in negative.items()},
    }


def parse_reaction_candidates(text: str) -> list[dict]:
    """Accept SMILES, reaction SMILES, or labeled/positional comma-separated SMILES."""
    text = text.strip()
    if not text:
        return []
    if ">" in text:
        parts = text.split(">")
        if len(parts) != 3:
            return []
        entries = (
            [(s, "Starting Material") for s in parts[0].split(".")] +
            [(s, "Reagent") for s in parts[1].split(".")] +
            [(s, "Target Product") for s in parts[2].split(".")]
        )
    elif "," in text:
        pieces = [s.strip() for s in text.split(",") if s.strip()]
        entries = []
        for i, piece in enumerate(pieces):
            name, sep, smi = piece.partition(":")
            if sep:
                role = {"sm": "Starting Material", "starting material": "Starting Material",
                        "reagent": "Reagent", "reagents": "Reagent",
                        "product": "Target Product", "target product": "Target Product"}.get(name.strip().lower())
                entries.append((smi.strip(), role or "Reagent"))
            else:
                entries.append((piece, "Starting Material" if i == 0 else
                                "Target Product" if i == len(pieces) - 1 else "Reagent"))
    else:
        entries = [(text, "Target Product")]
    return [
        {"role": role, **ions}
        for smi, role in entries if (ions := calculate_expected_ions(smi)) and "error" not in ions
    ]


def reaction_mass_hypotheses(candidates: list[dict]) -> list[dict]:
    """Mass shifts are diagnostic hypotheses, not claimed molecular structures."""
    products = [c for c in candidates if c["role"] in ("Target Product", "Product")]
    if not products:
        return []
    product = products[0]
    hypotheses = []
    product_mol = Chem.MolFromSmiles(product["smiles"]) if product.get("smiles") else None
    for candidate in candidates:
        if candidate["role"] in ("Starting Material", "Reactant"):
            hypotheses.append({
                "name": "Unreacted SM", "mz": candidate["positive"]["[M+H]+"],
                "delta": round(candidate["exact_mass"] - product["exact_mass"], 4),
                "cLogP": candidate["cLogP"], "TPSA": candidate["TPSA"],
            })
    transformations = {
        "Hydrolysis": ("[C]#[N]", "C(=O)N"),
        "Des-Br": ("[Br]", None),
        "Des-Cl": ("[Cl]", None),
        "Protodeboronation": ("[B]([O;H1])[O;H1]", None),
        "Boc cleavage": ("[C](=[O])[O][C]([CH3])([CH3])[CH3]", None),
        "In-source isobutene / t-Bu ester": ("[O][C]([CH3])([CH3])[CH3]", "O"),
    }
    for name, shift in (
        ("Hydrolysis", 18.0106), ("Des-Br", -77.9105),
        ("Des-Cl", -33.9610), ("Protodeboronation", -43.9900),
        ("Boc cleavage", -100.0524), ("In-source isobutene / t-Bu ester", -56.0626),
        ("Fmoc cleavage", -222.0681),
    ):
        if name == "Des-Br" and "Br" not in product["formula"]:
            continue
        if name == "Des-Cl" and "Cl" not in product["formula"]:
            continue
        if name == "Protodeboronation" and "B" not in product["formula"]:
            continue
        mz = product["positive"]["[M+H]+"] + shift
        if mz > 0:
            entry = {"name": name, "mz": round(mz, 4), "delta": shift,
                     "cLogP": None, "TPSA": None}
            if product_mol and name in transformations:
                pattern, replacement = transformations[name]
                query = Chem.MolFromSmarts(pattern)
                if query and product_mol.HasSubstructMatch(query):
                    try:
                        if replacement:
                            altered = Chem.ReplaceSubstructs(product_mol, query,
                                Chem.MolFromSmiles(replacement), replaceAll=False)[0]
                        else:
                            altered = Chem.DeleteSubstructs(product_mol, query, onlyFrags=False)
                        Chem.SanitizeMol(altered)
                        if abs(Descriptors.ExactMolWt(altered) -
                               (product["exact_mass"] + shift)) < .05:
                            entry.update(smiles=Chem.MolToSmiles(altered),
                                cLogP=round(Crippen.MolLogP(altered), 2),
                                TPSA=round(rdMolDescriptors.CalcTPSA(altered), 1))
                    except Exception:
                        pass
            hypotheses.append(entry)
    return hypotheses


def theoretical_isotope_envelope(formula_or_smiles: str) -> list[dict]:
    """Natural-abundance nominal M..M+4 envelope with exact centroid mass offsets."""
    mol, _, _ = safe_mol_from_smiles(formula_or_smiles)
    formula = rdMolDescriptors.CalcMolFormula(mol) if mol is not None else formula_or_smiles.strip()
    matches = list(re.finditer(r"([A-Z][a-z]?)(\d*)", formula))
    if not matches or "".join(m.group(0) for m in matches) != formula:
        raise ValueError("Provide a neutral molecular formula or valid SMILES.")
    isotope = {
        "C": [(0, 0, .9893), (1, 1.003355, .0107)],
        "H": [(0, 0, .999885), (1, 1.006277, .000115)],
        "N": [(0, 0, .99636), (1, .997035, .00364)],
        "O": [(0, 0, .99757), (1, 1.004217, .00038), (2, 2.004245, .00205)],
        "S": [(0, 0, .9499), (1, .999388, .0075), (2, 1.995796, .0425), (4, 3.995010, .0001)],
        "Cl": [(0, 0, .7578), (2, 1.997050, .2422)],
        "Br": [(0, 0, .5069), (2, 1.997953, .4931)],
    }
    pt = Chem.GetPeriodicTable()
    mass = 0.0
    distribution = {0: (1.0, 0.0)}
    for match in matches:
        element, count = match.group(1), int(match.group(2) or 1)
        if pt.GetAtomicNumber(element) == 0 or count > 300:
            raise ValueError("Invalid or oversized molecular formula.")
        mass += pt.GetMostCommonIsotopeMass(element) * count
        for _ in range(count):
            next_dist = {}
            for shift, (prob, weighted) in distribution.items():
                for step, delta, abundance in isotope.get(element, [(0, 0.0, 1.0)]):
                    if shift + step > 4:
                        continue
                    p, w = next_dist.get(shift + step, (0.0, 0.0))
                    next_dist[shift + step] = (p + prob * abundance,
                                               w + (weighted + prob * delta) * abundance)
            distribution = next_dist
    max_prob = max(p for p, _ in distribution.values())
    return [
        {"label": f"M+{shift}" if shift else "M",
         "mz": round(mass + 1.007276 + weighted / prob, 5),
         "relative_intensity": round(100 * prob / max_prob, 2)}
        for shift in (0, 1, 2, 4)
        if (prob := distribution.get(shift, (0, 0))[0]) > 0
        for weighted in [distribution[shift][1]]
    ]


def annotate_neutral_losses(ions: list[dict], tolerance_da: float = 0.04) -> list[dict]:
    """Match neutral losses among the ten most intense baseline-subtracted ions."""
    losses = []
    top = sorted(ions, key=lambda ion: ion["intensity"], reverse=True)[:10]
    for parent in top:
        for fragment in top:
            delta = parent["mz"] - fragment["mz"]
            for name, mass in NEUTRAL_LOSSES.items():
                if abs(delta - mass) <= tolerance_da:
                    losses.append({"loss": name, "parent": parent["mz"], "fragment": fragment["mz"]})
    return losses


def _centroid_scans(scans: list[tuple], rt: float, onset: float) -> list[dict]:
    """Average three nearest MS1 scans; subtract a preceding background scan by m/z bin."""
    if not scans:
        return []
    closest = int(np.argmin([abs(s[0] - rt) for s in scans]))
    selected = scans[max(0, closest - 1):closest + 2]
    def bins(spectrum):
        result = {}
        for mz, intensity in zip(spectrum[1], spectrum[2]):
            key = round(float(mz), 2)
            if intensity > 0:
                total, weighted_mz = result.get(key, (0.0, 0.0))
                result[key] = (total + float(intensity),
                               weighted_mz + float(mz * intensity))
        return result
    averaged = {}
    for scan in selected:
        for mz, (intensity, weighted_mz) in bins(scan).items():
            total, weighted = averaged.get(mz, (0.0, 0.0))
            averaged[mz] = (total + intensity / len(selected),
                            weighted + weighted_mz / len(selected))
    preceding_idx = [i for i, scan in enumerate(scans) if scan[0] < onset]
    if preceding_idx:
        best_bg_idx = min(preceding_idx, key=lambda i: abs(scans[i][0] - (onset - 0.05)))
        background = bins(scans[best_bg_idx])
    else:
        background = {}
    corrected = {mz: (max(0.0, intensity - background.get(mz, (0.0, 0.0))[0]),
                      weighted / intensity)
                 for mz, (intensity, weighted) in averaged.items() if intensity > 0}
    if not corrected:
        return []
    threshold = max(intensity for intensity, _ in corrected.values()) * .02
    return [{"mz": round(centroid, 4), "intensity": round(intensity, 1)}
            for mz, (intensity, centroid) in sorted(corrected.items(),
                                                    key=lambda item: -item[1][0])[:10]
            if intensity >= threshold]


def _flag_artifacts(ions: list[dict]) -> list[str]:
    masses = [ion["mz"] for ion in ions]
    flags = [name for name, mz in BENCH_CONTAMINANTS.items()
             if any(abs(observed - mz) <= .03 for observed in masses)]
    if any(abs(a - b - 44.026) <= .03 for a in masses for b in masses if a > b) or any(
        abs(mz - (18.0106 + n * 44.0262 + adduct)) <= .03
        for mz in masses for n in range(6, 13) for adduct in (1.0073, 22.9892)
    ):
        flags.append("PEG-400")
    return flags


def prepare_spectral_state(max_peaks: int = 6, uv_ms_delay_min: float = 0.08) -> dict:
    """Keep raw arrays locally; only small peak metadata may enter the model context."""
    if not 1 <= max_peaks <= 20 or not math.isfinite(uv_ms_delay_min) or uv_ms_delay_min < 0:
        raise ValueError("Use 1-20 peaks and a nonnegative UV/MS delay.")
    scans = []
    if "od" in globals():
        for spec in od:
            if spec.getMSLevel() == 1:
                mzs, intensities = spec.get_peaks()
                scans.append((float(spec.getRT()) / 60.0, np.asarray(mzs), np.asarray(intensities)))
    scans.sort(key=lambda s: s[0])
    times = np.array([s[0] for s in scans])
    tic = np.array([np.sum(s[2]) for s in scans])
    bpc = np.array([np.max(s[2]) if len(s[2]) else 0 for s in scans])
    if not scans and "hplc_df" in globals():
        times = np.asarray(hplc_df["Time"], dtype=float) / 60.0
        tic = np.asarray(hplc_df["Signal"], dtype=float)
        bpc = np.zeros_like(tic)
    signal = np.maximum(0, tic - np.percentile(tic, 10)) if len(tic) else tic
    indices = find_peaks(signal, prominence=max(float(np.max(signal)) * .05, 1e-10),
                         distance=3)[0] if len(signal) >= 3 else []
    indices = sorted(indices, key=lambda i: -signal[i])[:max_peaks]
    candidates = parse_reaction_candidates(reaction_input.value) if "reaction_input" in globals() else []
    if not candidates and "get_candidates_from_table" in globals():
        candidates = [
            {"role": classification, **ions}
            for _, classification, smi in get_candidates_from_table()
            if (ions := calculate_expected_ions(smi)) and "error" not in ions
        ]
    hypotheses = reaction_mass_hypotheses(candidates)
    peaks = []
    for idx in indices:
        width = peak_widths(signal, [idx], rel_height=.8)
        onset = float(times[max(0, int(width[2][0]))])
        ions = _centroid_scans(scans, float(times[idx]), onset)
        matches = [c for c in candidates
                   if any(abs(ion["mz"] - mz) <= max(.03, mz * 15e-6)
                          for mz in c["positive"].values() for ion in ions)]
        peaks.append({
            "rt_min": round(float(times[idx]), 3), "onset_min": onset,
            "ions": ions, "artifacts": _flag_artifacts(ions),
            "neutral_losses": annotate_neutral_losses(ions),
            "candidates": [{"role": c["role"], "formula": c["formula"],
                            "cLogP": c["cLogP"], "TPSA": c["TPSA"]} for c in matches],
        })
    SPECTRAL_STATE.update(scans=scans, times=times, tic=tic, bpc=bpc, peaks=peaks,
                          candidates=candidates, hypotheses=hypotheses,
                          uv_ms_delay_min=uv_ms_delay_min,
                          data_id=id(globals().get("od", globals().get("hplc_df"))),
                          reaction_text=reaction_input.value if "reaction_input" in globals() else "")
    SPECTRAL_STATE.pop("pending_plot_html", None)
    return SPECTRAL_STATE


def register_uv_trace(uv_data: pd.DataFrame, time_col: str = "Time",
                      time_unit: str = "min") -> list[str]:
    """Attach an optional DAD CSV (Time, 254 nm, 214 nm, etc.) without replacing MS data."""
    if time_unit not in ("min", "s") or time_col not in uv_data:
        raise ValueError("Provide a Time column and specify 'min' or 's'.")
    times = np.asarray(uv_data[time_col], dtype=float) / (60.0 if time_unit == "s" else 1.0)
    if not len(times) or not np.all(np.isfinite(times)) or np.any(np.diff(times) <= 0):
        raise ValueError("UV retention times must be finite and strictly increasing.")
    traces = {}
    for column in uv_data.columns:
        if column == time_col:
            continue
        values = np.asarray(uv_data[column], dtype=float)
        if len(values) == len(times) and np.all(np.isfinite(values)):
            traces[str(column)] = values
    SPECTRAL_STATE["uv"] = {"times": times, "traces": traces}
    return list(traces)


def match_uv_apex_to_ms_scan(uv_rt_min: float, uv_ms_delay_min: float = 0.08):
    """Map a UV apex to the closest MS1 scan, compensating for detector dead volume."""
    scans = SPECTRAL_STATE.get("scans", [])
    if not scans:
        return None
    if uv_ms_delay_min < 0:
        raise ValueError("UV/MS delay must be nonnegative.")
    target = uv_rt_min + uv_ms_delay_min
    return min(scans, key=lambda scan: abs(scan[0] - target))[0]


def _xic(scans: list[tuple], target_mz: float, ppm: float) -> np.ndarray:
    return np.array([
        float(np.sum(scan[2][np.abs(scan[1] - target_mz) <= target_mz * ppm * 1e-6]))
        for scan in scans
    ])


def update_lcms_plot(
    extracted_ion_mzs_csv: str = "",
    rt_start_min: float = 0.0,
    rt_end_min: float = 0.0,
    ppm_tolerance: float = 15.0,
    select_scan_rt_min: float = 0.0,
    overlay_formula_isotopes: str = ""
) -> str:
    """
    Updates the local multi-panel LCMS Plotly dashboard (UV/DAD, MS1 TIC/BPC/XIC in minutes,
    and optional MS1 scan with theoretical isotope envelope overlay).
    - extracted_ion_mzs_csv: Comma-separated target m/z values for XIC traces (e.g. '268.1041, 252.1091').
    - rt_start_min, rt_end_min: Optional retention time zoom window in minutes (leave 0.0 for full range).
    - ppm_tolerance: Mass tolerance in ppm (default 15.0).
    - select_scan_rt_min: Retention time in minutes for MS1 spectrum bar plot (0.0 for auto).
    - overlay_formula_isotopes: Optional molecular formula or SMILES to overlay theoretical isotope bars.
    """
    if not SPECTRAL_STATE or "times" not in SPECTRAL_STATE:
        prepare_spectral_state()
    if not 0 < ppm_tolerance <= 1000:
        return "Invalid ppm tolerance."

    extracted_ion_mz = []
    if extracted_ion_mzs_csv.strip():
        try:
            extracted_ion_mz = [float(x.strip()) for x in extracted_ion_mzs_csv.split(",") if x.strip()]
        except ValueError:
            return "Invalid extracted_ion_mzs_csv format; provide comma-separated numbers."

    if len(extracted_ion_mz) > 8 or any(not math.isfinite(mz) or mz <= 0 for mz in extracted_ion_mz):
        return "Provide at most eight positive finite extracted-ion masses."

    rt_window = [rt_start_min, rt_end_min] if (rt_end_min > rt_start_min >= 0.0) else None
    select_scan_rt = select_scan_rt_min if select_scan_rt_min > 0.0 else None
    iso_str = overlay_formula_isotopes.strip()

    if iso_str and not SPECTRAL_STATE.get("scans"):
        return "No MS1 scan is available for an isotope overlay."
    try:
        envelope = theoretical_isotope_envelope(iso_str) if iso_str else []
    except (ValueError, RuntimeError) as exc:
        return f"Invalid isotope formula: {exc}"

    scans = SPECTRAL_STATE.get("scans", [])
    times = SPECTRAL_STATE["times"]
    tic = SPECTRAL_STATE["tic"]
    bpc = SPECTRAL_STATE["bpc"]
    uv = SPECTRAL_STATE.get("uv")
    show_ms = bool(scans) and (select_scan_rt is not None or bool(envelope))
    fig = make_subplots(rows=3 if show_ms else 2, cols=1, shared_xaxes=False,
                        row_heights=[.32, .36, .32] if show_ms else [.44, .56],
                        vertical_spacing=.09,
                        subplot_titles=("UV/DAD (if supplied)", "MS1 chromatogram (minutes)") +
                                       (("MS1 scan (m/z)",) if show_ms else ()))
    if uv:
        uv_times = uv["times"]
        for wavelength, values in uv["traces"].items():
            fig.add_trace(go.Scatter(x=uv_times, y=values, name=f"UV {wavelength}",
                                     mode="lines"), row=1, col=1)
            baseline = float(np.percentile(values, 10))
            corrected = np.maximum(0, values - baseline)
            apices = find_peaks(corrected, prominence=max(float(np.max(corrected)) * .05, 1e-10),
                                distance=3)[0] if len(values) >= 3 else []
            if len(apices):
                widths = peak_widths(corrected, apices, rel_height=.8)
                bounds = [(max(0, int(left)), min(len(values), int(right) + 1))
                          for left, right in zip(widths[2], widths[3])]
                areas = [float(trapezoid(corrected[left:right], uv_times[left:right]))
                         for left, right in bounds]
                total = sum(areas)
                for idx, area in zip(apices, areas):
                    if total:
                        fig.add_annotation(x=float(uv_times[idx]), y=float(values[idx]),
                                           text=f"{area / total * 100:.1f}%", showarrow=True,
                                           arrowhead=2, row=1, col=1)
    else:
        fig.add_annotation(x=.5, y=.5, xref="x domain", yref="y domain",
                           text="UV/DAD not supplied — upload a Time + wavelength CSV",
                           showarrow=False, row=1, col=1)
    fig.add_trace(go.Scatter(x=times, y=tic, mode="lines", name="TIC"), row=2, col=1)
    fig.add_trace(go.Scatter(x=times, y=bpc, mode="lines", name="BPC",
                             visible="legendonly" if len(bpc) and len(tic) and
                             np.max(bpc) < np.max(tic) * .05 else True), row=2, col=1)
    for mz in extracted_ion_mz:
        if scans:
            fig.add_trace(go.Scatter(x=times, y=_xic(scans, mz, ppm_tolerance),
                                     mode="lines", name=f"XIC {mz:.4f}"), row=2, col=1)
    for peak in SPECTRAL_STATE.get("peaks", []):
        fig.add_vline(x=peak["rt_min"], line_dash="dot", line_color="#64748b", row=2, col=1)
    for h in SPECTRAL_STATE.get("hypotheses", []):
        if not scans:
            break
        intensities = _xic(scans, h["mz"], ppm_tolerance)
        indices = find_peaks(intensities, prominence=max(float(np.max(intensities)) * .05, 1e-10))[0] if len(intensities) >= 3 else []
        if len(indices):
            apex = float(times[max(indices, key=lambda idx: intensities[idx])])
            fig.add_vline(x=apex, line_dash="dash",
                          line_color="#e11d48" if "SM" in h["name"] else "#7c3aed",
                          annotation_text=html.escape(h["name"]), row=2, col=1)
    if rt_window:
        fig.update_xaxes(range=rt_window, row=2, col=1)
        fig.update_xaxes(range=rt_window, row=1, col=1)
    if show_ms:
        if select_scan_rt is None and uv and uv["traces"]:
            first = next(iter(uv["traces"].values()))
            select_scan_rt = match_uv_apex_to_ms_scan(float(uv["times"][int(np.argmax(first))]),
                                                       SPECTRAL_STATE.get("uv_ms_delay_min", 0.08))
        if select_scan_rt is None:
            select_scan_rt = SPECTRAL_STATE["peaks"][0]["rt_min"] if SPECTRAL_STATE.get("peaks") else scans[0][0]
        scan = min(scans, key=lambda s: abs(s[0] - select_scan_rt))
        fig.add_trace(go.Bar(x=scan[1], y=scan[2], name=f"Scan at {scan[0]:.3f} min",
                             marker_color="#2563eb"), row=3, col=1)
        if envelope:
            observed = float(_xic([scan], envelope[0]["mz"], ppm_tolerance)[0])
            scale = max(observed, float(np.max(scan[2])) * .1 if len(scan[2]) else 0, 1)
            fig.add_trace(go.Bar(
                x=[entry["mz"] for entry in envelope],
                y=[entry["relative_intensity"] * scale / 100.0 for entry in envelope],
                width=.08, opacity=.45, marker=dict(color="#e11d48",
                                                      line=dict(color="#e11d48", width=2)),
                name=f"Theoretical {iso_str}",
                text=[entry["label"] for entry in envelope], hovertemplate="%{text}: %{x:.4f}<extra></extra>"
            ), row=3, col=1)
        for peak in SPECTRAL_STATE.get("peaks", []):
            if abs(peak["rt_min"] - scan[0]) < .06:
                for loss in peak["neutral_losses"][:4]:
                    mask = np.abs(scan[1] - loss["fragment"]) < .03
                    if np.any(mask):
                        fig.add_annotation(x=loss["fragment"], y=float(np.max(scan[2][mask])),
                                           text=f"-{loss['loss']}", showarrow=True,
                                           ay=-50, row=3, col=1)
        fig.update_xaxes(title_text="m/z", row=3, col=1)
    fig.update_xaxes(title_text="Retention time (min)", row=2, col=1)
    fig.update_yaxes(title_text="UV absorbance", row=1, col=1)
    fig.update_yaxes(title_text="MS intensity", row=2, col=1)
    fig.update_layout(title="LCMS reaction monitoring", template="plotly_white",
                      barmode="overlay", height=850 if show_ms else 660)
    SPECTRAL_STATE["figure"] = fig
    SPECTRAL_STATE["pending_plot_html"] = fig.to_html(include_plotlyjs="cdn", full_html=False,
                                                       div_id=f"lcms_{uuid.uuid4().hex[:8]}")
    return "LCMS dashboard updated locally; the plot will be displayed below your response."

In [ ]:
#@title 8C. Interactive Plotly Chromatogram & Floating Hover Cards

# --- SELF-CONTAINED INTERACTIVE PLOTLY CHROMATOGRAM BUILDER ---

def _peak_label_for_legend(p_id: int) -> str:
    if 'peak_assignments' in globals() and p_id in peak_assignments:
        a_info = peak_assignments[p_id]
        iso_short = " [Isobaric Isomers]" if a_info.get("is_isobaric", False) else ""
        adduct_summary = ", ".join([f"{x['adduct']} ({x['obs_mz']:.2f})" for x in a_info['adducts']])
        return f"Peak {p_id}: {a_info['display_name']}{iso_short} [{adduct_summary}] (Conf: {a_info['confidence']:.0f}%)"
    if 'detected_df' in globals():
        dom_rows = detected_df.loc[detected_df['peak_id'] == p_id, 'dominant m/z']
        if len(dom_rows):
            dom_str = dom_rows.values[0]
            first_ion = str(dom_str).split(',')[0] if dom_str != "None" else "N/A"
            return f"Peak {p_id}: Unassigned (Base m/z: {first_ion})"
    return f"Peak {p_id}: Unassigned"


def build_interactive_chromatogram_html(div_id: str = "hplc_interactive_chrom_div") -> str:
    """
    Rebuilds the interactive Plotly chromatogram HTML with floating RDKit 2D structure hover cards
    from the live `detected_df` and `peak_assignments` dictionaries, and updates `_notebook_figure_registry`.
    """
    FIG_WIDTH_PX, FIG_HEIGHT_PX = 1220, 760
    MARGIN_L, MARGIN_R, MARGIN_T, MARGIN_B = 80, 40, 65, 210
    X_RANGE = [190, 310]

    time_arr = chrom.df[chrom.time_col].values
    raw_signal = chrom.df[chrom.int_col].values
    convolved = np.sum(chrom.unmixed_chromatograms, axis=1) if chrom.peaks is not None else None
    y_max_plot = float(np.max(raw_signal))
    prom_val = globals().get('current_prominence_val', 0.05)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    hover_cards_by_peak = {}
    qc_struct_images = {}

    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        norm_area = peak_row.get('normalised area', 'N/A')
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            assign = peak_assignments[p_id]
            mol_images = [Draw.MolToImage(m, size=(175, 145)) for m in assign["mols"]]
            if len(mol_images) > 1:
                total_w = sum(im.width for im in mol_images)
                struct_img = Image.new('RGBA', (total_w, mol_images[0].height), (255, 255, 255, 255))
                x_off = 0
                for im in mol_images:
                    struct_img.paste(im, (x_off, 0))
                    x_off += im.width
            else:
                struct_img = mol_images[0]

            qc_struct_images[p_id] = struct_img
            buf = io.BytesIO()
            struct_img.save(buf, format='PNG')
            b64_png = base64.b64encode(buf.getvalue()).decode('utf-8')

            adduct_lines = "<br>".join([
                f"<b>{a['adduct']}</b>: {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ])
            iso_badge = (
                f"<div style='color:#b45309; font-weight:600; font-size:11px; margin-top:2px;'>"
                f"⚠️ Isobaric Isomers ({assign['formula']})</div>"
            ) if assign.get("is_isobaric", False) else ""

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:10px; background:#ffffff; "
                f"padding:10px 12px; box-shadow:0 8px 20px rgba(0,0,0,0.18); font-family:sans-serif; "
                f"color:#111827; min-width:220px; max-width:390px; text-align:center;'>"
                f"<div style='font-weight:700; font-size:13px; border-bottom:1px solid #e5e7eb; "
                f"padding-bottom:4px; margin-bottom:4px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s): {assign['display_name']}</div>"
                f"{iso_badge}"
                f"<div style='font-size:11.5px; color:#374151; line-height:1.45; margin-bottom:6px;'>"
                f"{adduct_lines}<br>"
                f"<b>Area:</b> {norm_area} &nbsp;|&nbsp; "
                f"<b>Confidence:</b> <span style='color:#047857; font-weight:700;'>{assign['confidence']:.1f}%</span>"
                f"</div>"
                f"<img src='data:image/png;base64,{b64_png}' "
                f"style='max-width:100%; height:auto; display:block; margin:0 auto; border-radius:4px;'/>"
                f"</div>"
            )
        else:
            dom_str = str(peak_row.get('dominant m/z', 'None'))
            dom_tokens = [x.strip() for x in dom_str.split(',') if x.strip() and x.strip() != 'None']
            top_ions_str = "<br>".join(dom_tokens[:4]) if dom_tokens else "N/A"

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:8px; background:#f9fafb; "
                f"padding:8px 12px; box-shadow:0 6px 16px rgba(0,0,0,0.15); font-family:sans-serif; "
                f"color:#1f2937; min-width:190px; text-align:left;'>"
                f"<div style='font-weight:700; font-size:12.5px; border-bottom:1px solid #d1d5db; "
                f"padding-bottom:3px; margin-bottom:5px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s) — Unassigned</div>"
                f"<div style='font-size:11px; color:#4b5563; line-height:1.4;'>"
                f"<b>Norm. Area:</b> {norm_area}<br>"
                f"<b>Dominant m/z:</b><br>{top_ions_str}</div>"
                f"</div>"
            )

    chrom_fig = go.Figure()
    chrom_fig.add_trace(go.Scatter(
        x=time_arr, y=raw_signal, mode='lines', name='raw chromatogram',
        line=dict(color='black', width=1.6),
        hovertemplate='<b>Raw Chromatogram</b><br>RT=%{x:.2f}s<br>Signal=%{y:.3e}<extra></extra>'
    ))
    if convolved is not None:
        chrom_fig.add_trace(go.Scatter(
            x=time_arr, y=convolved, mode='lines', name='inferred mixture',
            line=dict(color='red', width=1.4, dash='dash'),
            hovertemplate='<b>Inferred Mixture</b><br>RT=%{x:.2f}s<br>Mixture=%{y:.3e}<extra></extra>'
        ))

    if chrom.peaks is not None:
        for g, _ in chrom.peaks.groupby('peak_id'):
            p_id = int(g)
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            component = chrom.unmixed_chromatograms[:, p_id - 1]
            chrom_fig.add_trace(go.Scatter(
                x=time_arr, y=component, mode='lines', fill='tozeroy',
                line=dict(color=peak_color, width=1.0), opacity=0.55,
                name=_peak_label_for_legend(p_id),
                meta=p_id,
                hoverinfo='none'
            ))

    shapes, compact_annotations = [], []
    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        c_idx = int(np.argmin(np.abs(time_arr - rt_apex)))
        apex_y = float(raw_signal[c_idx])
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        shapes.append(dict(
            type='line', x0=rt_apex, x1=rt_apex, y0=0, y1=apex_y,
            line=dict(color='dodgerblue', width=1.1, dash='dot'), opacity=0.55
        ))

        badge_text = (
            f"<b>P{p_id}: {peak_assignments[p_id]['display_name']}</b>"
            if ('peak_assignments' in globals() and p_id in peak_assignments)
            else f"<b>P{p_id}</b>"
        )
        compact_annotations.append(dict(
            x=rt_apex, y=apex_y + y_max_plot * 0.035, xref='x', yref='y',
            text=badge_text,
            showarrow=False,
            font=dict(size=10, color='#111827'),
            bgcolor='rgba(255,255,255,0.88)',
            bordercolor=peak_color,
            borderwidth=1.5,
            borderpad=2
        ))

        chrom_fig.add_trace(go.Scatter(
            x=[rt_apex], y=[apex_y],
            mode='markers',
            marker=dict(size=9, color=peak_color, line=dict(width=1.5, color='white')),
            showlegend=False,
            meta=p_id,
            hoverinfo='none'
        ))

    chrom_fig.update_layout(
        shapes=shapes,
        annotations=compact_annotations,
        title=dict(
            text=(
                f"HPLC Chromatogram Deconvolution & Peak Assignments (Prominence: {prom_val:.4f}) "
                f"— Hover over any peak to view 2D structure & MS1 details"
            ),
            font=dict(size=14.5, color='#111827')
        ),
        xaxis=dict(
            title='Retention Time (seconds)',
            range=X_RANGE,
            rangeslider=dict(visible=True, thickness=0.10)
        ),
        yaxis=dict(
            title='Total Ion Count (baseline corrected)',
            range=[0, y_max_plot * 1.18]
        ),
        legend=dict(
            orientation='h',
            x=0.5, y=-0.26,
            xanchor='center', yanchor='top',
            font=dict(size=10),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#d1d5db',
            borderwidth=1
        ),
        margin=dict(l=MARGIN_L, r=MARGIN_R, t=MARGIN_T, b=MARGIN_B),
        template='plotly_white',
        width=FIG_WIDTH_PX,
        height=FIG_HEIGHT_PX,
        hovermode='closest'
    )

    globals()['_last_chromatogram_figure'] = chrom_fig

    # Update static Kaleido PNG snapshot in `_notebook_figure_registry` for Step 4 Vision QC
    try:
        qc_fig = go.Figure(chrom_fig)
        qc_images = []
        qc_annotations = list(compact_annotations)
        banner_xs = np.linspace(200.0, 300.0, max(len(qc_struct_images), 1))
        for idx_p, (p_id, struct_img) in enumerate(sorted(qc_struct_images.items())):
            bx = float(banner_xs[idx_p])
            by = y_max_plot * 1.48
            assign = peak_assignments[p_id]
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            qc_annotations.append(dict(
                x=bx, y=by + y_max_plot * 0.04, xref='x', yref='y',
                text=f"<b>Peak {p_id}: {assign['display_name']}</b> ({assign['confidence']:.0f}%)",
                showarrow=False, font=dict(size=9.5, color='#111827'),
                bgcolor='white', bordercolor=peak_color, borderwidth=1.5
            ))
            qc_images.append(dict(
                source=struct_img, xref='x', yref='y',
                x=bx, y=by,
                sizex=18.0, sizey=y_max_plot * 0.32,
                xanchor='center', yanchor='top', layer='above'
            ))
        qc_fig.update_layout(
            yaxis=dict(title='Total Ion Count (baseline corrected)', range=[0, y_max_plot * 1.62]),
            annotations=qc_annotations,
            images=qc_images
        )
        globals().setdefault('_notebook_figure_registry', {})['deconvoluted_chromatogram_with_structures'] = qc_fig.to_image(format='png', scale=2)
    except Exception:
        pass

    fig_html = chrom_fig.to_html(include_plotlyjs='cdn', full_html=False, div_id=div_id)
    cards_json = json.dumps({str(k): v for k, v in hover_cards_by_peak.items()})

    return f"""
    <div style="position:relative; width:{FIG_WIDTH_PX}px; margin: 12px 0;">
        {fig_html}
        <div id="{div_id}_tooltip" style="display:none; position:fixed; pointer-events:none; z-index:99999; transition: opacity 0.08s ease-in-out;"></div>
    </div>
    <script>
    (function() {{
        var cards = {cards_json};
        var gd = document.getElementById('{div_id}');
        var tooltip = document.getElementById('{div_id}_tooltip');

        function attachHoverEvents() {{
            if (!gd || typeof gd.on !== 'function') {{
                setTimeout(attachHoverEvents, 150);
                return;
            }}
            gd.on('plotly_hover', function(data) {{
                if (!data || !data.points || data.points.length === 0) return;
                var pt = data.points[0];
                var p_id = pt.data.meta;
                if (p_id && cards[String(p_id)]) {{
                    tooltip.innerHTML = cards[String(p_id)];
                    tooltip.style.display = 'block';
                    if (data.event) {{
                        var x = data.event.clientX + 18;
                        var y = data.event.clientY - 30;
                        var rect = tooltip.getBoundingClientRect();
                        if (x + rect.width > window.innerWidth - 15) x = data.event.clientX - rect.width - 18;
                        if (y + rect.height > window.innerHeight - 15) y = window.innerHeight - rect.height - 15;
                        if (y < 10) y = 10;
                        tooltip.style.left = x + 'px';
                        tooltip.style.top = y + 'px';
                    }}
                }}
            }});
            gd.on('plotly_unhover', function() {{
                tooltip.style.display = 'none';
            }});
            gd.addEventListener('mousemove', function(e) {{
                if (tooltip.style.display === 'block') {{
                    var x = e.clientX + 18;
                    var y = e.clientY - 30;
                    var rect = tooltip.getBoundingClientRect();
                    if (x + rect.width > window.innerWidth - 15) x = e.clientX - rect.width - 18;
                    if (y + rect.height > window.innerHeight - 15) y = window.innerHeight - rect.height - 15;
                    if (y < 10) y = 10;
                    tooltip.style.left = x + 'px';
                    tooltip.style.top = y + 'px';
                }}
            }});
        }}
        attachHoverEvents();
    }})();
    </script>
    """

In [ ]:
#@title 8D. Gemini Agent Tools (RDKit, PubChem, ChEMBL, MassBank & Replotting)
# --- RDKIT, PUBCHEM, CHEMBL, MASSBANK & REPLOTTING TOOLS FOR GEMINI ---

def get_user_candidate_molecules() -> str:
    """Returns the user's input table of candidate molecules, classifications, SMILES, formulas, and exact masses."""
    candidates = get_candidates_from_table() if 'get_candidates_from_table' in globals() else []
    if not candidates:
        return "No candidate molecules found."

    rows = []
    for name, classification, smi in candidates:
        mol, clean_smi, _ = safe_mol_from_smiles(smi)
        if mol:
            em = Descriptors.ExactMolWt(mol)
            formula = rdMolDescriptors.CalcMolFormula(mol)
            rows.append({
                "Name": name,
                "Classification": classification,
                "SMILES": clean_smi,
                "Formula": formula,
                "Exact_Mass": round(em, 4),
                "[M+H]+_mz": round(em + 1.007276, 4),
                "cLogP": round(Crippen.MolLogP(mol), 2),
                "TPSA": round(rdMolDescriptors.CalcTPSA(mol), 1)
            })
    return pd.DataFrame(rows).to_markdown(index=False)


def get_detected_peaks_summary() -> str:
    """Returns a full tabular summary of all deconvoluted HPLC-MS peaks, retention times, areas, dominant m/z ions, and assignments."""
    if 'detected_df' not in globals():
        return "Error: detected_df not found in memory. Run Cell 4 first."
    cols = [c for c in ['peak_id', 'retention_time', 'peak start time', 'peak end time', 'normalised area', 'dominant m/z', 'Assignment', 'Confidence (%)'] if c in detected_df.columns]
    return detected_df[cols].to_markdown(index=False)


def get_distilled_peak_details(peak_number: int) -> dict:
    """Retrieves background-subtracted ions, neutral losses, and contaminant flags for a 1-based intensity-ranked peak."""
    if not SPECTRAL_STATE or "peaks" not in SPECTRAL_STATE:
        prepare_spectral_state()
    peaks = SPECTRAL_STATE.get("peaks", [])
    if not 1 <= peak_number <= len(peaks):
        return {"error": f"Distilled peak {peak_number} not found (valid range: 1..{len(peaks)})."}
    peak = peaks[peak_number - 1]
    return {
        "peak_number": peak_number,
        "rt_min": peak["rt_min"],
        "rt_sec": round(peak["rt_min"] * 60.0, 1),
        "ions": peak["ions"],
        "artifacts": peak["artifacts"],
        "neutral_losses": peak["neutral_losses"],
        "candidates": peak["candidates"]
    }


def get_peak_details(peak_id: int) -> dict:
    """Retrieves detailed analytical evidence, observed adducts, and raw MS1 ions for a specific deconvoluted peak ID."""
    result = {}
    if 'detected_df' in globals():
        row = detected_df[detected_df['peak_id'] == peak_id]
        if not row.empty:
            r_dict = row.iloc[0].to_dict()
            result["chromatogram_row"] = {k: str(v) for k, v in r_dict.items()}
            rt_start = r_dict.get('peak start time', np.nan)
            rt_end = r_dict.get('peak end time', np.nan)
            if 'od' in globals() and not pd.isna(rt_start) and not pd.isna(rt_end):
                agg = {}
                for spec in od:
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
                        mzs, ints = spec.get_peaks()
                        for m, inten in zip(np.round(mzs, 4), ints):
                            if inten > 1e3:
                                agg[m] = agg.get(m, 0.0) + float(inten)
                if agg:
                    top_ions = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:25]
                    max_i = top_ions[0][1]
                    result["raw_ms1_top_25_ions_in_peak_window"] = [
                        {"mz": float(m), "intensity": round(i, 1), "rel_pct": round((i / max_i) * 100, 2)}
                        for m, i in top_ions
                    ]

    if 'peak_assignments' in globals() and peak_id in peak_assignments:
        assign = peak_assignments[peak_id]
        result["assigned_compound"] = {
            "display_name": assign["display_name"],
            "formula": assign["formula"],
            "confidence": assign["confidence"],
            "evidence": assign["evidence"],
            "adducts_observed": [
                f"{a['adduct']} at {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ]
        }
    return result if result else {"error": f"Peak {peak_id} not found."}


def check_mz_in_peak(peak_id: int, target_mz: float, tolerance_da: float = 0.05) -> dict:
    """Checks all raw MS1 scans across a specific peak's retention time window for a target m/z."""
    if 'detected_df' not in globals() or 'od' not in globals():
        return {"error": "detected_df or pyOpenMS 'od' experiment not found in memory."}

    row = detected_df[detected_df['peak_id'] == peak_id]
    if row.empty:
        return {"error": f"Peak {peak_id} not found in detected_df."}

    rt_start = float(row.iloc[0]['peak start time'])
    rt_end = float(row.iloc[0]['peak end time'])

    matches = []
    for spec in od:
        if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
            mzs, ints = spec.get_peaks()
            mask = (mzs >= target_mz - tolerance_da) & (mzs <= target_mz + tolerance_da)
            if np.any(mask):
                for m, inten in zip(mzs[mask], ints[mask]):
                    matches.append({
                        "rt": round(float(spec.getRT()), 2),
                        "observed_mz": round(float(m), 4),
                        "error_da": round(float(m - target_mz), 4),
                        "ppm_error": round(float((m - target_mz) / target_mz * 1e6), 1),
                        "intensity": round(float(inten), 1)
                    })

    matches = sorted(matches, key=lambda x: x["intensity"], reverse=True)
    return {
        "peak_id": peak_id,
        "rt_window": f"{rt_start:.1f}s - {rt_end:.1f}s",
        "target_mz_searched": target_mz,
        "found_in_raw_scans": len(matches) > 0,
        "max_observed_hit": matches[0] if matches else None
    }


def generate_in_silico_fragments(parent_smiles: str, check_peak_id: int = 0) -> dict:
    """
    Generates RDKit-sanitized ESI+ fragments for a parent molecule, automatically pre-renders
    the 2D SVG structure image for the parent AND every fragment (returning an `image_token`
    like `[[MOL_IMG_1]]` for each), and checks if each fragment m/z is present in `check_peak_id`.
    """
    mol, clean_parent, err = safe_mol_from_smiles(parent_smiles)
    if not mol:
        return {"error": err}

    parent_em = Descriptors.ExactMolWt(mol)
    parent_formula = rdMolDescriptors.CalcMolFormula(mol)
    parent_mh = round(parent_em + 1.007276, 4)
    parent_img_token = _register_mol_svg(mol, label="Parent Molecule")

    fragments_dict = {}

    def _add_frag(frag_smi, pathway_label, is_intrinsic_cation=False):
        f_mol, f_clean, _ = safe_mol_from_smiles(frag_smi)
        if not f_mol:
            return
        em = Descriptors.ExactMolWt(f_mol)
        if em < 40.0 or em >= parent_em - 1.0:
            return
        form = rdMolDescriptors.CalcMolFormula(f_mol)
        f_charge = Chem.GetFormalCharge(f_mol)
        if is_intrinsic_cation or f_charge > 0:
            mz_val = round(em - 0.00054858, 4)
            ion_type = "[F]+ (intrinsic cation)"
        else:
            mz_val = round(em + 1.007276, 4)
            ion_type = "[F+H]+"

        key = (form, round(mz_val, 3))
        if key not in fragments_dict:
            entry = {
                "fragment_name": pathway_label,
                "smiles": f_clean,
                "formula": form,
                "neutral_exact_mass": round(em, 4),
                "ion_type": ion_type,
                "expected_esi_pos_mz": mz_val,
                "neutral_loss_from_parent_Da": round(parent_mh - mz_val, 4),
                "_temp_mol": f_mol
            }
            if check_peak_id > 0:
                hit_info = check_mz_in_peak(check_peak_id, mz_val, tolerance_da=0.05)
                entry["present_in_peak"] = hit_info.get("found_in_raw_scans", False)
                entry["peak_observation"] = hit_info.get("max_observed_hit", None)
            fragments_dict[key] = entry

    ribose_smarts = Chem.MolFromSmarts("[C@@H,C@H,CH]1O[C@H,C@@H,CH](CO)[C@@H,C@H,CH](O)[C@H,C@@H,CH]1")
    if mol.HasSubstructMatch(ribose_smarts):
        _add_frag("OC[C@H]1O[C+]=[C@H][C@@H]1O", "Ribose oxocarbenium fragment [C5H7O3]+", is_intrinsic_cation=True)

    cleavable_bonds = [b.GetIdx() for b in mol.GetBonds() if not b.IsInRing() and b.GetBondType() == Chem.BondType.SINGLE]
    for b_idx in cleavable_bonds:
        try:
            frag_mol = Chem.FragmentOnBonds(mol, [b_idx], addDummies=False)
            for f in Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False):
                _add_frag(Chem.MolToSmiles(f), "Single-bond cleavage fragment ([F+H]+)")
        except Exception:
            pass

    if "ncnc2c1ncn2" in clean_parent or "ncnc2c1nc[nH]2" in clean_parent:
        _add_frag("Nc1ncnc2c1nc[nH]2", "Protonated Adenine [B+H]+ (glycosidic cleavage, -Ribose)")
        _add_frag("c1ncnc2c1nc[nH]2", "Purine core after NH3 loss ([B+H-NH3]+)")
        _add_frag("Nc1ncnc2c1ncn2C=C", "0,2X Cross-ring cleavage (Adenine + vinyl remnant)")
        _add_frag("Nc1ncnc2c1ncn2CC=O", "0,2X Cross-ring ribose cleavage (Adenine + C2H3O)")
        _add_frag("Nc1ncnc2c1ncn2[C@@H](O)C=O", "0,3X Cross-ring ribose cleavage (Adenine + C2H3O2)")

    sorted_frags = sorted(fragments_dict.values(), key=lambda x: x["expected_esi_pos_mz"], reverse=True)
    for f_entry in sorted_frags:
        f_mol = f_entry.pop("_temp_mol")
        f_entry["image_token"] = _register_mol_svg(f_mol, label=f_entry["fragment_name"])

    return {
        "parent_smiles": clean_parent,
        "parent_formula": parent_formula,
        "parent_exact_mass": round(parent_em, 4),
        "parent_[M+H]+_mz": parent_mh,
        "parent_image_token": parent_img_token,
        "generated_fragments": sorted_frags,
        "rendering_instruction": (
            "Each molecule above already has its 2D image pre-rendered! "
            "Simply include the exact `parent_image_token` or `image_token` (e.g. [[MOL_IMG_1]]) "
            "on its own line in your response where you want that structure displayed."
        )
    }


def analyze_structure_and_mass(smiles_or_formula: str, target_mz: float = 0.0, label: str = "") -> dict:
    """
    Calculates exact monoisotopic mass, formula, and ESI m/z values ([M]+, [M+H]+, [M+Na]+),
    and automatically pre-renders the 2D SVG structure image (returning `image_token`).
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles_or_formula)
    if mol is None:
        try:
            em = rdMolDescriptors.CalcExactMolWt(smiles_or_formula.strip())
            return {
                "input_interpreted_as": "Molecular Formula (no 2D structure)",
                "formula": smiles_or_formula.strip(),
                "exact_monoisotopic_mass": round(em, 5),
                "protonated_[M+H]+": round(em + 1.007276, 4)
            }
        except Exception:
            return {"error": err}

    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    formal_charge = Chem.GetFormalCharge(mol)
    mz_mh_plus = exact_mass + 1.007276 if formal_charge == 0 else exact_mass - (formal_charge * 0.00054858)

    img_token = _register_mol_svg(mol, label=label or formula)

    res = {
        "canonical_smiles": clean_smi,
        "formula": formula,
        "exact_monoisotopic_mass": round(exact_mass, 5),
        "protonated_[M+H]+": round(mz_mh_plus, 4),
        "rdkit_cLogP": round(Crippen.MolLogP(mol), 3),
        "TPSA_A2": round(rdMolDescriptors.CalcTPSA(mol), 2),
        "image_token": img_token,
        "instruction": f"Place {img_token} on its own line in your response to display the 2D structure."
    }
    if target_mz > 0:
        diff_da = mz_mh_plus - target_mz
        res["target_mz_comparison"] = {
            "target_mz": target_mz,
            "error_Da": round(diff_da, 4),
            "ppm_error": round((diff_da / target_mz) * 1e6, 1)
        }
    return res


def generate_2d_structure_svg(smiles: str, label: str = "") -> dict:
    """Explicit tool to generate a 2D chemical structure SVG and return its `image_token` (e.g. [[MOL_IMG_1]])."""
    mol, clean_smi, err = safe_mol_from_smiles(smiles)
    if not mol:
        return {"error": err}
    token = _register_mol_svg(mol, label=label)
    return {
        "status": "success",
        "canonical_smiles": clean_smi,
        "image_token": token,
        "instruction": f"Insert the exact token {token} on its own line in your response to display this structure."
    }


def search_pubchem_by_mass_or_formula(
    observed_mh_mz: float = 0.0,
    molecular_formula: str = "",
    compound_name: str = "",
    mass_window_da: float = 0.015,
    max_results: int = 8
) -> dict:
    """
    Searches PubChem for known metabolites or compounds by compound_name, molecular_formula,
    OR observed [M+H]+ m/z (automatically subtracting H+ mass 1.007276 Da via PubChem PUG REST),
    and pre-renders 2D structure SVGs (`image_token`) for each hit.
    """
    try:
        max_results = max(1, min(int(max_results), 8))
        compounds = []
        if compound_name.strip():
            compounds = pcp.get_compounds(compound_name.strip(), 'name')[:max_results]
        elif molecular_formula.strip():
            compounds = pcp.get_compounds(molecular_formula.strip(), 'formula')[:max_results]
        elif observed_mh_mz > 0:
            neutral_mass = observed_mh_mz - 1.007276
            low_m = round(neutral_mass - mass_window_da, 4)
            high_m = round(neutral_mass + mass_window_da, 4)
            pug_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/exact_mass/range/{low_m}/{high_m}/cids/JSON"
            req = urllib.request.Request(pug_url, headers={"User-Agent": "Colab-LCMS-Agent/1.0"})
            with urllib.request.urlopen(req, timeout=10) as resp:
                cid_data = json.loads(resp.read().decode('utf-8'))
            cids = cid_data.get("IdentifierList", {}).get("CID", [])[:max_results]
            if cids:
                compounds = pcp.get_compounds(cids, 'cid')

        hits = []
        for c in compounds[:max_results]:
            smi = getattr(c, 'isomeric_smiles', None) or getattr(c, 'canonical_smiles', None)
            if not smi:
                continue
            mol, clean_smi, _ = safe_mol_from_smiles(smi)
            if not mol:
                continue
            em = Descriptors.ExactMolWt(mol)
            mh_calc = em + 1.007276
            ppm_err = ((observed_mh_mz - mh_calc) / mh_calc * 1e6) if observed_mh_mz > 0 else 0.0
            synonyms = (getattr(c, 'synonyms', None) or [])[:5]
            pref_label = synonyms[0] if synonyms else (c.iupac_name or f"CID {c.cid}")
            img_token = _register_mol_svg(mol, label=f"{pref_label} (CID {c.cid})")

            hits.append({
                "cid": c.cid,
                "preferred_name": pref_label,
                "synonyms": synonyms,
                "iupac_name": c.iupac_name,
                "formula": c.molecular_formula,
                "smiles": clean_smi,
                "exact_mass": round(em, 4),
                "calculated_[M+H]+_mz": round(mh_calc, 4),
                "ppm_error_vs_observed": round(ppm_err, 2),
                "rdkit_cLogP": round(Crippen.MolLogP(mol), 3),
                "TPSA_A2": round(rdMolDescriptors.CalcTPSA(mol), 2),
                "image_token": img_token
            })
        return {
            "query_mh_mz": observed_mh_mz,
            "query_formula": molecular_formula,
            "query_name": compound_name,
            "pubchem_hits": hits
        }
    except Exception as e:
        return {"error": f"PubChem search failed: {e}"}


def search_chembl_metabolites_and_isomers(
    molecular_formula: str = "",
    smiles: str = "",
    max_results: int = 8
) -> dict:
    """
    Queries the EBI ChEMBL REST API by molecular formula or canonical SMILES
    to retrieve known bioactive metabolites, drug standards, ALogP, polar surface area (PSA),
    RDKit Crippen cLogP for reverse-phase retention time ranking, and 2D structure `image_token`s.
    """
    try:
        max_results = max(1, min(int(max_results), 8))
        hits = []
        if smiles.strip():
            mol, clean_smi, err = safe_mol_from_smiles(smiles)
            if not mol:
                return {"error": err}
            inchikey = Chem.MolToInchiKey(mol)
            url = f"https://www.ebi.ac.uk/chembl/api/data/molecule.json?molecule_structures__standard_inchi_key={inchikey}"
        elif molecular_formula.strip():
            url = f"https://www.ebi.ac.uk/chembl/api/data/molecule.json?molecule_properties__full_molformula={urllib.parse.quote(molecular_formula.strip())}&limit={max_results}"
        else:
            return {"error": "Provide either `molecular_formula` (e.g. 'C10H15N3O5') or `smiles`."}

        req = urllib.request.Request(url, headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/1.0"})
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = json.loads(resp.read().decode('utf-8'))

        for mol_entry in data.get("molecules", [])[:max_results]:
            struct = mol_entry.get("molecule_structures") or {}
            props = mol_entry.get("molecule_properties") or {}
            can_smi = struct.get("canonical_smiles")
            if not can_smi:
                continue

            r_mol, clean_s, _ = safe_mol_from_smiles(can_smi)
            if not r_mol:
                continue
            rdkit_clogp = round(Crippen.MolLogP(r_mol), 3)
            em = round(Descriptors.ExactMolWt(r_mol), 4)
            pref_name = mol_entry.get("pref_name") or mol_entry.get("molecule_chembl_id")
            img_token = _register_mol_svg(r_mol, label=f"{pref_name} ({mol_entry.get('molecule_chembl_id')})")

            hits.append({
                "chembl_id": mol_entry.get("molecule_chembl_id"),
                "pref_name": pref_name,
                "max_clinical_phase": mol_entry.get("max_phase"),
                "formula": props.get("full_molformula"),
                "exact_mass": em,
                "expected_[M+H]+_mz": round(em + 1.007276, 4),
                "chembl_alogp": props.get("alogp"),
                "rdkit_clogp": rdkit_clogp,
                "psa": props.get("psa"),
                "canonical_smiles": clean_s,
                "image_token": img_token
            })

        return {
            "query_formula": molecular_formula,
            "query_smiles": smiles,
            "chembl_candidates": hits,
            "retention_note": "In Reverse-Phase (C18) HPLC, more polar compounds (lower cLogP / higher PSA) elute earlier."
        }
    except Exception as e:
        return {"error": f"ChEMBL query failed: {e}"}


def query_massbank_reference_spectrum(
    compound_name_or_smiles: str = "",
    exact_mass: float = 0.0,
    max_records: int = 5
) -> dict:
    """
    Queries the MassBank Europe REST API (api.massbank.eu) by InChIKey (from SMILES),
    compound name, or exact neutral mass to retrieve experimental ESI+ MS/MS fragment peaks.
    """
    try:
        max_records = max(1, min(int(max_records), 8))
        params = {"ion_mode": "POSITIVE", "ms_type": "MS2"}
        mol, clean_smi, _ = safe_mol_from_smiles(compound_name_or_smiles)
        if mol:
            params["inchi_key"] = Chem.MolToInchiKey(mol)
        elif compound_name_or_smiles.strip():
            params["compound_name"] = compound_name_or_smiles.strip()
        elif exact_mass > 0:
            params["exact_mass"] = f"{exact_mass:.4f}"
            params["mass_tolerance"] = "0.02"
        else:
            return {"error": "Provide a SMILES string, compound name, or exact_mass."}

        query_str = urllib.parse.urlencode(params)
        search_url = f"https://api.massbank.eu/records/search?{query_str}"
        req = urllib.request.Request(search_url, headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/1.0"})

        records_summary = []
        with urllib.request.urlopen(req, timeout=10) as resp:
            search_res = json.loads(resp.read().decode('utf-8'))

        items = search_res if isinstance(search_res, list) else search_res.get("results", search_res.get("data", []))
        for item in items[:max_records]:
            accession = item if isinstance(item, str) else item.get("accession")
            if not accession:
                continue
            rec_url = f"https://api.massbank.eu/records/{urllib.parse.quote(str(accession), safe='')}"
            req_rec = urllib.request.Request(rec_url, headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/1.0"})
            with urllib.request.urlopen(req_rec, timeout=8) as r_resp:
                rec_data = json.loads(r_resp.read().decode('utf-8'))

            peaks_raw = rec_data.get("peak", {}).get("peak", {}).get("values", [])
            sorted_peaks = sorted(peaks_raw, key=lambda x: x.get("rel", x.get("intensity", 0)), reverse=True)[:10]
            records_summary.append({
                "accession": accession,
                "title": rec_data.get("record_title"),
                "precursor_mz": rec_data.get("ms_focused_ion", {}).get("precursor_m/z"),
                "top_ms2_fragments": [
                    {"mz": round(float(p.get("mz", 0)), 4), "rel_intensity": p.get("rel", p.get("intensity", 0))}
                    for p in sorted_peaks
                ]
            })

        return {
            "query": compound_name_or_smiles or exact_mass,
            "massbank_records_found": len(records_summary),
            "reference_spectra": records_summary
        }
    except Exception as e:
        return {
            "warning": f"Live MassBank REST lookup returned: {e}. Use `disambiguate_isobaric_peak_isomers` for deterministic in-source MS1 fragment matching."
        }


def search_chemical_databases(
    molecular_formula: str = "",
    smiles: str = "",
    observed_mh_mz: float = 0.0
) -> dict:
    """Runs PubChem, ChEMBL, and MassBank searches and registers 2D structure SVG tokens."""
    pubchem_res = search_pubchem_by_mass_or_formula(
        observed_mh_mz=observed_mh_mz,
        molecular_formula=molecular_formula
    )
    chembl_res = (
        search_chembl_metabolites_and_isomers(molecular_formula=molecular_formula, smiles=smiles)
        if (molecular_formula.strip() or smiles.strip())
        else {"note": "Provide molecular_formula or smiles to query ChEMBL."}
    )
    neutral_mass = (observed_mh_mz - 1.007276) if observed_mh_mz > 1.0 else 0.0
    massbank_res = (
        query_massbank_reference_spectrum(compound_name_or_smiles=smiles, exact_mass=neutral_mass)
        if (smiles.strip() or neutral_mass > 0)
        else {"note": "Provide smiles or observed_mh_mz to query MassBank."}
    )
    return {
        "pubchem": pubchem_res,
        "chembl": chembl_res,
        "massbank": massbank_res
    }


def disambiguate_isobaric_peak_isomers(
    peak_id: int,
    candidate_names: list[str],
    candidate_smiles: list[str]
) -> dict:
    """
    Resolves isobaric isomers (e.g., 2'-O-methylcytidine vs 5-methylcytidine) for a given `peak_id`
    by checking raw MS1 scans for isomer-specific glycosidic cleavage ions ([BH2]+ losing
    Ribose -132.0423 Da vs 2'-O-Methylribose -146.0579 Da vs Deoxyribose -116.0473 Da)
    and comparing RDKit Crippen cLogP & TPSA to predict reverse-phase C18 elution order.
    """
    if len(candidate_names) != len(candidate_smiles):
        return {"error": "candidate_names and candidate_smiles must have the same length."}

    isomer_reports = []
    for name, smi in zip(candidate_names, candidate_smiles):
        mol, clean_smi, err = safe_mol_from_smiles(smi)
        if not mol:
            isomer_reports.append({"name": name, "error": err})
            continue

        em = Descriptors.ExactMolWt(mol)
        mh_mz = round(em + 1.007276, 4)
        clogp = round(Crippen.MolLogP(mol), 3)
        tpsa = round(rdMolDescriptors.CalcTPSA(mol), 2)
        img_token = _register_mol_svg(mol, label=f"Candidate: {name}")

        frag_res = generate_in_silico_fragments(clean_smi, check_peak_id=peak_id)
        observed_diagnostic_frags = [
            {
                "fragment_name": f["fragment_name"],
                "expected_mz": f["expected_esi_pos_mz"],
                "neutral_loss_Da": f["neutral_loss_from_parent_Da"],
                "observed_in_peak": f["peak_observation"],
                "fragment_image_token": f["image_token"]
            }
            for f in frag_res.get("generated_fragments", [])
            if f.get("present_in_peak")
        ]

        sugar_checks = []
        for loss_label, loss_da in [
            ("Unmodified Ribose loss (-132.0423 Da -> modified nucleobase [BH2]+)", 132.042259),
            ("2'-O-Methylribose loss (-146.0579 Da -> unmodified nucleobase [BH2]+)", 146.057909),
            ("2'-Deoxyribose loss (-116.0473 Da -> nucleobase [BH2]+)", 116.047344),
        ]:
            expected_bh2_mz = round(mh_mz - loss_da, 4)
            if expected_bh2_mz > 70.0:
                scan_hit = check_mz_in_peak(peak_id, expected_bh2_mz, tolerance_da=0.025)
                sugar_checks.append({
                    "pathway": loss_label,
                    "target_bh2_mz": expected_bh2_mz,
                    "detected_in_raw_ms1": scan_hit.get("found_in_raw_scans", False),
                    "max_hit": scan_hit.get("max_observed_hit")
                })

        isomer_reports.append({
            "candidate_name": name,
            "canonical_smiles": clean_smi,
            "formula": rdMolDescriptors.CalcMolFormula(mol),
            "parent_[M+H]+_mz": mh_mz,
            "rdkit_cLogP": clogp,
            "TPSA_A2": tpsa,
            "structure_image_token": img_token,
            "observed_in_silico_fragments": observed_diagnostic_frags,
            "diagnostic_glycosidic_cleavage_checks": sugar_checks
        })

    return {
        "peak_id": peak_id,
        "isobaric_comparison": isomer_reports,
        "interpretation_guide": (
            "1. Check `diagnostic_glycosidic_cleavage_checks` and `observed_in_silico_fragments` for direct in-source fragment hits. "
            "2. For reverse-phase C18 elution order, lower `rdkit_cLogP` / higher `TPSA_A2` (more polar, e.g. 5-methylcytidine with 3 free ribose OHs) "
            "elutes EARLIER than higher `rdkit_cLogP` / lower `TPSA_A2` (e.g. 2'-O-methylcytidine with a methylated 2'-OH)."
        )
    }


def assign_peaks_and_replot_chromatogram(
    peak_ids: list[int],
    compound_names: list[str],
    smiles_list: list[str],
    classifications: list[str] = None
) -> dict:
    """
    Assigns one or more peaks (including previously unassigned peaks or resolved isobaric peaks)
    to specific chemical structures, calculates mass accuracy & confidence scores against raw MS1
    scans in `od`, updates `detected_df` and `peak_assignments` in place, and generates a brand-new
    interactive Plotly chromatogram with floating RDKit 2D hover cards.
    Returns a `plot_token` (e.g. `[[PLOTLY_CHROM_1]]`) that you MUST place on its own line in your response.
    """
    global detected_df, peak_assignments

    if 'detected_df' not in globals() or 'chrom' not in globals() or chrom is None:
        return {"error": "detected_df or chrom not initialized. Run Cell 4 first."}

    if not (len(peak_ids) == len(compound_names) == len(smiles_list)):
        return {"error": "peak_ids, compound_names, and smiles_list must have the exact same length."}

    if not classifications or len(classifications) != len(peak_ids):
        classifications = ["Metabolite"] * len(peak_ids)

    adduct_defs = globals().get('esi_pos_adducts', [
        ("[M+H]+",   1, 1.007276,  1),
        ("[M+NH4]+", 1, 18.033823, 1),
        ("[M+Na]+",  1, 22.989218, 1),
        ("[M+K]+",   1, 38.963158, 1),
        ("[2M+H]+",  2, 1.007276,  1),
    ])

    updated_summary = []
    for p_id, name, smi, cls_label in zip(peak_ids, compound_names, smiles_list, classifications):
        p_id = int(p_id)
        mol, clean_smi, err = safe_mol_from_smiles(smi)
        if not mol:
            updated_summary.append({"peak_id": p_id, "status": "failed", "error": err})
            continue

        em = round(Descriptors.ExactMolWt(mol), 4)
        formula = rdMolDescriptors.CalcMolFormula(mol)

        observed_adducts = []
        for adduct_name, mult, shift, z in adduct_defs:
            calc_mz = (em * mult + shift) / z
            hit = check_mz_in_peak(p_id, calc_mz, tolerance_da=0.03)
            if hit.get("found_in_raw_scans") and hit.get("max_observed_hit"):
                best_hit = hit["max_observed_hit"]
                observed_adducts.append({
                    "adduct": adduct_name,
                    "obs_mz": best_hit["observed_mz"],
                    "calc_mz": calc_mz,
                    "ppm": best_hit["ppm_error"],
                    "intensity": best_hit["intensity"]
                })

        if not observed_adducts:
            calc_mh = em + 1.007276
            observed_adducts.append({
                "adduct": "[M+H]+",
                "obs_mz": round(calc_mh, 4),
                "calc_mz": calc_mh,
                "ppm": 0.0,
                "intensity": 1e6
            })
        else:
            observed_adducts = sorted(observed_adducts, key=lambda x: x["intensity"], reverse=True)

        row_mask = detected_df['peak_id'] == p_id
        dom_str = detected_df.loc[row_mask, 'dominant m/z'].values[0] if row_mask.any() else "None"
        dom_parsed = parse_dominant_mz_string(dom_str) if 'parse_dominant_mz_string' in globals() else []

        best_group = {
            "exact_mass": em,
            "formula": formula,
            "mols": [mol],
            "adducts": observed_adducts
        }
        if 'compute_confidence_from_dominant_mz' in globals():
            conf_score, evidence_str = compute_confidence_from_dominant_mz(
                best_group, dom_parsed, globals().get('peak_ms1_spectra', {}).get(p_id, {})
            )
        else:
            conf_score, evidence_str = 88.0, f"Assigned ({observed_adducts[0]['ppm']:+.1f} ppm)"

        peak_assignments[p_id] = {
            "display_name": name,
            "full_assignment": f"{name} ({cls_label})",
            "is_isobaric": False,
            "formula": formula,
            "mols": [mol],
            "adducts": observed_adducts,
            "confidence": conf_score,
            "evidence": evidence_str
        }

        top_a = observed_adducts[0]
        if row_mask.any():
            detected_df.loc[row_mask, 'Assignment'] = (
                f"{name} ({cls_label}) [{top_a['adduct']}: {top_a['obs_mz']:.4f} m/z ({top_a['ppm']:+.1f} ppm)]"
            )
            detected_df.loc[row_mask, 'Confidence (%)'] = f"{conf_score:.1f}% ({evidence_str})"

        struct_token = _register_mol_svg(mol, label=f"Peak {p_id}: {name}")
        updated_summary.append({
            "peak_id": p_id,
            "assigned_name": name,
            "classification": cls_label,
            "formula": formula,
            "primary_adduct": top_a["adduct"],
            "observed_mz": top_a["obs_mz"],
            "ppm_error": top_a["ppm"],
            "confidence_pct": conf_score,
            "evidence": evidence_str,
            "structure_image_token": struct_token
        })

    unique_div_id = f"hplc_chrom_replot_{uuid.uuid4().hex[:6]}"
    interactive_html = build_interactive_chromatogram_html(div_id=unique_div_id)

    plot_idx = sum(1 for k in _rendered_svg_registry if k.startswith("[[PLOTLY_CHROM_")) + 1
    plot_token = f"[[PLOTLY_CHROM_{plot_idx}]]"
    _rendered_svg_registry[plot_token] = interactive_html

    return {
        "status": "success",
        "updated_peaks": updated_summary,
        "plot_token": plot_token,
        "rendering_instruction": (
            f"CRITICAL: Place the exact token {plot_token} on its own line in your response "
            "to render the updated interactive Plotly HPLC chromatogram with floating 2D hover cards!"
        )
    }


# --- BIND TOOLS & STATEFUL CHAT UI ---

client = genai.Client(api_key=userdata.get("KDD_GEMINI_API_KEY"))

rdkit_tools = [
    get_user_candidate_molecules,
    get_detected_peaks_summary,
    get_distilled_peak_details,
    get_peak_details,
    check_mz_in_peak,
    generate_in_silico_fragments,
    analyze_structure_and_mass,
    generate_2d_structure_svg,
    search_pubchem_by_mass_or_formula,
    search_chembl_metabolites_and_isomers,
    query_massbank_reference_spectrum,
    search_chemical_databases,
    update_lcms_plot,
    disambiguate_isobaric_peak_isomers,
    assign_peaks_and_replot_chromatogram,
]

**Bonus Tip for Cell 3E:**

* If you edit a tool in Cell 3D while a chat session is already running, click your orange **Reset Chat / Refresh Data button** in the UI (or re-run Cell 3E) so the active chat session picks up the updated rdkit_tools list.

In [ ]:
#@title 8E. Gemini Client, System Prompt & Interactive Chat UI

# --- BIND TOOLS & STATEFUL CHAT UI ---

client = genai.Client(api_key=userdata.get("KDD_GEMINI_API_KEY"))

rdkit_tools = [
    get_user_candidate_molecules,
    get_detected_peaks_summary,
    get_distilled_peak_details,
    get_peak_details,
    check_mz_in_peak,
    generate_in_silico_fragments,
    analyze_structure_and_mass,
    generate_2d_structure_svg,
    search_pubchem_by_mass_or_formula,
    search_chembl_metabolites_and_isomers,
    query_massbank_reference_spectrum,
    search_chemical_databases,
    update_lcms_plot,
    disambiguate_isobaric_peak_isomers,
    assign_peaks_and_replot_chromatogram,
]

def build_system_instruction() -> str:
    candidates_md = get_user_candidate_molecules()
    peaks_md = get_detected_peaks_summary()
    hypotheses = SPECTRAL_STATE.get("hypotheses", [])[:10]
    hyp_str = f"\n**3. Reaction Mass-Shift Hypotheses:**\n{json.dumps(hypotheses)}\n" if hypotheses else ""
    return f"""You are an expert analytical chemist and mass spectrometrist assistant integrated into an HPLC-ESI-MS / GC-MS notebook.

### Live Experimental Context:
**1. User's Candidate Molecules Table:**
{candidates_md}

**2. Deconvoluted HPLC-MS Peaks Summary (`detected_df`):**
{peaks_md}
{hyp_str}
### Mandatory Operating Rules:
1. **Use `generate_in_silico_fragments` First for Fragment Questions:**
   - Whenever the user asks for potential ESI fragments of a molecule (e.g., adenosine) and/or asks if they appear in a specific peak (e.g., Peak 2), call `generate_in_silico_fragments(parent_smiles, check_peak_id=2)`.
   - `generate_in_silico_fragments`, `analyze_structure_and_mass`, `search_pubchem_by_mass_or_formula`, and `search_chembl_metabolites_and_isomers` AUTOMATICALLY render the 2D chemical structure SVG for every molecule and return an `image_token` (such as `[[MOL_IMG_1]]`, `[[MOL_IMG_2]]`, etc.).
2. **How to Display 2D Structure Images:**
   - Copy the exact `parent_image_token`, `structure_image_token`, or `image_token` (e.g., `[[MOL_IMG_1]]`) returned by a tool and place it on its own line right below each molecule or fragment description.
   - Never invent tokens that were not returned by a tool call.
3. **Identifying Unassigned Peaks & Disambiguating Isobaric Isomers:**
   - To identify unassigned peaks, first call `get_peak_details(peak_id)` to inspect the raw MS1 top ions, potential `[M+H]+` / `[M+Na]+` / `[M+NH4]+` adducts, and in-source fragments (such as `-132.0423 Da` ribose loss or `-116.0473 Da` deoxyribose loss).
   - Call `search_pubchem_by_mass_or_formula` and/or `search_chembl_metabolites_and_isomers` (or `search_chemical_databases`) to find matching known metabolites and compare their `rdkit_cLogP` and `PSA` against reverse-phase retention times.
   - To resolve `[Isobaric Isomers]` (e.g. `n2-o-methylcytidine` vs `n5-methylcytidine`), call `disambiguate_isobaric_peak_isomers(peak_id, candidate_names, candidate_smiles)` and/or `query_massbank_reference_spectrum` to evaluate diagnostic glycosidic cleavage ions (`[BH2]+`) and C18 polarity elution order.
4. **Replotting the Interactive Plotly Chromatogram & LCMS Dashboard:**
   - Whenever the user asks to assign new peaks, update peak assignments, or replot the deconvoluted chromatogram, call `assign_peaks_and_replot_chromatogram(peak_ids, compound_names, smiles_list, classifications)`.
   - You MUST copy the returned `plot_token` (e.g. `[[PLOTLY_CHROM_1]]`) and place it on its own line in your markdown response so the live interactive Plotly chromatogram renders in the notebook output!
   - If the user asks for an Extracted Ion Chromatogram (XIC), UV/DAD trace, or theoretical isotope envelope overlay, call `update_lcms_plot`.
"""

curated_stable_models = [
    'gemini-3.8-flash',
    'gemini-3.6-flash',
    'gemini-3.5-flash',
    'gemini-3.5-flash-lite',
    'gemini-3.1-flash-lite',
    'gemini-2.5-pro',
    'gemini-2.5-flash',
    'gemini-2.5-flash-lite',
]

try:
    api_models = [
        m.name.replace('models/', '')
        for m in client.models.list()
        if 'generateContent' in (getattr(m, 'supported_actions', None) or [])
        and m.name.startswith('models/gemini-')
        and not any(x in m.name for x in ['preview', 'exp', 'image', 'tts', 'live', 'transcribe', 'embedding', '1.5', '2.0'])
    ]
    for m_id in sorted(api_models, reverse=True):
        if m_id not in curated_stable_models:
            curated_stable_models.append(m_id)
except Exception:
    pass

model_dropdown = widgets.Dropdown(
    options=curated_stable_models,
    value='gemini-3.8-flash',
    description='Model:',
    disabled=False,
    layout=widgets.Layout(width='320px')
)

reset_button = widgets.Button(
    description='Reset Chat / Refresh Data',
    button_style='warning',
    icon='refresh',
    layout=widgets.Layout(width='200px')
)

chat_input = widgets.Textarea(
    value='',
    placeholder='Ask about your HPLC-MS data (e.g., "Identify the unassigned peaks using PubChem and ChEMBL, resolve any isobaric isomers, and replot the interactive HPLC chromatogram with the new peaks assigned")...',
    description='Input:',
    layout=widgets.Layout(width='85%', height='90px')
)

submit_button = widgets.Button(
    description='Send',
    button_style='primary',
    icon='paper-plane'
)

output_area = widgets.Output(
    layout=widgets.Layout(border='1px solid #e5e7eb', padding='12px', height='750px', overflow_y='auto')
)
output_area.add_class("lcms-chat-output")
chat_style = widgets.HTML(
    "<style>.lcms-chat-output{line-height:1.6;font-size:15px;padding:14px;}"
    ".lcms-chat-output table{border-collapse:collapse;display:block;overflow-x:auto;}"
    ".lcms-chat-output td,.lcms-chat-output th{border:1px solid #cbd5e1;padding:6px 10px;}"
    ".lcms-chat-output pre{background:#f1f5f9;padding:12px;overflow-x:auto;}"
    ".lcms-chat-output code{font-family:monospace;}</style>"
)
reaction_input = widgets.Text(
    placeholder="SM1.SM2>>Product, SM: SMILES, Reagent: SMILES, Product: SMILES, or a single SMILES",
    description="Reaction:", layout=widgets.Layout(width='95%')
)
uv_ms_delay_input = widgets.FloatText(value=0.08, description="UV→MS delay (min):",
                                      style={"description_width": "initial"})
uv_upload = widgets.FileUpload(accept=".csv", multiple=False, description="Upload UV/DAD CSV")
plot_button = widgets.Button(description="Show LCMS dashboard", icon="chart-line")
plot_options = widgets.Text(placeholder="XIC m/z (comma-separated); blank for TIC/BPC",
                            description="XIC:", layout=widgets.Layout(width="65%"))
isotope_input = widgets.Text(placeholder="Formula or SMILES for isotope overlay (optional)",
                             description="Isotopes:", layout=widgets.Layout(width="65%"))
scan_input = widgets.FloatText(value=0, description="MS scan RT (min, 0=auto):",
                               style={"description_width": "initial"})

chat_session = None
active_model_name = None
chat_history_ui = []

def init_chat_session(force_reset=False):
    global chat_session, active_model_name, chat_history_ui
    current_data_id = id(globals().get("od", globals().get("hplc_df")))
    if (force_reset or not SPECTRAL_STATE or
        SPECTRAL_STATE.get("data_id") != current_data_id or
        SPECTRAL_STATE.get("reaction_text") != reaction_input.value):
        try:
            prepare_spectral_state(uv_ms_delay_min=uv_ms_delay_input.value)
        except Exception:
            pass
    else:
        SPECTRAL_STATE["uv_ms_delay_min"] = uv_ms_delay_input.value

    if force_reset or chat_session is None or active_model_name != model_dropdown.value:
        active_model_name = model_dropdown.value
        chat_session = client.chats.create(
            model=active_model_name,
            config=types.GenerateContentConfig(
                system_instruction=build_system_instruction(),
                tools=rdkit_tools,
                temperature=0.2,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=False,
                    maximum_remote_calls=30
                )
            )
        )
        if force_reset:
            chat_history_ui.clear()
            _rendered_svg_registry.clear()
            with output_area:
                clear_output()
                print("🔄 Chat session reset and synced with latest HPLC-MS DataFrame.")

def extract_response_text(response) -> str:
    text_chunks = []
    if response.candidates:
        for part in response.candidates[0].content.parts:
            if getattr(part, 'text', None):
                text_chunks.append(part.text)
    return "".join(text_chunks) if text_chunks else "*(No text returned.)*"

def render_message_with_svgs(markdown_text: str):
    """
    Splits the response on [[MOL_IMG_X]], {{SVG_X}}, or [[PLOTLY_CHROM_X]] placeholder tokens
    and renders the stored RDKit 2D image cards or interactive Plotly chromatograms via IPython.display.HTML.
    """
    pattern = r'(\[\[MOL_IMG_\d+\]\]|\{\{SVG_\d+\}\}|\[\[PLOTLY_CHROM_\d+\]\])'
    parts = re.split(pattern, markdown_text)
    for part in parts:
        part_stripped = part.strip()
        if part_stripped in _rendered_svg_registry:
            display(HTML(_rendered_svg_registry[part_stripped]))
        elif part_stripped:
            display(Markdown(part))

def on_submit_clicked(b):
    user_text = chat_input.value.strip()
    if not user_text:
        return

    chat_input.disabled = True
    submit_button.disabled = True

    try:
        init_chat_session()
        with output_area:
            print(f"👤 You: {user_text}\n")
            print(f"🤖 {model_dropdown.value} is analyzing with RDKit, PubChem, ChEMBL & Plotly tools...\n")

        response = chat_session.send_message(user_text)
        reply_text = extract_response_text(response)

        chat_history_ui.append(("user", user_text))
        chat_history_ui.append(("model", f"**🤖 Gemini [{model_dropdown.value}]:**\n\n{reply_text}"))

        with output_area:
            clear_output(wait=True)
            for role, msg in chat_history_ui:
                if role == "user":
                    display(Markdown(f"**👤 You:** {msg}"))
                else:
                    render_message_with_svgs(msg)
                display(HTML("<hr style='border: 0; border-top: 1px solid #374151; margin: 14px 0;'>"))
            pending_plot = SPECTRAL_STATE.pop("pending_plot_html", None)
            if pending_plot:
                display(HTML(pending_plot))

    except Exception as e:
        with output_area:
            print(f"❌ Error: {e}")

    finally:
        chat_input.value = ''
        chat_input.disabled = False
        submit_button.disabled = False

def on_reset_clicked(b):
    init_chat_session(force_reset=True)

def on_uv_upload(change):
    uploaded = uv_upload.value
    if not uploaded:
        return
    item = next(iter(uploaded.values())) if isinstance(uploaded, dict) else uploaded[0]
    content = item["content"] if isinstance(item, dict) else item.content
    try:
        if len(content) > 5_000_000:
            raise ValueError("UV CSV must be under 5 MB.")
        frame = pd.read_csv(io.BytesIO(bytes(content)))
        time_col = "Time" if "Time" in frame else "time_min"
        channels = register_uv_trace(frame, time_col=time_col)
        with output_area:
            display(Markdown(f"UV/DAD loaded: {', '.join(html.escape(c) for c in channels)}. Times interpreted as minutes."))
    except (ValueError, KeyError) as exc:
        with output_area:
            display(Markdown(f"UV/DAD upload failed: {exc}"))

def on_plot_clicked(b):
    try:
        if ("times" not in SPECTRAL_STATE or
            SPECTRAL_STATE.get("data_id") != id(globals().get("od", globals().get("hplc_df"))) or
            SPECTRAL_STATE.get("reaction_text") != reaction_input.value):
            prepare_spectral_state(uv_ms_delay_min=uv_ms_delay_input.value)
        SPECTRAL_STATE["uv_ms_delay_min"] = uv_ms_delay_input.value
        status = update_lcms_plot(
            extracted_ion_mzs_csv=plot_options.value.strip(),
            select_scan_rt_min=float(scan_input.value) if scan_input.value > 0 else 0.0,
            overlay_formula_isotopes=isotope_input.value.strip()
        )
        with output_area:
            display(Markdown(status))
            if SPECTRAL_STATE.get("pending_plot_html"):
                display(HTML(SPECTRAL_STATE.pop("pending_plot_html")))
    except ValueError as exc:
        with output_area:
            display(Markdown(f"Invalid plot options: {exc}"))

submit_button.on_click(on_submit_clicked)
reset_button.on_click(on_reset_clicked)
uv_upload.observe(on_uv_upload, names="value")
plot_button.on_click(on_plot_clicked)

ui = widgets.VBox([
    chat_style,
    widgets.HTML("<h3>💬 Gemini + RDKit / PubChem / ChEMBL / MassBank HPLC-MS Assistant</h3>"),
    widgets.HBox([model_dropdown, reset_button]),
    reaction_input,
    widgets.HBox([uv_upload, uv_ms_delay_input]),
    widgets.HBox([plot_options, scan_input]),
    widgets.HBox([isotope_input, plot_button]),
    chat_input,
    submit_button,
    output_area
])
display(ui)

In [ ]:
#@title Step 3 (Complete Self-Contained Script): RDKit + PubChem/ChEMBL/MassBank Tools + Interactive Plotly Replotting

import re
import io
import json
import uuid
import base64
import html
import math
import sqlite3
import urllib.request
import urllib.parse
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import pubchempy as pcp
from scipy.integrate import trapezoid
from scipy.signal import find_peaks, peak_widths
from google.colab import userdata
from google import genai
from google.genai import types
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, Crippen, Draw
from rdkit.Chem.Draw import rdMolDraw2D

# 1. Silence RDKit C++ warnings
RDLogger.DisableLog('rdApp.*')

# Global dictionary mapping placeholder tokens ('[[MOL_IMG_X]]', '{{SVG_X}}', '[[PLOTLY_CHROM_X]]') to HTML blocks
_rendered_svg_registry = {}
SPECTRAL_STATE = {}
CHEM_CACHE_PATH = "/tmp/lcms_chem_cache.sqlite"
BENCH_CONTAMINANTS = {
    "TPPO": 279.093, "DIPEA": 130.159, "Et3N": 102.128,
    "DMAP": 123.092, "DCU": 225.196, "DIU": 145.134,
    "dibutyl phthalate": 279.159,
}
NEUTRAL_LOSSES = {
    "H2O": 18.0106, "NH3": 17.0265, "CO/N2": 28.0061,
    "HCl": 35.9767, "HCl (average)": 36.46, "CO2": 43.9898, "isobutene": 56.0626,
    "Boc": 100.0524,
}

if 'to_plotly_color' not in globals():
    def to_plotly_color(color_val):
        rgba = mcolors.to_rgba(color_val)
        return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"


def _register_mol_svg(mol, label: str = "", width: int = 300, height: int = 220) -> str:
    """
    Renders an RDKit Mol object to a Base64-encoded SVG <img> HTML card,
    stores it in `_rendered_svg_registry` under both `[[MOL_IMG_N]]` and `{{SVG_N}}`,
    and returns the token `[[MOL_IMG_N]]`.
    """
    mol_copy = Chem.Mol(mol)
    AllChem.Compute2DCoords(mol_copy)

    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addStereoAnnotation = True
    opts.padding = 0.12

    exact_mass = Descriptors.ExactMolWt(mol_copy)
    formula = rdMolDescriptors.CalcMolFormula(mol_copy)
    f_charge = Chem.GetFormalCharge(mol_copy)
    mh_mz = exact_mass + 1.007276 if f_charge == 0 else exact_mass - (f_charge * 0.00054858)

    drawer.DrawMolecule(mol_copy)
    drawer.FinishDrawing()
    svg_str = drawer.GetDrawingText()

    b64_svg = base64.b64encode(svg_str.encode('utf-8')).decode('utf-8')
    mol_count = sum(1 for k in _rendered_svg_registry if k.startswith("[[MOL_IMG_")) + 1

    caption_title = f"<div style='font-weight:700; font-size:12px; color:#111827; margin-bottom:4px;'>{label}</div>" if label else ""
    caption_sub = f"<div style='font-size:11px; color:#374151; margin-top:4px;'><b>{formula}</b> | Exact: {exact_mass:.4f} | m/z: {mh_mz:.4f}</div>"

    html_card = (
        f"<div style='display:inline-block; margin:10px 6px; padding:10px 14px; border:2px solid #cbd5e1; "
        f"border-radius:10px; background:#ffffff; text-align:center; box-shadow:0 2px 6px rgba(0,0,0,0.15);'>"
        f"{caption_title}"
        f"<img src='data:image/svg+xml;base64,{b64_svg}' width='{width}' height='{height}' style='display:block; margin:0 auto; background:#ffffff;'/>"
        f"{caption_sub}"
        f"</div>"
    )

    token_primary = f"[[MOL_IMG_{mol_count}]]"
    token_alt = f"{{{{SVG_{mol_count}}}}}"
    _rendered_svg_registry[token_primary] = html_card
    _rendered_svg_registry[token_alt] = html_card
    return token_primary


def safe_mol_from_smiles(smiles_or_str: str):
    """Auto-repairs unkekulized pyrrole/purine/imidazole nitrogens (missing [nH])."""
    s = smiles_or_str.strip()
    if not s:
        return None, None, "Empty input string."

    if re.fullmatch(r'([A-Z][a-z]?\d*)+([+-]\d*)?', s) and not any(c in s for c in ['c', 'n', 'o', 's', '=', '#', '(', ')', '[']):
        if any(char.isdigit() for char in s) and not re.search(r'[1-9].*[1-9]', s[:3]):
            return None, None, f"'{s}' is a molecular formula, not a SMILES string."

    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        return mol, Chem.CanonSmiles(s), None

    mol_no_san = Chem.MolFromSmiles(s, sanitize=False)
    if mol_no_san is not None:
        try:
            mol_no_san.UpdatePropertyCache(strict=False)
            Chem.GetSymmSSSR(mol_no_san)
            for atom in mol_no_san.GetAtoms():
                if atom.GetAtomicNum() == 7 and atom.GetIsAromatic() and atom.GetDegree() == 2 and atom.GetFormalCharge() == 0 and atom.GetNumExplicitHs() == 0:
                    test_mol = Chem.Mol(mol_no_san)
                    test_mol.GetAtomWithIdx(atom.GetIdx()).SetNumExplicitHs(1)
                    try:
                        Chem.SanitizeMol(test_mol)
                        return test_mol, Chem.MolToSmiles(test_mol), None
                    except Exception:
                        continue
        except Exception:
            pass

    return None, None, f"RDKit could not parse '{s}'. Ensure aromatic 5-membered ring NH groups are written as '[nH]'."


# --- LOCAL SPECTRAL STATE & UV/DAD DASHBOARD HELPERS ---

def calculate_expected_ions(smiles: str) -> dict:
    """Monoisotopic ESI ions (singly charged) from a neutral, valid SMILES."""
    mol, clean, error = safe_mol_from_smiles(smiles)
    if mol is None:
        return {"error": error}
    mass = Descriptors.ExactMolWt(mol)
    positive = {
        "[M+H]+": mass + 1.0073, "[M+NH4]+": mass + 18.0338,
        "[M+Na]+": mass + 22.9892, "[M+K]+": mass + 38.9632,
        "[M+CH3CN+H]+": mass + 42.0338,
        "[2M+H]+": 2 * mass + 1.0073, "[2M+Na]+": 2 * mass + 22.9892,
    }
    negative = {
        "[M-H]-": mass - 1.0073, "[M+Cl]-": mass + 34.9694,
        "[M+HCOO]-": mass + 44.9982, "[M+CH3COO]-": mass + 59.0139,
    }
    return {
        "smiles": clean, "formula": rdMolDescriptors.CalcMolFormula(mol),
        "exact_mass": round(mass, 5),
        "cLogP": round(Crippen.MolLogP(mol), 2),
        "TPSA": round(rdMolDescriptors.CalcTPSA(mol), 1),
        "positive": {k: round(v, 4) for k, v in positive.items()},
        "negative": {k: round(v, 4) for k, v in negative.items()},
    }


def parse_reaction_candidates(text: str) -> list[dict]:
    """Accept SMILES, reaction SMILES, or labeled/positional comma-separated SMILES."""
    text = text.strip()
    if not text:
        return []
    if ">" in text:
        parts = text.split(">")
        if len(parts) != 3:
            return []
        entries = (
            [(s, "Starting Material") for s in parts[0].split(".")] +
            [(s, "Reagent") for s in parts[1].split(".")] +
            [(s, "Target Product") for s in parts[2].split(".")]
        )
    elif "," in text:
        pieces = [s.strip() for s in text.split(",") if s.strip()]
        entries = []
        for i, piece in enumerate(pieces):
            name, sep, smi = piece.partition(":")
            if sep:
                role = {"sm": "Starting Material", "starting material": "Starting Material",
                        "reagent": "Reagent", "reagents": "Reagent",
                        "product": "Target Product", "target product": "Target Product"}.get(name.strip().lower())
                entries.append((smi.strip(), role or "Reagent"))
            else:
                entries.append((piece, "Starting Material" if i == 0 else
                                "Target Product" if i == len(pieces) - 1 else "Reagent"))
    else:
        entries = [(text, "Target Product")]
    return [
        {"role": role, **ions}
        for smi, role in entries if (ions := calculate_expected_ions(smi)) and "error" not in ions
    ]


def reaction_mass_hypotheses(candidates: list[dict]) -> list[dict]:
    """Mass shifts are diagnostic hypotheses, not claimed molecular structures."""
    products = [c for c in candidates if c["role"] in ("Target Product", "Product")]
    if not products:
        return []
    product = products[0]
    hypotheses = []
    product_mol = Chem.MolFromSmiles(product["smiles"]) if product.get("smiles") else None
    for candidate in candidates:
        if candidate["role"] in ("Starting Material", "Reactant"):
            hypotheses.append({
                "name": "Unreacted SM", "mz": candidate["positive"]["[M+H]+"],
                "delta": round(candidate["exact_mass"] - product["exact_mass"], 4),
                "cLogP": candidate["cLogP"], "TPSA": candidate["TPSA"],
            })
    transformations = {
        "Hydrolysis": ("[C]#[N]", "C(=O)N"),
        "Des-Br": ("[Br]", None),
        "Des-Cl": ("[Cl]", None),
        "Protodeboronation": ("[B]([O;H1])[O;H1]", None),
        "Boc cleavage": ("[C](=[O])[O][C]([CH3])([CH3])[CH3]", None),
        "In-source isobutene / t-Bu ester": ("[O][C]([CH3])([CH3])[CH3]", "O"),
    }
    for name, shift in (
        ("Hydrolysis", 18.0106), ("Des-Br", -77.9105),
        ("Des-Cl", -33.9610), ("Protodeboronation", -43.9900),
        ("Boc cleavage", -100.0524), ("In-source isobutene / t-Bu ester", -56.0626),
        ("Fmoc cleavage", -222.0681),
    ):
        if name == "Des-Br" and "Br" not in product["formula"]:
            continue
        if name == "Des-Cl" and "Cl" not in product["formula"]:
            continue
        if name == "Protodeboronation" and "B" not in product["formula"]:
            continue
        mz = product["positive"]["[M+H]+"] + shift
        if mz > 0:
            entry = {"name": name, "mz": round(mz, 4), "delta": shift,
                     "cLogP": None, "TPSA": None}
            if product_mol and name in transformations:
                pattern, replacement = transformations[name]
                query = Chem.MolFromSmarts(pattern)
                if query and product_mol.HasSubstructMatch(query):
                    try:
                        if replacement:
                            altered = Chem.ReplaceSubstructs(product_mol, query,
                                Chem.MolFromSmiles(replacement), replaceAll=False)[0]
                        else:
                            altered = Chem.DeleteSubstructs(product_mol, query, onlyFrags=False)
                        Chem.SanitizeMol(altered)
                        if abs(Descriptors.ExactMolWt(altered) -
                               (product["exact_mass"] + shift)) < .05:
                            entry.update(smiles=Chem.MolToSmiles(altered),
                                cLogP=round(Crippen.MolLogP(altered), 2),
                                TPSA=round(rdMolDescriptors.CalcTPSA(altered), 1))
                    except Exception:
                        pass
            hypotheses.append(entry)
    return hypotheses


def theoretical_isotope_envelope(formula_or_smiles: str) -> list[dict]:
    """Natural-abundance nominal M..M+4 envelope with exact centroid mass offsets."""
    mol, _, _ = safe_mol_from_smiles(formula_or_smiles)
    formula = rdMolDescriptors.CalcMolFormula(mol) if mol is not None else formula_or_smiles.strip()
    matches = list(re.finditer(r"([A-Z][a-z]?)(\d*)", formula))
    if not matches or "".join(m.group(0) for m in matches) != formula:
        raise ValueError("Provide a neutral molecular formula or valid SMILES.")
    isotope = {
        "C": [(0, 0, .9893), (1, 1.003355, .0107)],
        "H": [(0, 0, .999885), (1, 1.006277, .000115)],
        "N": [(0, 0, .99636), (1, .997035, .00364)],
        "O": [(0, 0, .99757), (1, 1.004217, .00038), (2, 2.004245, .00205)],
        "S": [(0, 0, .9499), (1, .999388, .0075), (2, 1.995796, .0425), (4, 3.995010, .0001)],
        "Cl": [(0, 0, .7578), (2, 1.997050, .2422)],
        "Br": [(0, 0, .5069), (2, 1.997953, .4931)],
    }
    pt = Chem.GetPeriodicTable()
    mass = 0.0
    distribution = {0: (1.0, 0.0)}
    for match in matches:
        element, count = match.group(1), int(match.group(2) or 1)
        if pt.GetAtomicNumber(element) == 0 or count > 300:
            raise ValueError("Invalid or oversized molecular formula.")
        mass += pt.GetMostCommonIsotopeMass(element) * count
        for _ in range(count):
            next_dist = {}
            for shift, (prob, weighted) in distribution.items():
                for step, delta, abundance in isotope.get(element, [(0, 0.0, 1.0)]):
                    if shift + step > 4:
                        continue
                    p, w = next_dist.get(shift + step, (0.0, 0.0))
                    next_dist[shift + step] = (p + prob * abundance,
                                               w + (weighted + prob * delta) * abundance)
            distribution = next_dist
    max_prob = max(p for p, _ in distribution.values())
    return [
        {"label": f"M+{shift}" if shift else "M",
         "mz": round(mass + 1.007276 + weighted / prob, 5),
         "relative_intensity": round(100 * prob / max_prob, 2)}
        for shift in (0, 1, 2, 4)
        if (prob := distribution.get(shift, (0, 0))[0]) > 0
        for weighted in [distribution[shift][1]]
    ]


def annotate_neutral_losses(ions: list[dict], tolerance_da: float = 0.04) -> list[dict]:
    """Match neutral losses among the ten most intense baseline-subtracted ions."""
    losses = []
    top = sorted(ions, key=lambda ion: ion["intensity"], reverse=True)[:10]
    for parent in top:
        for fragment in top:
            delta = parent["mz"] - fragment["mz"]
            for name, mass in NEUTRAL_LOSSES.items():
                if abs(delta - mass) <= tolerance_da:
                    losses.append({"loss": name, "parent": parent["mz"], "fragment": fragment["mz"]})
    return losses


def _centroid_scans(scans: list[tuple], rt: float, onset: float) -> list[dict]:
    """Average three nearest MS1 scans; subtract a preceding background scan by m/z bin."""
    if not scans:
        return []
    closest = int(np.argmin([abs(s[0] - rt) for s in scans]))
    selected = scans[max(0, closest - 1):closest + 2]
    def bins(spectrum):
        result = {}
        for mz, intensity in zip(spectrum[1], spectrum[2]):
            key = round(float(mz), 2)
            if intensity > 0:
                total, weighted_mz = result.get(key, (0.0, 0.0))
                result[key] = (total + float(intensity),
                               weighted_mz + float(mz * intensity))
        return result
    averaged = {}
    for scan in selected:
        for mz, (intensity, weighted_mz) in bins(scan).items():
            total, weighted = averaged.get(mz, (0.0, 0.0))
            averaged[mz] = (total + intensity / len(selected),
                            weighted + weighted_mz / len(selected))
    preceding_idx = [i for i, scan in enumerate(scans) if scan[0] < onset]
    if preceding_idx:
        best_bg_idx = min(preceding_idx, key=lambda i: abs(scans[i][0] - (onset - 0.05)))
        background = bins(scans[best_bg_idx])
    else:
        background = {}
    corrected = {mz: (max(0.0, intensity - background.get(mz, (0.0, 0.0))[0]),
                      weighted / intensity)
                 for mz, (intensity, weighted) in averaged.items() if intensity > 0}
    if not corrected:
        return []
    threshold = max(intensity for intensity, _ in corrected.values()) * .02
    return [{"mz": round(centroid, 4), "intensity": round(intensity, 1)}
            for mz, (intensity, centroid) in sorted(corrected.items(),
                                                    key=lambda item: -item[1][0])[:10]
            if intensity >= threshold]


def _flag_artifacts(ions: list[dict]) -> list[str]:
    masses = [ion["mz"] for ion in ions]
    flags = [name for name, mz in BENCH_CONTAMINANTS.items()
             if any(abs(observed - mz) <= .03 for observed in masses)]
    if any(abs(a - b - 44.026) <= .03 for a in masses for b in masses if a > b) or any(
        abs(mz - (18.0106 + n * 44.0262 + adduct)) <= .03
        for mz in masses for n in range(6, 13) for adduct in (1.0073, 22.9892)
    ):
        flags.append("PEG-400")
    return flags


def prepare_spectral_state(max_peaks: int = 6, uv_ms_delay_min: float = 0.08) -> dict:
    """Keep raw arrays locally; only small peak metadata may enter the model context."""
    if not 1 <= max_peaks <= 20 or not math.isfinite(uv_ms_delay_min) or uv_ms_delay_min < 0:
        raise ValueError("Use 1-20 peaks and a nonnegative UV/MS delay.")
    scans = []
    if "od" in globals():
        for spec in od:
            if spec.getMSLevel() == 1:
                mzs, intensities = spec.get_peaks()
                scans.append((float(spec.getRT()) / 60.0, np.asarray(mzs), np.asarray(intensities)))
    scans.sort(key=lambda s: s[0])
    times = np.array([s[0] for s in scans])
    tic = np.array([np.sum(s[2]) for s in scans])
    bpc = np.array([np.max(s[2]) if len(s[2]) else 0 for s in scans])
    if not scans and "hplc_df" in globals():
        times = np.asarray(hplc_df["Time"], dtype=float) / 60.0
        tic = np.asarray(hplc_df["Signal"], dtype=float)
        bpc = np.zeros_like(tic)
    signal = np.maximum(0, tic - np.percentile(tic, 10)) if len(tic) else tic
    indices = find_peaks(signal, prominence=max(float(np.max(signal)) * .05, 1e-10),
                         distance=3)[0] if len(signal) >= 3 else []
    indices = sorted(indices, key=lambda i: -signal[i])[:max_peaks]
    candidates = parse_reaction_candidates(reaction_input.value) if "reaction_input" in globals() else []
    if not candidates and "get_candidates_from_table" in globals():
        candidates = [
            {"role": classification, **ions}
            for _, classification, smi in get_candidates_from_table()
            if (ions := calculate_expected_ions(smi)) and "error" not in ions
        ]
    hypotheses = reaction_mass_hypotheses(candidates)
    peaks = []
    for idx in indices:
        width = peak_widths(signal, [idx], rel_height=.8)
        onset = float(times[max(0, int(width[2][0]))])
        ions = _centroid_scans(scans, float(times[idx]), onset)
        matches = [c for c in candidates
                   if any(abs(ion["mz"] - mz) <= max(.03, mz * 15e-6)
                          for mz in c["positive"].values() for ion in ions)]
        peaks.append({
            "rt_min": round(float(times[idx]), 3), "onset_min": onset,
            "ions": ions, "artifacts": _flag_artifacts(ions),
            "neutral_losses": annotate_neutral_losses(ions),
            "candidates": [{"role": c["role"], "formula": c["formula"],
                            "cLogP": c["cLogP"], "TPSA": c["TPSA"]} for c in matches],
        })
    SPECTRAL_STATE.update(scans=scans, times=times, tic=tic, bpc=bpc, peaks=peaks,
                          candidates=candidates, hypotheses=hypotheses,
                          uv_ms_delay_min=uv_ms_delay_min,
                          data_id=id(globals().get("od", globals().get("hplc_df"))),
                          reaction_text=reaction_input.value if "reaction_input" in globals() else "")
    SPECTRAL_STATE.pop("pending_plot_html", None)
    return SPECTRAL_STATE


def register_uv_trace(uv_data: pd.DataFrame, time_col: str = "Time",
                      time_unit: str = "min") -> list[str]:
    """Attach an optional DAD CSV (Time, 254 nm, 214 nm, etc.) without replacing MS data."""
    if time_unit not in ("min", "s") or time_col not in uv_data:
        raise ValueError("Provide a Time column and specify 'min' or 's'.")
    times = np.asarray(uv_data[time_col], dtype=float) / (60.0 if time_unit == "s" else 1.0)
    if not len(times) or not np.all(np.isfinite(times)) or np.any(np.diff(times) <= 0):
        raise ValueError("UV retention times must be finite and strictly increasing.")
    traces = {}
    for column in uv_data.columns:
        if column == time_col:
            continue
        values = np.asarray(uv_data[column], dtype=float)
        if len(values) == len(times) and np.all(np.isfinite(values)):
            traces[str(column)] = values
    SPECTRAL_STATE["uv"] = {"times": times, "traces": traces}
    return list(traces)


def match_uv_apex_to_ms_scan(uv_rt_min: float, uv_ms_delay_min: float = 0.08):
    """Map a UV apex to the closest MS1 scan, compensating for detector dead volume."""
    scans = SPECTRAL_STATE.get("scans", [])
    if not scans:
        return None
    if uv_ms_delay_min < 0:
        raise ValueError("UV/MS delay must be nonnegative.")
    target = uv_rt_min + uv_ms_delay_min
    return min(scans, key=lambda scan: abs(scan[0] - target))[0]


def _xic(scans: list[tuple], target_mz: float, ppm: float) -> np.ndarray:
    return np.array([
        float(np.sum(scan[2][np.abs(scan[1] - target_mz) <= target_mz * ppm * 1e-6]))
        for scan in scans
    ])


def update_lcms_plot(
    extracted_ion_mzs_csv: str = "",
    rt_start_min: float = 0.0,
    rt_end_min: float = 0.0,
    ppm_tolerance: float = 15.0,
    select_scan_rt_min: float = 0.0,
    overlay_formula_isotopes: str = ""
) -> str:
    """
    Updates the local multi-panel LCMS Plotly dashboard (UV/DAD, MS1 TIC/BPC/XIC in minutes,
    and optional MS1 scan with theoretical isotope envelope overlay).
    - extracted_ion_mzs_csv: Comma-separated target m/z values for XIC traces (e.g. '268.1041, 252.1091').
    - rt_start_min, rt_end_min: Optional retention time zoom window in minutes (leave 0.0 for full range).
    - ppm_tolerance: Mass tolerance in ppm (default 15.0).
    - select_scan_rt_min: Retention time in minutes for MS1 spectrum bar plot (0.0 for auto).
    - overlay_formula_isotopes: Optional molecular formula or SMILES to overlay theoretical isotope bars.
    """
    if not SPECTRAL_STATE or "times" not in SPECTRAL_STATE:
        prepare_spectral_state()
    if not 0 < ppm_tolerance <= 1000:
        return "Invalid ppm tolerance."

    extracted_ion_mz = []
    if extracted_ion_mzs_csv.strip():
        try:
            extracted_ion_mz = [float(x.strip()) for x in extracted_ion_mzs_csv.split(",") if x.strip()]
        except ValueError:
            return "Invalid extracted_ion_mzs_csv format; provide comma-separated numbers."

    if len(extracted_ion_mz) > 8 or any(not math.isfinite(mz) or mz <= 0 for mz in extracted_ion_mz):
        return "Provide at most eight positive finite extracted-ion masses."

    rt_window = [rt_start_min, rt_end_min] if (rt_end_min > rt_start_min >= 0.0) else None
    select_scan_rt = select_scan_rt_min if select_scan_rt_min > 0.0 else None
    iso_str = overlay_formula_isotopes.strip()

    if iso_str and not SPECTRAL_STATE.get("scans"):
        return "No MS1 scan is available for an isotope overlay."
    try:
        envelope = theoretical_isotope_envelope(iso_str) if iso_str else []
    except (ValueError, RuntimeError) as exc:
        return f"Invalid isotope formula: {exc}"

    scans = SPECTRAL_STATE.get("scans", [])
    times = SPECTRAL_STATE["times"]
    tic = SPECTRAL_STATE["tic"]
    bpc = SPECTRAL_STATE["bpc"]
    uv = SPECTRAL_STATE.get("uv")
    show_ms = bool(scans) and (select_scan_rt is not None or bool(envelope))
    fig = make_subplots(rows=3 if show_ms else 2, cols=1, shared_xaxes=False,
                        row_heights=[.32, .36, .32] if show_ms else [.44, .56],
                        vertical_spacing=.09,
                        subplot_titles=("UV/DAD (if supplied)", "MS1 chromatogram (minutes)") +
                                       (("MS1 scan (m/z)",) if show_ms else ()))
    if uv:
        uv_times = uv["times"]
        for wavelength, values in uv["traces"].items():
            fig.add_trace(go.Scatter(x=uv_times, y=values, name=f"UV {wavelength}",
                                     mode="lines"), row=1, col=1)
            baseline = float(np.percentile(values, 10))
            corrected = np.maximum(0, values - baseline)
            apices = find_peaks(corrected, prominence=max(float(np.max(corrected)) * .05, 1e-10),
                                distance=3)[0] if len(values) >= 3 else []
            if len(apices):
                widths = peak_widths(corrected, apices, rel_height=.8)
                bounds = [(max(0, int(left)), min(len(values), int(right) + 1))
                          for left, right in zip(widths[2], widths[3])]
                areas = [float(trapezoid(corrected[left:right], uv_times[left:right]))
                         for left, right in bounds]
                total = sum(areas)
                for idx, area in zip(apices, areas):
                    if total:
                        fig.add_annotation(x=float(uv_times[idx]), y=float(values[idx]),
                                           text=f"{area / total * 100:.1f}%", showarrow=True,
                                           arrowhead=2, row=1, col=1)
    else:
        fig.add_annotation(x=.5, y=.5, xref="x domain", yref="y domain",
                           text="UV/DAD not supplied — upload a Time + wavelength CSV",
                           showarrow=False, row=1, col=1)
    fig.add_trace(go.Scatter(x=times, y=tic, mode="lines", name="TIC"), row=2, col=1)
    fig.add_trace(go.Scatter(x=times, y=bpc, mode="lines", name="BPC",
                             visible="legendonly" if len(bpc) and len(tic) and
                             np.max(bpc) < np.max(tic) * .05 else True), row=2, col=1)
    for mz in extracted_ion_mz:
        if scans:
            fig.add_trace(go.Scatter(x=times, y=_xic(scans, mz, ppm_tolerance),
                                     mode="lines", name=f"XIC {mz:.4f}"), row=2, col=1)
    for peak in SPECTRAL_STATE.get("peaks", []):
        fig.add_vline(x=peak["rt_min"], line_dash="dot", line_color="#64748b", row=2, col=1)
    for h in SPECTRAL_STATE.get("hypotheses", []):
        if not scans:
            break
        intensities = _xic(scans, h["mz"], ppm_tolerance)
        indices = find_peaks(intensities, prominence=max(float(np.max(intensities)) * .05, 1e-10))[0] if len(intensities) >= 3 else []
        if len(indices):
            apex = float(times[max(indices, key=lambda idx: intensities[idx])])
            fig.add_vline(x=apex, line_dash="dash",
                          line_color="#e11d48" if "SM" in h["name"] else "#7c3aed",
                          annotation_text=html.escape(h["name"]), row=2, col=1)
    if rt_window:
        fig.update_xaxes(range=rt_window, row=2, col=1)
        fig.update_xaxes(range=rt_window, row=1, col=1)
    if show_ms:
        if select_scan_rt is None and uv and uv["traces"]:
            first = next(iter(uv["traces"].values()))
            select_scan_rt = match_uv_apex_to_ms_scan(float(uv["times"][int(np.argmax(first))]),
                                                       SPECTRAL_STATE.get("uv_ms_delay_min", 0.08))
        if select_scan_rt is None:
            select_scan_rt = SPECTRAL_STATE["peaks"][0]["rt_min"] if SPECTRAL_STATE.get("peaks") else scans[0][0]
        scan = min(scans, key=lambda s: abs(s[0] - select_scan_rt))
        fig.add_trace(go.Bar(x=scan[1], y=scan[2], name=f"Scan at {scan[0]:.3f} min",
                             marker_color="#2563eb"), row=3, col=1)
        if envelope:
            observed = float(_xic([scan], envelope[0]["mz"], ppm_tolerance)[0])
            scale = max(observed, float(np.max(scan[2])) * .1 if len(scan[2]) else 0, 1)
            fig.add_trace(go.Bar(
                x=[entry["mz"] for entry in envelope],
                y=[entry["relative_intensity"] * scale / 100.0 for entry in envelope],
                width=.08, opacity=.45, marker=dict(color="#e11d48",
                                                      line=dict(color="#e11d48", width=2)),
                name=f"Theoretical {iso_str}",
                text=[entry["label"] for entry in envelope], hovertemplate="%{text}: %{x:.4f}<extra></extra>"
            ), row=3, col=1)
        for peak in SPECTRAL_STATE.get("peaks", []):
            if abs(peak["rt_min"] - scan[0]) < .06:
                for loss in peak["neutral_losses"][:4]:
                    mask = np.abs(scan[1] - loss["fragment"]) < .03
                    if np.any(mask):
                        fig.add_annotation(x=loss["fragment"], y=float(np.max(scan[2][mask])),
                                           text=f"-{loss['loss']}", showarrow=True,
                                           ay=-50, row=3, col=1)
        fig.update_xaxes(title_text="m/z", row=3, col=1)
    fig.update_xaxes(title_text="Retention time (min)", row=2, col=1)
    fig.update_yaxes(title_text="UV absorbance", row=1, col=1)
    fig.update_yaxes(title_text="MS intensity", row=2, col=1)
    fig.update_layout(title="LCMS reaction monitoring", template="plotly_white",
                      barmode="overlay", height=850 if show_ms else 660)
    SPECTRAL_STATE["figure"] = fig
    SPECTRAL_STATE["pending_plot_html"] = fig.to_html(include_plotlyjs="cdn", full_html=False,
                                                       div_id=f"lcms_{uuid.uuid4().hex[:8]}")
    return "LCMS dashboard updated locally; the plot will be displayed below your response."


# --- SELF-CONTAINED INTERACTIVE PLOTLY CHROMATOGRAM BUILDER ---

def _peak_label_for_legend(p_id: int) -> str:
    if 'peak_assignments' in globals() and p_id in peak_assignments:
        a_info = peak_assignments[p_id]
        iso_short = " [Isobaric Isomers]" if a_info.get("is_isobaric", False) else ""
        adduct_summary = ", ".join([f"{x['adduct']} ({x['obs_mz']:.2f})" for x in a_info['adducts']])
        return f"Peak {p_id}: {a_info['display_name']}{iso_short} [{adduct_summary}] (Conf: {a_info['confidence']:.0f}%)"
    if 'detected_df' in globals():
        dom_rows = detected_df.loc[detected_df['peak_id'] == p_id, 'dominant m/z']
        if len(dom_rows):
            dom_str = dom_rows.values[0]
            first_ion = str(dom_str).split(',')[0] if dom_str != "None" else "N/A"
            return f"Peak {p_id}: Unassigned (Base m/z: {first_ion})"
    return f"Peak {p_id}: Unassigned"


def build_interactive_chromatogram_html(div_id: str = "hplc_interactive_chrom_div") -> str:
    """
    Rebuilds the interactive Plotly chromatogram HTML with floating RDKit 2D structure hover cards
    from the live `detected_df` and `peak_assignments` dictionaries, and updates `_notebook_figure_registry`.
    """
    FIG_WIDTH_PX, FIG_HEIGHT_PX = 1220, 760
    MARGIN_L, MARGIN_R, MARGIN_T, MARGIN_B = 80, 40, 65, 210
    X_RANGE = [190, 310]

    time_arr = chrom.df[chrom.time_col].values
    raw_signal = chrom.df[chrom.int_col].values
    convolved = np.sum(chrom.unmixed_chromatograms, axis=1) if chrom.peaks is not None else None
    y_max_plot = float(np.max(raw_signal))
    prom_val = globals().get('current_prominence_val', 0.05)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    hover_cards_by_peak = {}
    qc_struct_images = {}

    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        norm_area = peak_row.get('normalised area', 'N/A')
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            assign = peak_assignments[p_id]
            mol_images = [Draw.MolToImage(m, size=(175, 145)) for m in assign["mols"]]
            if len(mol_images) > 1:
                total_w = sum(im.width for im in mol_images)
                struct_img = Image.new('RGBA', (total_w, mol_images[0].height), (255, 255, 255, 255))
                x_off = 0
                for im in mol_images:
                    struct_img.paste(im, (x_off, 0))
                    x_off += im.width
            else:
                struct_img = mol_images[0]

            qc_struct_images[p_id] = struct_img
            buf = io.BytesIO()
            struct_img.save(buf, format='PNG')
            b64_png = base64.b64encode(buf.getvalue()).decode('utf-8')

            adduct_lines = "<br>".join([
                f"<b>{a['adduct']}</b>: {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ])
            iso_badge = (
                f"<div style='color:#b45309; font-weight:600; font-size:11px; margin-top:2px;'>"
                f"⚠️ Isobaric Isomers ({assign['formula']})</div>"
            ) if assign.get("is_isobaric", False) else ""

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:10px; background:#ffffff; "
                f"padding:10px 12px; box-shadow:0 8px 20px rgba(0,0,0,0.18); font-family:sans-serif; "
                f"color:#111827; min-width:220px; max-width:390px; text-align:center;'>"
                f"<div style='font-weight:700; font-size:13px; border-bottom:1px solid #e5e7eb; "
                f"padding-bottom:4px; margin-bottom:4px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s): {assign['display_name']}</div>"
                f"{iso_badge}"
                f"<div style='font-size:11.5px; color:#374151; line-height:1.45; margin-bottom:6px;'>"
                f"{adduct_lines}<br>"
                f"<b>Area:</b> {norm_area} &nbsp;|&nbsp; "
                f"<b>Confidence:</b> <span style='color:#047857; font-weight:700;'>{assign['confidence']:.1f}%</span>"
                f"</div>"
                f"<img src='data:image/png;base64,{b64_png}' "
                f"style='max-width:100%; height:auto; display:block; margin:0 auto; border-radius:4px;'/>"
                f"</div>"
            )
        else:
            dom_str = str(peak_row.get('dominant m/z', 'None'))
            dom_tokens = [x.strip() for x in dom_str.split(',') if x.strip() and x.strip() != 'None']
            top_ions_str = "<br>".join(dom_tokens[:4]) if dom_tokens else "N/A"

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:8px; background:#f9fafb; "
                f"padding:8px 12px; box-shadow:0 6px 16px rgba(0,0,0,0.15); font-family:sans-serif; "
                f"color:#1f2937; min-width:190px; text-align:left;'>"
                f"<div style='font-weight:700; font-size:12.5px; border-bottom:1px solid #d1d5db; "
                f"padding-bottom:3px; margin-bottom:5px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s) — Unassigned</div>"
                f"<div style='font-size:11px; color:#4b5563; line-height:1.4;'>"
                f"<b>Norm. Area:</b> {norm_area}<br>"
                f"<b>Dominant m/z:</b><br>{top_ions_str}</div>"
                f"</div>"
            )

    chrom_fig = go.Figure()
    chrom_fig.add_trace(go.Scatter(
        x=time_arr, y=raw_signal, mode='lines', name='raw chromatogram',
        line=dict(color='black', width=1.6),
        hovertemplate='<b>Raw Chromatogram</b><br>RT=%{x:.2f}s<br>Signal=%{y:.3e}<extra></extra>'
    ))
    if convolved is not None:
        chrom_fig.add_trace(go.Scatter(
            x=time_arr, y=convolved, mode='lines', name='inferred mixture',
            line=dict(color='red', width=1.4, dash='dash'),
            hovertemplate='<b>Inferred Mixture</b><br>RT=%{x:.2f}s<br>Mixture=%{y:.3e}<extra></extra>'
        ))

    if chrom.peaks is not None:
        for g, _ in chrom.peaks.groupby('peak_id'):
            p_id = int(g)
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            component = chrom.unmixed_chromatograms[:, p_id - 1]
            chrom_fig.add_trace(go.Scatter(
                x=time_arr, y=component, mode='lines', fill='tozeroy',
                line=dict(color=peak_color, width=1.0), opacity=0.55,
                name=_peak_label_for_legend(p_id),
                meta=p_id,
                hoverinfo='none'
            ))

    shapes, compact_annotations = [], []
    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        c_idx = int(np.argmin(np.abs(time_arr - rt_apex)))
        apex_y = float(raw_signal[c_idx])
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        shapes.append(dict(
            type='line', x0=rt_apex, x1=rt_apex, y0=0, y1=apex_y,
            line=dict(color='dodgerblue', width=1.1, dash='dot'), opacity=0.55
        ))

        badge_text = (
            f"<b>P{p_id}: {peak_assignments[p_id]['display_name']}</b>"
            if ('peak_assignments' in globals() and p_id in peak_assignments)
            else f"<b>P{p_id}</b>"
        )
        compact_annotations.append(dict(
            x=rt_apex, y=apex_y + y_max_plot * 0.035, xref='x', yref='y',
            text=badge_text,
            showarrow=False,
            font=dict(size=10, color='#111827'),
            bgcolor='rgba(255,255,255,0.88)',
            bordercolor=peak_color,
            borderwidth=1.5,
            borderpad=2
        ))

        chrom_fig.add_trace(go.Scatter(
            x=[rt_apex], y=[apex_y],
            mode='markers',
            marker=dict(size=9, color=peak_color, line=dict(width=1.5, color='white')),
            showlegend=False,
            meta=p_id,
            hoverinfo='none'
        ))

    chrom_fig.update_layout(
        shapes=shapes,
        annotations=compact_annotations,
        title=dict(
            text=(
                f"HPLC Chromatogram Deconvolution & Peak Assignments (Prominence: {prom_val:.4f}) "
                f"— Hover over any peak to view 2D structure & MS1 details"
            ),
            font=dict(size=14.5, color='#111827')
        ),
        xaxis=dict(
            title='Retention Time (seconds)',
            range=X_RANGE,
            rangeslider=dict(visible=True, thickness=0.10)
        ),
        yaxis=dict(
            title='Total Ion Count (baseline corrected)',
            range=[0, y_max_plot * 1.18]
        ),
        legend=dict(
            orientation='h',
            x=0.5, y=-0.26,
            xanchor='center', yanchor='top',
            font=dict(size=10),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#d1d5db',
            borderwidth=1
        ),
        margin=dict(l=MARGIN_L, r=MARGIN_R, t=MARGIN_T, b=MARGIN_B),
        template='plotly_white',
        width=FIG_WIDTH_PX,
        height=FIG_HEIGHT_PX,
        hovermode='closest'
    )

    globals()['_last_chromatogram_figure'] = chrom_fig

    # Update static Kaleido PNG snapshot in `_notebook_figure_registry` for Step 4 Vision QC
    try:
        qc_fig = go.Figure(chrom_fig)
        qc_images = []
        qc_annotations = list(compact_annotations)
        banner_xs = np.linspace(200.0, 300.0, max(len(qc_struct_images), 1))
        for idx_p, (p_id, struct_img) in enumerate(sorted(qc_struct_images.items())):
            bx = float(banner_xs[idx_p])
            by = y_max_plot * 1.48
            assign = peak_assignments[p_id]
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            qc_annotations.append(dict(
                x=bx, y=by + y_max_plot * 0.04, xref='x', yref='y',
                text=f"<b>Peak {p_id}: {assign['display_name']}</b> ({assign['confidence']:.0f}%)",
                showarrow=False, font=dict(size=9.5, color='#111827'),
                bgcolor='white', bordercolor=peak_color, borderwidth=1.5
            ))
            qc_images.append(dict(
                source=struct_img, xref='x', yref='y',
                x=bx, y=by,
                sizex=18.0, sizey=y_max_plot * 0.32,
                xanchor='center', yanchor='top', layer='above'
            ))
        qc_fig.update_layout(
            yaxis=dict(title='Total Ion Count (baseline corrected)', range=[0, y_max_plot * 1.62]),
            annotations=qc_annotations,
            images=qc_images
        )
        globals().setdefault('_notebook_figure_registry', {})['deconvoluted_chromatogram_with_structures'] = qc_fig.to_image(format='png', scale=2)
    except Exception:
        pass

    fig_html = chrom_fig.to_html(include_plotlyjs='cdn', full_html=False, div_id=div_id)
    cards_json = json.dumps({str(k): v for k, v in hover_cards_by_peak.items()})

    return f"""
    <div style="position:relative; width:{FIG_WIDTH_PX}px; margin: 12px 0;">
        {fig_html}
        <div id="{div_id}_tooltip" style="display:none; position:fixed; pointer-events:none; z-index:99999; transition: opacity 0.08s ease-in-out;"></div>
    </div>
    <script>
    (function() {{
        var cards = {cards_json};
        var gd = document.getElementById('{div_id}');
        var tooltip = document.getElementById('{div_id}_tooltip');

        function attachHoverEvents() {{
            if (!gd || typeof gd.on !== 'function') {{
                setTimeout(attachHoverEvents, 150);
                return;
            }}
            gd.on('plotly_hover', function(data) {{
                if (!data || !data.points || data.points.length === 0) return;
                var pt = data.points[0];
                var p_id = pt.data.meta;
                if (p_id && cards[String(p_id)]) {{
                    tooltip.innerHTML = cards[String(p_id)];
                    tooltip.style.display = 'block';
                    if (data.event) {{
                        var x = data.event.clientX + 18;
                        var y = data.event.clientY - 30;
                        var rect = tooltip.getBoundingClientRect();
                        if (x + rect.width > window.innerWidth - 15) x = data.event.clientX - rect.width - 18;
                        if (y + rect.height > window.innerHeight - 15) y = window.innerHeight - rect.height - 15;
                        if (y < 10) y = 10;
                        tooltip.style.left = x + 'px';
                        tooltip.style.top = y + 'px';
                    }}
                }}
            }});
            gd.on('plotly_unhover', function() {{
                tooltip.style.display = 'none';
            }});
            gd.addEventListener('mousemove', function(e) {{
                if (tooltip.style.display === 'block') {{
                    var x = e.clientX + 18;
                    var y = e.clientY - 30;
                    var rect = tooltip.getBoundingClientRect();
                    if (x + rect.width > window.innerWidth - 15) x = e.clientX - rect.width - 18;
                    if (y + rect.height > window.innerHeight - 15) y = window.innerHeight - rect.height - 15;
                    if (y < 10) y = 10;
                    tooltip.style.left = x + 'px';
                    tooltip.style.top = y + 'px';
                }}
            }});
        }}
        attachHoverEvents();
    }})();
    </script>
    """


# --- RDKIT, PUBCHEM, CHEMBL, MASSBANK & REPLOTTING TOOLS FOR GEMINI ---

def get_user_candidate_molecules() -> str:
    """Returns the user's input table of candidate molecules, classifications, SMILES, formulas, and exact masses."""
    candidates = get_candidates_from_table() if 'get_candidates_from_table' in globals() else []
    if not candidates:
        return "No candidate molecules found."

    rows = []
    for name, classification, smi in candidates:
        mol, clean_smi, _ = safe_mol_from_smiles(smi)
        if mol:
            em = Descriptors.ExactMolWt(mol)
            formula = rdMolDescriptors.CalcMolFormula(mol)
            rows.append({
                "Name": name,
                "Classification": classification,
                "SMILES": clean_smi,
                "Formula": formula,
                "Exact_Mass": round(em, 4),
                "[M+H]+_mz": round(em + 1.007276, 4),
                "cLogP": round(Crippen.MolLogP(mol), 2),
                "TPSA": round(rdMolDescriptors.CalcTPSA(mol), 1)
            })
    return pd.DataFrame(rows).to_markdown(index=False)


def get_detected_peaks_summary() -> str:
    """Returns a full tabular summary of all deconvoluted HPLC-MS peaks, retention times, areas, dominant m/z ions, and assignments."""
    if 'detected_df' not in globals():
        return "Error: detected_df not found in memory. Run Cell 4 first."
    cols = [c for c in ['peak_id', 'retention_time', 'peak start time', 'peak end time', 'normalised area', 'dominant m/z', 'Assignment', 'Confidence (%)'] if c in detected_df.columns]
    return detected_df[cols].to_markdown(index=False)


def get_distilled_peak_details(peak_number: int) -> dict:
    """Retrieves background-subtracted ions, neutral losses, and contaminant flags for a 1-based intensity-ranked peak."""
    if not SPECTRAL_STATE or "peaks" not in SPECTRAL_STATE:
        prepare_spectral_state()
    peaks = SPECTRAL_STATE.get("peaks", [])
    if not 1 <= peak_number <= len(peaks):
        return {"error": f"Distilled peak {peak_number} not found (valid range: 1..{len(peaks)})."}
    peak = peaks[peak_number - 1]
    return {
        "peak_number": peak_number,
        "rt_min": peak["rt_min"],
        "rt_sec": round(peak["rt_min"] * 60.0, 1),
        "ions": peak["ions"],
        "artifacts": peak["artifacts"],
        "neutral_losses": peak["neutral_losses"],
        "candidates": peak["candidates"]
    }


def get_peak_details(peak_id: int) -> dict:
    """Retrieves detailed analytical evidence, observed adducts, and raw MS1 ions for a specific deconvoluted peak ID."""
    result = {}
    if 'detected_df' in globals():
        row = detected_df[detected_df['peak_id'] == peak_id]
        if not row.empty:
            r_dict = row.iloc[0].to_dict()
            result["chromatogram_row"] = {k: str(v) for k, v in r_dict.items()}
            rt_start = r_dict.get('peak start time', np.nan)
            rt_end = r_dict.get('peak end time', np.nan)
            if 'od' in globals() and not pd.isna(rt_start) and not pd.isna(rt_end):
                agg = {}
                for spec in od:
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
                        mzs, ints = spec.get_peaks()
                        for m, inten in zip(np.round(mzs, 4), ints):
                            if inten > 1e3:
                                agg[m] = agg.get(m, 0.0) + float(inten)
                if agg:
                    top_ions = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:25]
                    max_i = top_ions[0][1]
                    result["raw_ms1_top_25_ions_in_peak_window"] = [
                        {"mz": float(m), "intensity": round(i, 1), "rel_pct": round((i / max_i) * 100, 2)}
                        for m, i in top_ions
                    ]

    if 'peak_assignments' in globals() and peak_id in peak_assignments:
        assign = peak_assignments[peak_id]
        result["assigned_compound"] = {
            "display_name": assign["display_name"],
            "formula": assign["formula"],
            "confidence": assign["confidence"],
            "evidence": assign["evidence"],
            "adducts_observed": [
                f"{a['adduct']} at {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ]
        }
    return result if result else {"error": f"Peak {peak_id} not found."}


def check_mz_in_peak(peak_id: int, target_mz: float, tolerance_da: float = 0.05) -> dict:
    """Checks all raw MS1 scans across a specific peak's retention time window for a target m/z."""
    if 'detected_df' not in globals() or 'od' not in globals():
        return {"error": "detected_df or pyOpenMS 'od' experiment not found in memory."}

    row = detected_df[detected_df['peak_id'] == peak_id]
    if row.empty:
        return {"error": f"Peak {peak_id} not found in detected_df."}

    rt_start = float(row.iloc[0]['peak start time'])
    rt_end = float(row.iloc[0]['peak end time'])

    matches = []
    for spec in od:
        if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
            mzs, ints = spec.get_peaks()
            mask = (mzs >= target_mz - tolerance_da) & (mzs <= target_mz + tolerance_da)
            if np.any(mask):
                for m, inten in zip(mzs[mask], ints[mask]):
                    matches.append({
                        "rt": round(float(spec.getRT()), 2),
                        "observed_mz": round(float(m), 4),
                        "error_da": round(float(m - target_mz), 4),
                        "ppm_error": round(float((m - target_mz) / target_mz * 1e6), 1),
                        "intensity": round(float(inten), 1)
                    })

    matches = sorted(matches, key=lambda x: x["intensity"], reverse=True)
    return {
        "peak_id": peak_id,
        "rt_window": f"{rt_start:.1f}s - {rt_end:.1f}s",
        "target_mz_searched": target_mz,
        "found_in_raw_scans": len(matches) > 0,
        "max_observed_hit": matches[0] if matches else None
    }


def generate_in_silico_fragments(parent_smiles: str, check_peak_id: int = 0) -> dict:
    """
    Generates RDKit-sanitized ESI+ fragments for a parent molecule, automatically pre-renders
    the 2D SVG structure image for the parent AND every fragment (returning an `image_token`
    like `[[MOL_IMG_1]]` for each), and checks if each fragment m/z is present in `check_peak_id`.
    """
    mol, clean_parent, err = safe_mol_from_smiles(parent_smiles)
    if not mol:
        return {"error": err}

    parent_em = Descriptors.ExactMolWt(mol)
    parent_formula = rdMolDescriptors.CalcMolFormula(mol)
    parent_mh = round(parent_em + 1.007276, 4)
    parent_img_token = _register_mol_svg(mol, label="Parent Molecule")

    fragments_dict = {}

    def _add_frag(frag_smi, pathway_label, is_intrinsic_cation=False):
        f_mol, f_clean, _ = safe_mol_from_smiles(frag_smi)
        if not f_mol:
            return
        em = Descriptors.ExactMolWt(f_mol)
        if em < 40.0 or em >= parent_em - 1.0:
            return
        form = rdMolDescriptors.CalcMolFormula(f_mol)
        f_charge = Chem.GetFormalCharge(f_mol)
        if is_intrinsic_cation or f_charge > 0:
            mz_val = round(em - 0.00054858, 4)
            ion_type = "[F]+ (intrinsic cation)"
        else:
            mz_val = round(em + 1.007276, 4)
            ion_type = "[F+H]+"

        key = (form, round(mz_val, 3))
        if key not in fragments_dict:
            entry = {
                "fragment_name": pathway_label,
                "smiles": f_clean,
                "formula": form,
                "neutral_exact_mass": round(em, 4),
                "ion_type": ion_type,
                "expected_esi_pos_mz": mz_val,
                "neutral_loss_from_parent_Da": round(parent_mh - mz_val, 4),
                "_temp_mol": f_mol
            }
            if check_peak_id > 0:
                hit_info = check_mz_in_peak(check_peak_id, mz_val, tolerance_da=0.05)
                entry["present_in_peak"] = hit_info.get("found_in_raw_scans", False)
                entry["peak_observation"] = hit_info.get("max_observed_hit", None)
            fragments_dict[key] = entry

    ribose_smarts = Chem.MolFromSmarts("[C@@H,C@H,CH]1O[C@H,C@@H,CH](CO)[C@@H,C@H,CH](O)[C@H,C@@H,CH]1")
    if mol.HasSubstructMatch(ribose_smarts):
        _add_frag("OC[C@H]1O[C+]=[C@H][C@@H]1O", "Ribose oxocarbenium fragment [C5H7O3]+", is_intrinsic_cation=True)

    cleavable_bonds = [b.GetIdx() for b in mol.GetBonds() if not b.IsInRing() and b.GetBondType() == Chem.BondType.SINGLE]
    for b_idx in cleavable_bonds:
        try:
            frag_mol = Chem.FragmentOnBonds(mol, [b_idx], addDummies=False)
            for f in Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False):
                _add_frag(Chem.MolToSmiles(f), "Single-bond cleavage fragment ([F+H]+)")
        except Exception:
            pass

    if "ncnc2c1ncn2" in clean_parent or "ncnc2c1nc[nH]2" in clean_parent:
        _add_frag("Nc1ncnc2c1nc[nH]2", "Protonated Adenine [B+H]+ (glycosidic cleavage, -Ribose)")
        _add_frag("c1ncnc2c1nc[nH]2", "Purine core after NH3 loss ([B+H-NH3]+)")
        _add_frag("Nc1ncnc2c1ncn2C=C", "0,2X Cross-ring cleavage (Adenine + vinyl remnant)")
        _add_frag("Nc1ncnc2c1ncn2CC=O", "0,2X Cross-ring ribose cleavage (Adenine + C2H3O)")
        _add_frag("Nc1ncnc2c1ncn2[C@@H](O)C=O", "0,3X Cross-ring ribose cleavage (Adenine + C2H3O2)")

    sorted_frags = sorted(fragments_dict.values(), key=lambda x: x["expected_esi_pos_mz"], reverse=True)
    for f_entry in sorted_frags:
        f_mol = f_entry.pop("_temp_mol")
        f_entry["image_token"] = _register_mol_svg(f_mol, label=f_entry["fragment_name"])

    return {
        "parent_smiles": clean_parent,
        "parent_formula": parent_formula,
        "parent_exact_mass": round(parent_em, 4),
        "parent_[M+H]+_mz": parent_mh,
        "parent_image_token": parent_img_token,
        "generated_fragments": sorted_frags,
        "rendering_instruction": (
            "Each molecule above already has its 2D image pre-rendered! "
            "Simply include the exact `parent_image_token` or `image_token` (e.g. [[MOL_IMG_1]]) "
            "on its own line in your response where you want that structure displayed."
        )
    }


def analyze_structure_and_mass(smiles_or_formula: str, target_mz: float = 0.0, label: str = "") -> dict:
    """
    Calculates exact monoisotopic mass, formula, and ESI m/z values ([M]+, [M+H]+, [M+Na]+),
    and automatically pre-renders the 2D SVG structure image (returning `image_token`).
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles_or_formula)
    if mol is None:
        try:
            em = rdMolDescriptors.CalcExactMolWt(smiles_or_formula.strip())
            return {
                "input_interpreted_as": "Molecular Formula (no 2D structure)",
                "formula": smiles_or_formula.strip(),
                "exact_monoisotopic_mass": round(em, 5),
                "protonated_[M+H]+": round(em + 1.007276, 4)
            }
        except Exception:
            return {"error": err}

    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    formal_charge = Chem.GetFormalCharge(mol)
    mz_mh_plus = exact_mass + 1.007276 if formal_charge == 0 else exact_mass - (formal_charge * 0.00054858)

    img_token = _register_mol_svg(mol, label=label or formula)

    res = {
        "canonical_smiles": clean_smi,
        "formula": formula,
        "exact_monoisotopic_mass": round(exact_mass, 5),
        "protonated_[M+H]+": round(mz_mh_plus, 4),
        "rdkit_cLogP": round(Crippen.MolLogP(mol), 3),
        "TPSA_A2": round(rdMolDescriptors.CalcTPSA(mol), 2),
        "image_token": img_token,
        "instruction": f"Place {img_token} on its own line in your response to display the 2D structure."
    }
    if target_mz > 0:
        diff_da = mz_mh_plus - target_mz
        res["target_mz_comparison"] = {
            "target_mz": target_mz,
            "error_Da": round(diff_da, 4),
            "ppm_error": round((diff_da / target_mz) * 1e6, 1)
        }
    return res


def generate_2d_structure_svg(smiles: str, label: str = "") -> dict:
    """Explicit tool to generate a 2D chemical structure SVG and return its `image_token` (e.g. [[MOL_IMG_1]])."""
    mol, clean_smi, err = safe_mol_from_smiles(smiles)
    if not mol:
        return {"error": err}
    token = _register_mol_svg(mol, label=label)
    return {
        "status": "success",
        "canonical_smiles": clean_smi,
        "image_token": token,
        "instruction": f"Insert the exact token {token} on its own line in your response to display this structure."
    }


def search_pubchem_by_mass_or_formula(
    observed_mh_mz: float = 0.0,
    molecular_formula: str = "",
    compound_name: str = "",
    mass_window_da: float = 0.015,
    max_results: int = 8
) -> dict:
    """
    Searches PubChem for known metabolites or compounds by compound_name, molecular_formula,
    OR observed [M+H]+ m/z (automatically subtracting H+ mass 1.007276 Da via PubChem PUG REST),
    and pre-renders 2D structure SVGs (`image_token`) for each hit.
    """
    try:
        max_results = max(1, min(int(max_results), 8))
        compounds = []
        if compound_name.strip():
            compounds = pcp.get_compounds(compound_name.strip(), 'name')[:max_results]
        elif molecular_formula.strip():
            compounds = pcp.get_compounds(molecular_formula.strip(), 'formula')[:max_results]
        elif observed_mh_mz > 0:
            neutral_mass = observed_mh_mz - 1.007276
            low_m = round(neutral_mass - mass_window_da, 4)
            high_m = round(neutral_mass + mass_window_da, 4)
            pug_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/exact_mass/range/{low_m}/{high_m}/cids/JSON"
            req = urllib.request.Request(pug_url, headers={"User-Agent": "Colab-LCMS-Agent/1.0"})
            with urllib.request.urlopen(req, timeout=10) as resp:
                cid_data = json.loads(resp.read().decode('utf-8'))
            cids = cid_data.get("IdentifierList", {}).get("CID", [])[:max_results]
            if cids:
                compounds = pcp.get_compounds(cids, 'cid')

        hits = []
        for c in compounds[:max_results]:
            smi = getattr(c, 'isomeric_smiles', None) or getattr(c, 'canonical_smiles', None)
            if not smi:
                continue
            mol, clean_smi, _ = safe_mol_from_smiles(smi)
            if not mol:
                continue
            em = Descriptors.ExactMolWt(mol)
            mh_calc = em + 1.007276
            ppm_err = ((observed_mh_mz - mh_calc) / mh_calc * 1e6) if observed_mh_mz > 0 else 0.0
            synonyms = (getattr(c, 'synonyms', None) or [])[:5]
            pref_label = synonyms[0] if synonyms else (c.iupac_name or f"CID {c.cid}")
            img_token = _register_mol_svg(mol, label=f"{pref_label} (CID {c.cid})")

            hits.append({
                "cid": c.cid,
                "preferred_name": pref_label,
                "synonyms": synonyms,
                "iupac_name": c.iupac_name,
                "formula": c.molecular_formula,
                "smiles": clean_smi,
                "exact_mass": round(em, 4),
                "calculated_[M+H]+_mz": round(mh_calc, 4),
                "ppm_error_vs_observed": round(ppm_err, 2),
                "rdkit_cLogP": round(Crippen.MolLogP(mol), 3),
                "TPSA_A2": round(rdMolDescriptors.CalcTPSA(mol), 2),
                "image_token": img_token
            })
        return {
            "query_mh_mz": observed_mh_mz,
            "query_formula": molecular_formula,
            "query_name": compound_name,
            "pubchem_hits": hits
        }
    except Exception as e:
        return {"error": f"PubChem search failed: {e}"}


def search_chembl_metabolites_and_isomers(
    molecular_formula: str = "",
    smiles: str = "",
    max_results: int = 8
) -> dict:
    """
    Queries the EBI ChEMBL REST API by molecular formula or canonical SMILES
    to retrieve known bioactive metabolites, drug standards, ALogP, polar surface area (PSA),
    RDKit Crippen cLogP for reverse-phase retention time ranking, and 2D structure `image_token`s.
    """
    try:
        max_results = max(1, min(int(max_results), 8))
        hits = []
        if smiles.strip():
            mol, clean_smi, err = safe_mol_from_smiles(smiles)
            if not mol:
                return {"error": err}
            inchikey = Chem.MolToInchiKey(mol)
            url = f"https://www.ebi.ac.uk/chembl/api/data/molecule.json?molecule_structures__standard_inchi_key={inchikey}"
        elif molecular_formula.strip():
            url = f"https://www.ebi.ac.uk/chembl/api/data/molecule.json?molecule_properties__full_molformula={urllib.parse.quote(molecular_formula.strip())}&limit={max_results}"
        else:
            return {"error": "Provide either `molecular_formula` (e.g. 'C10H15N3O5') or `smiles`."}

        req = urllib.request.Request(url, headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/1.0"})
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = json.loads(resp.read().decode('utf-8'))

        for mol_entry in data.get("molecules", [])[:max_results]:
            struct = mol_entry.get("molecule_structures") or {}
            props = mol_entry.get("molecule_properties") or {}
            can_smi = struct.get("canonical_smiles")
            if not can_smi:
                continue

            r_mol, clean_s, _ = safe_mol_from_smiles(can_smi)
            if not r_mol:
                continue
            rdkit_clogp = round(Crippen.MolLogP(r_mol), 3)
            em = round(Descriptors.ExactMolWt(r_mol), 4)
            pref_name = mol_entry.get("pref_name") or mol_entry.get("molecule_chembl_id")
            img_token = _register_mol_svg(r_mol, label=f"{pref_name} ({mol_entry.get('molecule_chembl_id')})")

            hits.append({
                "chembl_id": mol_entry.get("molecule_chembl_id"),
                "pref_name": pref_name,
                "max_clinical_phase": mol_entry.get("max_phase"),
                "formula": props.get("full_molformula"),
                "exact_mass": em,
                "expected_[M+H]+_mz": round(em + 1.007276, 4),
                "chembl_alogp": props.get("alogp"),
                "rdkit_clogp": rdkit_clogp,
                "psa": props.get("psa"),
                "canonical_smiles": clean_s,
                "image_token": img_token
            })

        return {
            "query_formula": molecular_formula,
            "query_smiles": smiles,
            "chembl_candidates": hits,
            "retention_note": "In Reverse-Phase (C18) HPLC, more polar compounds (lower cLogP / higher PSA) elute earlier."
        }
    except Exception as e:
        return {"error": f"ChEMBL query failed: {e}"}


def query_massbank_reference_spectrum(
    compound_name_or_smiles: str = "",
    exact_mass: float = 0.0,
    max_records: int = 5
) -> dict:
    """
    Queries the MassBank Europe REST API (api.massbank.eu) by InChIKey (from SMILES),
    compound name, or exact neutral mass to retrieve experimental ESI+ MS/MS fragment peaks.
    """
    try:
        max_records = max(1, min(int(max_records), 8))
        params = {"ion_mode": "POSITIVE", "ms_type": "MS2"}
        mol, clean_smi, _ = safe_mol_from_smiles(compound_name_or_smiles)
        if mol:
            params["inchi_key"] = Chem.MolToInchiKey(mol)
        elif compound_name_or_smiles.strip():
            params["compound_name"] = compound_name_or_smiles.strip()
        elif exact_mass > 0:
            params["exact_mass"] = f"{exact_mass:.4f}"
            params["mass_tolerance"] = "0.02"
        else:
            return {"error": "Provide a SMILES string, compound name, or exact_mass."}

        query_str = urllib.parse.urlencode(params)
        search_url = f"https://api.massbank.eu/records/search?{query_str}"
        req = urllib.request.Request(search_url, headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/1.0"})

        records_summary = []
        with urllib.request.urlopen(req, timeout=10) as resp:
            search_res = json.loads(resp.read().decode('utf-8'))

        items = search_res if isinstance(search_res, list) else search_res.get("results", search_res.get("data", []))
        for item in items[:max_records]:
            accession = item if isinstance(item, str) else item.get("accession")
            if not accession:
                continue
            rec_url = f"https://api.massbank.eu/records/{urllib.parse.quote(str(accession), safe='')}"
            req_rec = urllib.request.Request(rec_url, headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/1.0"})
            with urllib.request.urlopen(req_rec, timeout=8) as r_resp:
                rec_data = json.loads(r_resp.read().decode('utf-8'))

            peaks_raw = rec_data.get("peak", {}).get("peak", {}).get("values", [])
            sorted_peaks = sorted(peaks_raw, key=lambda x: x.get("rel", x.get("intensity", 0)), reverse=True)[:10]
            records_summary.append({
                "accession": accession,
                "title": rec_data.get("record_title"),
                "precursor_mz": rec_data.get("ms_focused_ion", {}).get("precursor_m/z"),
                "top_ms2_fragments": [
                    {"mz": round(float(p.get("mz", 0)), 4), "rel_intensity": p.get("rel", p.get("intensity", 0))}
                    for p in sorted_peaks
                ]
            })

        return {
            "query": compound_name_or_smiles or exact_mass,
            "massbank_records_found": len(records_summary),
            "reference_spectra": records_summary
        }
    except Exception as e:
        return {
            "warning": f"Live MassBank REST lookup returned: {e}. Use `disambiguate_isobaric_peak_isomers` for deterministic in-source MS1 fragment matching."
        }


def search_chemical_databases(
    molecular_formula: str = "",
    smiles: str = "",
    observed_mh_mz: float = 0.0
) -> dict:
    """Runs PubChem, ChEMBL, and MassBank searches and registers 2D structure SVG tokens."""
    pubchem_res = search_pubchem_by_mass_or_formula(
        observed_mh_mz=observed_mh_mz,
        molecular_formula=molecular_formula
    )
    chembl_res = (
        search_chembl_metabolites_and_isomers(molecular_formula=molecular_formula, smiles=smiles)
        if (molecular_formula.strip() or smiles.strip())
        else {"note": "Provide molecular_formula or smiles to query ChEMBL."}
    )
    neutral_mass = (observed_mh_mz - 1.007276) if observed_mh_mz > 1.0 else 0.0
    massbank_res = (
        query_massbank_reference_spectrum(compound_name_or_smiles=smiles, exact_mass=neutral_mass)
        if (smiles.strip() or neutral_mass > 0)
        else {"note": "Provide smiles or observed_mh_mz to query MassBank."}
    )
    return {
        "pubchem": pubchem_res,
        "chembl": chembl_res,
        "massbank": massbank_res
    }


def disambiguate_isobaric_peak_isomers(
    peak_id: int,
    candidate_names: list[str],
    candidate_smiles: list[str]
) -> dict:
    """
    Resolves isobaric isomers (e.g., 2'-O-methylcytidine vs 5-methylcytidine) for a given `peak_id`
    by checking raw MS1 scans for isomer-specific glycosidic cleavage ions ([BH2]+ losing
    Ribose -132.0423 Da vs 2'-O-Methylribose -146.0579 Da vs Deoxyribose -116.0473 Da)
    and comparing RDKit Crippen cLogP & TPSA to predict reverse-phase C18 elution order.
    """
    if len(candidate_names) != len(candidate_smiles):
        return {"error": "candidate_names and candidate_smiles must have the same length."}

    isomer_reports = []
    for name, smi in zip(candidate_names, candidate_smiles):
        mol, clean_smi, err = safe_mol_from_smiles(smi)
        if not mol:
            isomer_reports.append({"name": name, "error": err})
            continue

        em = Descriptors.ExactMolWt(mol)
        mh_mz = round(em + 1.007276, 4)
        clogp = round(Crippen.MolLogP(mol), 3)
        tpsa = round(rdMolDescriptors.CalcTPSA(mol), 2)
        img_token = _register_mol_svg(mol, label=f"Candidate: {name}")

        frag_res = generate_in_silico_fragments(clean_smi, check_peak_id=peak_id)
        observed_diagnostic_frags = [
            {
                "fragment_name": f["fragment_name"],
                "expected_mz": f["expected_esi_pos_mz"],
                "neutral_loss_Da": f["neutral_loss_from_parent_Da"],
                "observed_in_peak": f["peak_observation"],
                "fragment_image_token": f["image_token"]
            }
            for f in frag_res.get("generated_fragments", [])
            if f.get("present_in_peak")
        ]

        sugar_checks = []
        for loss_label, loss_da in [
            ("Unmodified Ribose loss (-132.0423 Da -> modified nucleobase [BH2]+)", 132.042259),
            ("2'-O-Methylribose loss (-146.0579 Da -> unmodified nucleobase [BH2]+)", 146.057909),
            ("2'-Deoxyribose loss (-116.0473 Da -> nucleobase [BH2]+)", 116.047344),
        ]:
            expected_bh2_mz = round(mh_mz - loss_da, 4)
            if expected_bh2_mz > 70.0:
                scan_hit = check_mz_in_peak(peak_id, expected_bh2_mz, tolerance_da=0.025)
                sugar_checks.append({
                    "pathway": loss_label,
                    "target_bh2_mz": expected_bh2_mz,
                    "detected_in_raw_ms1": scan_hit.get("found_in_raw_scans", False),
                    "max_hit": scan_hit.get("max_observed_hit")
                })

        isomer_reports.append({
            "candidate_name": name,
            "canonical_smiles": clean_smi,
            "formula": rdMolDescriptors.CalcMolFormula(mol),
            "parent_[M+H]+_mz": mh_mz,
            "rdkit_cLogP": clogp,
            "TPSA_A2": tpsa,
            "structure_image_token": img_token,
            "observed_in_silico_fragments": observed_diagnostic_frags,
            "diagnostic_glycosidic_cleavage_checks": sugar_checks
        })

    return {
        "peak_id": peak_id,
        "isobaric_comparison": isomer_reports,
        "interpretation_guide": (
            "1. Check `diagnostic_glycosidic_cleavage_checks` and `observed_in_silico_fragments` for direct in-source fragment hits. "
            "2. For reverse-phase C18 elution order, lower `rdkit_cLogP` / higher `TPSA_A2` (more polar, e.g. 5-methylcytidine with 3 free ribose OHs) "
            "elutes EARLIER than higher `rdkit_cLogP` / lower `TPSA_A2` (e.g. 2'-O-methylcytidine with a methylated 2'-OH)."
        )
    }


def assign_peaks_and_replot_chromatogram(
    peak_ids: list[int],
    compound_names: list[str],
    smiles_list: list[str],
    classifications: list[str] = None
) -> dict:
    """
    Assigns one or more peaks (including previously unassigned peaks or resolved isobaric peaks)
    to specific chemical structures, calculates mass accuracy & confidence scores against raw MS1
    scans in `od`, updates `detected_df` and `peak_assignments` in place, and generates a brand-new
    interactive Plotly chromatogram with floating RDKit 2D hover cards.
    Returns a `plot_token` (e.g. `[[PLOTLY_CHROM_1]]`) that you MUST place on its own line in your response.
    """
    global detected_df, peak_assignments

    if 'detected_df' not in globals() or 'chrom' not in globals() or chrom is None:
        return {"error": "detected_df or chrom not initialized. Run Cell 4 first."}

    if not (len(peak_ids) == len(compound_names) == len(smiles_list)):
        return {"error": "peak_ids, compound_names, and smiles_list must have the exact same length."}

    if not classifications or len(classifications) != len(peak_ids):
        classifications = ["Metabolite"] * len(peak_ids)

    adduct_defs = globals().get('esi_pos_adducts', [
        ("[M+H]+",   1, 1.007276,  1),
        ("[M+NH4]+", 1, 18.033823, 1),
        ("[M+Na]+",  1, 22.989218, 1),
        ("[M+K]+",   1, 38.963158, 1),
        ("[2M+H]+",  2, 1.007276,  1),
    ])

    updated_summary = []
    for p_id, name, smi, cls_label in zip(peak_ids, compound_names, smiles_list, classifications):
        p_id = int(p_id)
        mol, clean_smi, err = safe_mol_from_smiles(smi)
        if not mol:
            updated_summary.append({"peak_id": p_id, "status": "failed", "error": err})
            continue

        em = round(Descriptors.ExactMolWt(mol), 4)
        formula = rdMolDescriptors.CalcMolFormula(mol)

        observed_adducts = []
        for adduct_name, mult, shift, z in adduct_defs:
            calc_mz = (em * mult + shift) / z
            hit = check_mz_in_peak(p_id, calc_mz, tolerance_da=0.03)
            if hit.get("found_in_raw_scans") and hit.get("max_observed_hit"):
                best_hit = hit["max_observed_hit"]
                observed_adducts.append({
                    "adduct": adduct_name,
                    "obs_mz": best_hit["observed_mz"],
                    "calc_mz": calc_mz,
                    "ppm": best_hit["ppm_error"],
                    "intensity": best_hit["intensity"]
                })

        if not observed_adducts:
            calc_mh = em + 1.007276
            observed_adducts.append({
                "adduct": "[M+H]+",
                "obs_mz": round(calc_mh, 4),
                "calc_mz": calc_mh,
                "ppm": 0.0,
                "intensity": 1e6
            })
        else:
            observed_adducts = sorted(observed_adducts, key=lambda x: x["intensity"], reverse=True)

        row_mask = detected_df['peak_id'] == p_id
        dom_str = detected_df.loc[row_mask, 'dominant m/z'].values[0] if row_mask.any() else "None"
        dom_parsed = parse_dominant_mz_string(dom_str) if 'parse_dominant_mz_string' in globals() else []

        best_group = {
            "exact_mass": em,
            "formula": formula,
            "mols": [mol],
            "adducts": observed_adducts
        }
        if 'compute_confidence_from_dominant_mz' in globals():
            conf_score, evidence_str = compute_confidence_from_dominant_mz(
                best_group, dom_parsed, globals().get('peak_ms1_spectra', {}).get(p_id, {})
            )
        else:
            conf_score, evidence_str = 88.0, f"Assigned ({observed_adducts[0]['ppm']:+.1f} ppm)"

        peak_assignments[p_id] = {
            "display_name": name,
            "full_assignment": f"{name} ({cls_label})",
            "is_isobaric": False,
            "formula": formula,
            "mols": [mol],
            "adducts": observed_adducts,
            "confidence": conf_score,
            "evidence": evidence_str
        }

        top_a = observed_adducts[0]
        if row_mask.any():
            detected_df.loc[row_mask, 'Assignment'] = (
                f"{name} ({cls_label}) [{top_a['adduct']}: {top_a['obs_mz']:.4f} m/z ({top_a['ppm']:+.1f} ppm)]"
            )
            detected_df.loc[row_mask, 'Confidence (%)'] = f"{conf_score:.1f}% ({evidence_str})"

        struct_token = _register_mol_svg(mol, label=f"Peak {p_id}: {name}")
        updated_summary.append({
            "peak_id": p_id,
            "assigned_name": name,
            "classification": cls_label,
            "formula": formula,
            "primary_adduct": top_a["adduct"],
            "observed_mz": top_a["obs_mz"],
            "ppm_error": top_a["ppm"],
            "confidence_pct": conf_score,
            "evidence": evidence_str,
            "structure_image_token": struct_token
        })

    unique_div_id = f"hplc_chrom_replot_{uuid.uuid4().hex[:6]}"
    interactive_html = build_interactive_chromatogram_html(div_id=unique_div_id)

    plot_idx = sum(1 for k in _rendered_svg_registry if k.startswith("[[PLOTLY_CHROM_")) + 1
    plot_token = f"[[PLOTLY_CHROM_{plot_idx}]]"
    _rendered_svg_registry[plot_token] = interactive_html

    return {
        "status": "success",
        "updated_peaks": updated_summary,
        "plot_token": plot_token,
        "rendering_instruction": (
            f"CRITICAL: Place the exact token {plot_token} on its own line in your response "
            "to render the updated interactive Plotly HPLC chromatogram with floating 2D hover cards!"
        )
    }


# --- BIND TOOLS & STATEFUL CHAT UI ---

client = genai.Client(api_key=userdata.get("KDD_GEMINI_API_KEY"))

rdkit_tools = [
    get_user_candidate_molecules,
    get_detected_peaks_summary,
    get_distilled_peak_details,
    get_peak_details,
    check_mz_in_peak,
    generate_in_silico_fragments,
    analyze_structure_and_mass,
    generate_2d_structure_svg,
    search_pubchem_by_mass_or_formula,
    search_chembl_metabolites_and_isomers,
    query_massbank_reference_spectrum,
    search_chemical_databases,
    update_lcms_plot,
    disambiguate_isobaric_peak_isomers,
    assign_peaks_and_replot_chromatogram,
]

def build_system_instruction() -> str:
    candidates_md = get_user_candidate_molecules()
    peaks_md = get_detected_peaks_summary()
    hypotheses = SPECTRAL_STATE.get("hypotheses", [])[:10]
    hyp_str = f"\n**3. Reaction Mass-Shift Hypotheses:**\n{json.dumps(hypotheses)}\n" if hypotheses else ""
    return f"""You are an expert analytical chemist and mass spectrometrist assistant integrated into an HPLC-ESI-MS / GC-MS notebook.

### Live Experimental Context:
**1. User's Candidate Molecules Table:**
{candidates_md}

**2. Deconvoluted HPLC-MS Peaks Summary (`detected_df`):**
{peaks_md}
{hyp_str}
### Mandatory Operating Rules:
1. **Use `generate_in_silico_fragments` First for Fragment Questions:**
   - Whenever the user asks for potential ESI fragments of a molecule (e.g., adenosine) and/or asks if they appear in a specific peak (e.g., Peak 2), call `generate_in_silico_fragments(parent_smiles, check_peak_id=2)`.
   - `generate_in_silico_fragments`, `analyze_structure_and_mass`, `search_pubchem_by_mass_or_formula`, and `search_chembl_metabolites_and_isomers` AUTOMATICALLY render the 2D chemical structure SVG for every molecule and return an `image_token` (such as `[[MOL_IMG_1]]`, `[[MOL_IMG_2]]`, etc.).
2. **How to Display 2D Structure Images:**
   - Copy the exact `parent_image_token`, `structure_image_token`, or `image_token` (e.g., `[[MOL_IMG_1]]`) returned by a tool and place it on its own line right below each molecule or fragment description.
   - Never invent tokens that were not returned by a tool call.
3. **Identifying Unassigned Peaks & Disambiguating Isobaric Isomers:**
   - To identify unassigned peaks, first call `get_peak_details(peak_id)` to inspect the raw MS1 top ions, potential `[M+H]+` / `[M+Na]+` / `[M+NH4]+` adducts, and in-source fragments (such as `-132.0423 Da` ribose loss or `-116.0473 Da` deoxyribose loss).
   - Call `search_pubchem_by_mass_or_formula` and/or `search_chembl_metabolites_and_isomers` (or `search_chemical_databases`) to find matching known metabolites and compare their `rdkit_cLogP` and `PSA` against reverse-phase retention times.
   - To resolve `[Isobaric Isomers]` (e.g. `n2-o-methylcytidine` vs `n5-methylcytidine`), call `disambiguate_isobaric_peak_isomers(peak_id, candidate_names, candidate_smiles)` and/or `query_massbank_reference_spectrum` to evaluate diagnostic glycosidic cleavage ions (`[BH2]+`) and C18 polarity elution order.
4. **Replotting the Interactive Plotly Chromatogram & LCMS Dashboard:**
   - Whenever the user asks to assign new peaks, update peak assignments, or replot the deconvoluted chromatogram, call `assign_peaks_and_replot_chromatogram(peak_ids, compound_names, smiles_list, classifications)`.
   - You MUST copy the returned `plot_token` (e.g. `[[PLOTLY_CHROM_1]]`) and place it on its own line in your markdown response so the live interactive Plotly chromatogram renders in the notebook output!
   - If the user asks for an Extracted Ion Chromatogram (XIC), UV/DAD trace, or theoretical isotope envelope overlay, call `update_lcms_plot`.
"""

curated_stable_models = [
    'gemini-3.8-flash',
    'gemini-3.6-flash',
    'gemini-3.5-flash',
    'gemini-3.5-flash-lite',
    'gemini-3.1-flash-lite',
    'gemini-2.5-pro',
    'gemini-2.5-flash',
    'gemini-2.5-flash-lite',
]

try:
    api_models = [
        m.name.replace('models/', '')
        for m in client.models.list()
        if 'generateContent' in (getattr(m, 'supported_actions', None) or [])
        and m.name.startswith('models/gemini-')
        and not any(x in m.name for x in ['preview', 'exp', 'image', 'tts', 'live', 'transcribe', 'embedding', '1.5', '2.0'])
    ]
    for m_id in sorted(api_models, reverse=True):
        if m_id not in curated_stable_models:
            curated_stable_models.append(m_id)
except Exception:
    pass

model_dropdown = widgets.Dropdown(
    options=curated_stable_models,
    value='gemini-3.8-flash',
    description='Model:',
    disabled=False,
    layout=widgets.Layout(width='320px')
)

reset_button = widgets.Button(
    description='Reset Chat / Refresh Data',
    button_style='warning',
    icon='refresh',
    layout=widgets.Layout(width='200px')
)

chat_input = widgets.Textarea(
    value='',
    placeholder='Ask about your HPLC-MS data (e.g., "Identify the unassigned peaks using PubChem and ChEMBL, resolve any isobaric isomers, and replot the interactive HPLC chromatogram with the new peaks assigned")...',
    description='Input:',
    layout=widgets.Layout(width='85%', height='90px')
)

submit_button = widgets.Button(
    description='Send',
    button_style='primary',
    icon='paper-plane'
)

output_area = widgets.Output(
    layout=widgets.Layout(border='1px solid #e5e7eb', padding='12px', height='750px', overflow_y='auto')
)
output_area.add_class("lcms-chat-output")
chat_style = widgets.HTML(
    "<style>.lcms-chat-output{line-height:1.6;font-size:15px;padding:14px;}"
    ".lcms-chat-output table{border-collapse:collapse;display:block;overflow-x:auto;}"
    ".lcms-chat-output td,.lcms-chat-output th{border:1px solid #cbd5e1;padding:6px 10px;}"
    ".lcms-chat-output pre{background:#f1f5f9;padding:12px;overflow-x:auto;}"
    ".lcms-chat-output code{font-family:monospace;}</style>"
)
reaction_input = widgets.Text(
    placeholder="SM1.SM2>>Product, SM: SMILES, Reagent: SMILES, Product: SMILES, or a single SMILES",
    description="Reaction:", layout=widgets.Layout(width='95%')
)
uv_ms_delay_input = widgets.FloatText(value=0.08, description="UV→MS delay (min):",
                                      style={"description_width": "initial"})
uv_upload = widgets.FileUpload(accept=".csv", multiple=False, description="Upload UV/DAD CSV")
plot_button = widgets.Button(description="Show LCMS dashboard", icon="chart-line")
plot_options = widgets.Text(placeholder="XIC m/z (comma-separated); blank for TIC/BPC",
                            description="XIC:", layout=widgets.Layout(width="65%"))
isotope_input = widgets.Text(placeholder="Formula or SMILES for isotope overlay (optional)",
                             description="Isotopes:", layout=widgets.Layout(width="65%"))
scan_input = widgets.FloatText(value=0, description="MS scan RT (min, 0=auto):",
                               style={"description_width": "initial"})

chat_session = None
active_model_name = None
chat_history_ui = []

def init_chat_session(force_reset=False):
    global chat_session, active_model_name, chat_history_ui
    current_data_id = id(globals().get("od", globals().get("hplc_df")))
    if (force_reset or not SPECTRAL_STATE or
        SPECTRAL_STATE.get("data_id") != current_data_id or
        SPECTRAL_STATE.get("reaction_text") != reaction_input.value):
        try:
            prepare_spectral_state(uv_ms_delay_min=uv_ms_delay_input.value)
        except Exception:
            pass
    else:
        SPECTRAL_STATE["uv_ms_delay_min"] = uv_ms_delay_input.value

    if force_reset or chat_session is None or active_model_name != model_dropdown.value:
        active_model_name = model_dropdown.value
        chat_session = client.chats.create(
            model=active_model_name,
            config=types.GenerateContentConfig(
                system_instruction=build_system_instruction(),
                tools=rdkit_tools,
                temperature=0.2,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=False,
                    maximum_remote_calls=30
                )
            )
        )
        if force_reset:
            chat_history_ui.clear()
            _rendered_svg_registry.clear()
            with output_area:
                clear_output()
                print("🔄 Chat session reset and synced with latest HPLC-MS DataFrame.")

def extract_response_text(response) -> str:
    text_chunks = []
    if response.candidates:
        for part in response.candidates[0].content.parts:
            if getattr(part, 'text', None):
                text_chunks.append(part.text)
    return "".join(text_chunks) if text_chunks else "*(No text returned.)*"

def render_message_with_svgs(markdown_text: str):
    """
    Splits the response on [[MOL_IMG_X]], {{SVG_X}}, or [[PLOTLY_CHROM_X]] placeholder tokens
    and renders the stored RDKit 2D image cards or interactive Plotly chromatograms via IPython.display.HTML.
    """
    pattern = r'(\[\[MOL_IMG_\d+\]\]|\{\{SVG_\d+\}\}|\[\[PLOTLY_CHROM_\d+\]\])'
    parts = re.split(pattern, markdown_text)
    for part in parts:
        part_stripped = part.strip()
        if part_stripped in _rendered_svg_registry:
            display(HTML(_rendered_svg_registry[part_stripped]))
        elif part_stripped:
            display(Markdown(part))

def on_submit_clicked(b):
    user_text = chat_input.value.strip()
    if not user_text:
        return

    chat_input.disabled = True
    submit_button.disabled = True

    try:
        init_chat_session()
        with output_area:
            print(f"👤 You: {user_text}\n")
            print(f"🤖 {model_dropdown.value} is analyzing with RDKit, PubChem, ChEMBL & Plotly tools...\n")

        response = chat_session.send_message(user_text)
        reply_text = extract_response_text(response)

        chat_history_ui.append(("user", user_text))
        chat_history_ui.append(("model", f"**🤖 Gemini [{model_dropdown.value}]:**\n\n{reply_text}"))

        with output_area:
            clear_output(wait=True)
            for role, msg in chat_history_ui:
                if role == "user":
                    display(Markdown(f"**👤 You:** {msg}"))
                else:
                    render_message_with_svgs(msg)
                display(HTML("<hr style='border: 0; border-top: 1px solid #374151; margin: 14px 0;'>"))
            pending_plot = SPECTRAL_STATE.pop("pending_plot_html", None)
            if pending_plot:
                display(HTML(pending_plot))

    except Exception as e:
        with output_area:
            print(f"❌ Error: {e}")

    finally:
        chat_input.value = ''
        chat_input.disabled = False
        submit_button.disabled = False

def on_reset_clicked(b):
    init_chat_session(force_reset=True)

def on_uv_upload(change):
    uploaded = uv_upload.value
    if not uploaded:
        return
    item = next(iter(uploaded.values())) if isinstance(uploaded, dict) else uploaded[0]
    content = item["content"] if isinstance(item, dict) else item.content
    try:
        if len(content) > 5_000_000:
            raise ValueError("UV CSV must be under 5 MB.")
        frame = pd.read_csv(io.BytesIO(bytes(content)))
        time_col = "Time" if "Time" in frame else "time_min"
        channels = register_uv_trace(frame, time_col=time_col)
        with output_area:
            display(Markdown(f"UV/DAD loaded: {', '.join(html.escape(c) for c in channels)}. Times interpreted as minutes."))
    except (ValueError, KeyError) as exc:
        with output_area:
            display(Markdown(f"UV/DAD upload failed: {exc}"))

def on_plot_clicked(b):
    try:
        if ("times" not in SPECTRAL_STATE or
            SPECTRAL_STATE.get("data_id") != id(globals().get("od", globals().get("hplc_df"))) or
            SPECTRAL_STATE.get("reaction_text") != reaction_input.value):
            prepare_spectral_state(uv_ms_delay_min=uv_ms_delay_input.value)
        SPECTRAL_STATE["uv_ms_delay_min"] = uv_ms_delay_input.value
        status = update_lcms_plot(
            extracted_ion_mzs_csv=plot_options.value.strip(),
            select_scan_rt_min=float(scan_input.value) if scan_input.value > 0 else 0.0,
            overlay_formula_isotopes=isotope_input.value.strip()
        )
        with output_area:
            display(Markdown(status))
            if SPECTRAL_STATE.get("pending_plot_html"):
                display(HTML(SPECTRAL_STATE.pop("pending_plot_html")))
    except ValueError as exc:
        with output_area:
            display(Markdown(f"Invalid plot options: {exc}"))

submit_button.on_click(on_submit_clicked)
reset_button.on_click(on_reset_clicked)
uv_upload.observe(on_uv_upload, names="value")
plot_button.on_click(on_plot_clicked)

ui = widgets.VBox([
    chat_style,
    widgets.HTML("<h3>💬 Gemini + RDKit / PubChem / ChEMBL / MassBank HPLC-MS Assistant</h3>"),
    widgets.HBox([model_dropdown, reset_button]),
    reaction_input,
    widgets.HBox([uv_upload, uv_ms_delay_input]),
    widgets.HBox([plot_options, scan_input]),
    widgets.HBox([isotope_input, plot_button]),
    chat_input,
    submit_button,
    output_area
])
display(ui)

---
### Step 4: Multimodal Vision QC — Let Gemini *See* the Chromatogram

Until now, Gemini has only reasoned over numeric data (`detected_df`, dominant m/z lists). The cell below instead feeds the **actual rendered plot images** — the raw HPLC trace from Cell 4 and the deconvoluted, annotated chromatogram with embedded 2D structure cards from Cell 7 — directly into Gemini's multimodal vision context. This lets the model visually spot peak shapes and areas, baseline distortions, peak tailing/fronting, co-elution anomalies, and whether the drawn chemical structures look consistent with their assigned peaks — things a numeric table alone can miss.

**Vision quality check**

- Send saved chromatogram and structure images to Gemini for optional visual QC.
- Summarize peak shapes and assignment quality alongside the numerical workflow.


In [ ]:
#@title Step 4: Multimodal Vision QC — Feed Chromatogram & Structure Images to Gemini
import io
from google import genai
from google.genai import types
from google.colab import userdata
from IPython.display import display, Markdown


def _figure_to_png_bytes(fig_or_bytes, dpi: int = 160) -> bytes:
    """Rasterizes a chromatogram image to PNG bytes. Accepts either a matplotlib `Figure`
    (rasterized via `savefig`, even if already auto-closed by the inline backend — the
    Figure object itself remains valid and renderable) or raw PNG bytes already exported
    from Cell 7's interactive Plotly chromatogram via `fig.to_image()`/kaleido, so Gemini
    sees the exact same zoomable chromatogram the user inspected."""
    if isinstance(fig_or_bytes, (bytes, bytearray)):
        return bytes(fig_or_bytes)
    buf = io.BytesIO()
    fig_or_bytes.savefig(buf, format='png', dpi=dpi, bbox_inches='tight')
    buf.seek(0)
    return buf.getvalue()


def collect_vision_qc_images() -> dict:
    """
    Gathers every chromatogram image stashed by Cells 4 & 7 in `_notebook_figure_registry`.
    Cell 4 caches a matplotlib Figure (auto-closed by the inline backend, but still
    rasterizable). Cell 7's chromatogram is an interactive Plotly figure; rather than
    re-rendering it statically, Cell 7 exports the *exact* figure the user is
    zooming/panning through to PNG bytes (via kaleido) and caches those bytes here, so
    this QC step sees precisely what the user sees.
    """
    registry = globals().get('_notebook_figure_registry', {})
    images = {}
    for name, fig in registry.items():
        try:
            images[name] = _figure_to_png_bytes(fig)
        except Exception as e:
            print(f"⚠️ Could not rasterize '{name}': {e}")
    return images


_vision_client = globals().get('client') or genai.Client(api_key=userdata.get("KDD_GEMINI_API_KEY"))
_vision_model_name = globals()['model_dropdown'].value if 'model_dropdown' in globals() else 'gemini-3.8-flash'

VISION_QC_PROMPT = """You are an expert analytical chemist visually inspecting these HPLC/MS chromatogram images (not just numeric data). Examine every image provided and report, peak-by-peak where possible:

1. **Peaks & areas** — how many distinct peaks are visible, their approximate retention times, and relative areas.
2. **Baseline distortions** — drift, noise, humps, or a non-zero baseline between peaks.
3. **Peak tailing / fronting** — any visible asymmetry versus an ideal Gaussian peak shape.
4. **Co-elution anomalies** — shoulders, unresolved doublets, or overlapping peaks that may hide a second compound.
5. **Chemical structures** — for any 2D molecular structure drawings embedded on the chromatogram, describe the structure and flag if it looks inconsistent with the peak it is attached to (e.g. wrong ring system, mismatched size).

Present your findings as a concise markdown report with one section per image."""


def run_multimodal_chromatogram_vision_qc(user_question: str = None) -> str:
    """Sends the cached chromatogram/structure images into Gemini's vision context and displays its QC report."""
    images = collect_vision_qc_images()
    if not images:
        msg = "⚠️ No chromatogram figures found in `_notebook_figure_registry`. Run Cells 4 and 7 first."
        display(Markdown(msg))
        return msg

    parts = [types.Part.from_text(text=user_question or VISION_QC_PROMPT)]
    for name, png_bytes in images.items():
        parts.append(types.Part.from_text(text=f"--- Image: {name} ---"))
        parts.append(types.Part.from_bytes(data=png_bytes, mime_type='image/png'))

    response = _vision_client.models.generate_content(
        model=_vision_model_name,
        contents=[types.Content(role='user', parts=parts)],
    )

    report = getattr(response, 'text', None) or "*(No text returned.)*"
    display(Markdown(f"### 🔬 Gemini Multimodal Vision QC Report ({len(images)} image(s) analyzed)\n\n{report}"))
    return report


# Run once automatically with the default QC prompt.
# Re-run any time with a custom, targeted question, e.g.:
# run_multimodal_chromatogram_vision_qc("Does the structure card on Peak 2 look right for adenosine?")
_ = run_multimodal_chromatogram_vision_qc()